In [ ]:
from pathlib import Path
import os
import json
import zipfile
import hashlib
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage as ndi

import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink

# 通常全部留空，自动寻找。
# 多个候选时，按提示将路径填在这里，然后重跑。
RESULT_PATH = ""  # V3 ZIP，或包含 manifest.csv 的已解压目录
IMAGE_ZIP = ""    # 原始图像若以 ZIP 上传，可以填写；已解压不用填
RESTORE_REVIEW = ""  # 以前保存的 manual_review.csv；首次使用留空

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
BASE = WORK / "bus_recovery"
BASE.mkdir(parents=True, exist_ok=True)

print("环境准备完成。")
print("\n当前 Input：")
if INPUT.exists():
    for p in sorted(INPUT.iterdir()):
        print(p)
else:
    print("未发现 /kaggle/input，请在 Kaggle Notebook 中运行。")

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

print("当前 Input：")
if root.exists():
    for p in sorted(root.iterdir()):
        print(p)

print("\n结果包或结果清单：")
matches = [
    p for p in root.rglob("*")
    if p.is_file()
    and (
        p.name.lower().endswith(".zip")
        or p.name == "manifest.csv"
    )
] if root.exists() else []

for p in matches:
    print(p)

if not matches:
    print("\n尚未检测到结果文件，请确认上传后已添加到当前 Notebook。")

In [ ]:
REQUIRED = {
    "manifest.csv",
    "region_quality.csv",
    "config.json",
    "manual_review.csv",
}

def unzip_safe(src, target):
    target = Path(target)
    target.mkdir(parents=True, exist_ok=True)
    root = target.resolve()

    with zipfile.ZipFile(src) as z:
        for item in z.infolist():
            dest = (target / item.filename).resolve()
            if root != dest and root not in dest.parents:
                raise ValueError("压缩包含有异常路径。")
            if item.file_size > 2_000_000_000:
                raise ValueError("压缩包内存在异常大的文件。")
        z.extractall(target)

def valid_result_dir(path):
    path = Path(path)
    return all((path / name).is_file() for name in REQUIRED)

def find_result_dirs(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted({
        p.parent.resolve()
        for p in root.rglob("manifest.csv")
        if valid_result_dir(p.parent)
    })

def is_result_zip(path):
    try:
        with zipfile.ZipFile(path) as z:
            names = {Path(n).name for n in z.namelist()}
            return REQUIRED.issubset(names)
    except (OSError, zipfile.BadZipFile):
        return False

def choose_one(candidates, description):
    candidates = sorted(set(candidates))
    if len(candidates) == 1:
        return candidates[0]

    print(f"\n{description}：找到 {len(candidates)} 个")
    for p in candidates:
        print(p)

    if not candidates:
        raise RuntimeError(
            "没有找到完整 V3 结果。请将结果 ZIP 添加为 Input，"
            "或者把实际结果目录填入第一段 RESULT_PATH。"
        )

    raise RuntimeError(
        "存在多个结果，请把要使用的 ZIP 或目录路径"
        "填入第一段 RESULT_PATH，然后重跑第一、二段。"
    )

if RESULT_PATH.strip():
    selected = Path(RESULT_PATH.strip())
    if not selected.exists():
        raise FileNotFoundError(f"路径不存在：{selected}")
else:
    # 优先找 Input 内已解压的结果。
    directories = find_result_dirs(INPUT)
    if directories:
        selected = choose_one(directories, "已解压结果目录")
    else:
        archives = [
            p for p in INPUT.rglob("*")
            if p.is_file()
            and p.suffix.lower() == ".zip"
            and is_result_zip(p)
        ] if INPUT.exists() else []

        if archives:
            selected = choose_one(archives, "V3 结果压缩包")
        else:
            # 兼容结果还在当前工作目录的情况。
            selected = choose_one(
                find_result_dirs(WORK),
                "工作目录内的结果",
            )

if selected.is_file():
    if selected.name == "manifest.csv":
        RESULT_DIR = selected.parent
    elif is_result_zip(selected):
        digest = hashlib.sha256(selected.read_bytes()).hexdigest()[:16]
        extracted = BASE / ("results_" + digest)
        if not (extracted / ".complete").exists():
            unzip_safe(selected, extracted)
            (extracted / ".complete").touch()
        RESULT_DIR = choose_one(
            find_result_dirs(extracted),
            "压缩包内结果目录",
        )
    else:
        raise ValueError("所选文件不是完整 V3 结果 ZIP。")
else:
    RESULT_DIR = (
        selected if valid_result_dir(selected)
        else choose_one(find_result_dirs(selected), "目录内结果")
    )

if not valid_result_dir(RESULT_DIR):
    raise ValueError("结果目录缺少必要文件。")

manifest = pd.read_csv(
    RESULT_DIR / "manifest.csv",
    dtype={"image_id": str, "patient_id": str},
)
quality = pd.read_csv(
    RESULT_DIR / "region_quality.csv",
    dtype={"image_id": str},
).set_index("image_id")

if not manifest.image_id.is_unique or not quality.index.is_unique:
    raise ValueError("发现重复图像 ID。")
if set(manifest.image_id) != set(quality.index):
    raise ValueError("图像清单与区域质量表不匹配。")
if len(manifest) != 1875 or manifest.patient_id.nunique() != 1064:
    raise ValueError("这不是预期的 1875 幅图像、1064 名患者完整结果。")

cfg = json.loads((RESULT_DIR / "config.json").read_text())
SIDE = int(cfg["cfg"]["side"])

# 不同图像清单使用不同进度目录，避免误覆盖。
cohort_key = hashlib.sha256(
    "\n".join(sorted(manifest.image_id)).encode()
).hexdigest()[:12]

SAVE = BASE / ("review_" + cohort_key)
SAVE.mkdir(parents=True, exist_ok=True)

print("\n恢复成功！")
print("结果目录：", RESULT_DIR)
print("图像数量：", len(manifest))
print("患者数量：", manifest.patient_id.nunique())
print("质控保存目录：", SAVE)

In [ ]:
# 如果原始图像仍在 ZIP 中，按配置解压。
if IMAGE_ZIP.strip():
    image_archive = Path(IMAGE_ZIP.strip())
    if not image_archive.is_file():
        raise FileNotFoundError(image_archive)

    image_key = hashlib.sha256(
        image_archive.read_bytes()
    ).hexdigest()[:12]
    image_folder = BASE / ("images_" + image_key)

    if not (image_folder / ".complete").exists():
        unzip_safe(image_archive, image_folder)
        (image_folder / ".complete").touch()

# 只保留质控需要的字段，不显示标签或预测结果。
meta = manifest[
    ["image_id", "image_path", "mask_path", "pixel_hash"]
].copy()
meta = meta.sort_values("image_id").reset_index(drop=True)

needed_names = {
    Path(p).name
    for c in ["image_path", "mask_path"]
    for p in meta[c]
}
file_index = {}

for root in [INPUT, WORK]:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if p.name in needed_names and p.is_file():
            file_index.setdefault(p.name, set()).add(p.resolve())

def resolve_file(saved_path):
    direct = Path(saved_path)
    if direct.is_file():
        return str(direct.resolve())

    matches = sorted(file_index.get(direct.name, set()))
    if len(matches) == 1:
        return str(matches[0])

    if len(matches) > 1:
        # 同名副本完全相同才允许自动选择。
        hashes = {
            hashlib.sha256(p.read_bytes()).hexdigest()
            for p in matches
        }
        if len(hashes) == 1:
            return str(matches[0])
        raise ValueError(
            f"{direct.name} 有多个不同版本，请只保留正确数据源。"
        )
    return None

missing = []
for column in ["image_path", "mask_path"]:
    paths = []
    for saved in meta[column]:
        resolved = resolve_file(saved)
        paths.append(resolved)
        if resolved is None:
            missing.append(Path(saved).name)
    meta[column] = paths

if missing:
    print("缺少文件示例：", missing[:10])
    print("\nInput 内可能的原始图像 ZIP：")
    if INPUT.exists():
        for p in INPUT.rglob("*.zip"):
            print(p)
    raise RuntimeError(
        "V3 结果已经恢复，但原始图像或 Masks 尚未找到。"
        "请添加原始 BUS-BRA 数据集。"
        "若图像装在 ZIP 中，将其路径填入 IMAGE_ZIP 后从第一段重跑。"
    )

for i, row in enumerate(meta.itertuples()):
    with Image.open(row.image_path) as im, Image.open(row.mask_path) as mask:
        a = np.asarray(im.convert("L"))
        if im.size != mask.size:
            raise ValueError(f"{row.image_id}：图像与掩膜尺寸不一致。")

        pixel_hash = hashlib.sha256(
            a.tobytes() + str(a.shape).encode()
        ).hexdigest()
        if pixel_hash != row.pixel_hash:
            raise ValueError(
                f"{row.image_id} 与原实验图像不一致，请核对数据版本。"
            )

    if (i + 1) % 500 == 0:
        print("已核对：", i + 1)

print("全部图像路径有效，图像哈希与原实验一致。")
print("下一段打开人工质控界面。")

In [ ]:
from pathlib import Path
import hashlib

# 利用清单确定每条旧路径对应原图还是掩膜。
image_paths = set(meta["image_path"].astype(str))
mask_paths = set(meta["mask_path"].astype(str))

def path_kind(path):
    folders = {part.lower() for part in Path(path).parts[:-1]}
    if folders & {"masks", "mask", "segmentation", "segmentations"}:
        return "mask"
    if folders & {"images", "image", "imgs"}:
        return "image"
    return None

def file_digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def locate(saved_path):
    saved_path = str(saved_path)
    p = Path(saved_path)

    if p.is_file():
        return p

    is_image = saved_path in image_paths
    is_mask = saved_path in mask_paths

    if is_image and not is_mask:
        expected = "image"
    elif is_mask and not is_image:
        expected = "mask"
    else:
        expected = path_kind(p)

    candidates = sorted({
        Path(f).resolve()
        for f in file_index.get(p.name, set())
        if Path(f).is_file()
    })

    # 同名原图与掩膜必须分开，不能随便取第一份。
    if expected:
        candidates = [
            f for f in candidates
            if path_kind(f) == expected
        ]

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1:
        hashes = {file_digest(f) for f in candidates}
        if len(hashes) == 1:
            return candidates[0]

    details = "\n".join(str(f) for f in candidates)
    raise FileNotFoundError(
        f"{p.name}：类型={expected}，候选数={len(candidates)}。\n"
        f"{details}\n"
        "无法安全确定文件，请把这段输出发给我。"
    )

# 使用修复后的路径匹配函数重新显示当前图像。
show_case()

In [ ]:
# 人工质控完整单元：请一直复制到最后的“show_case()”
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage as ndi
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------- 1. 使用前面恢复成功的结果目录 ----------
if "RESULT_DIR" not in globals():
    raise RuntimeError("请先运行第二段，看到“恢复成功”后再运行这里。")

result_dir = Path(RESULT_DIR)
meta = pd.read_csv(
    result_dir / "manifest.csv",
    dtype={"image_id": str}
)[["image_id", "image_path", "mask_path"]]

meta = meta.sort_values("image_id").reset_index(drop=True)
assert meta.image_id.is_unique, "发现重复图像 ID"

save_dir = Path(
    globals().get("SAVE", "/kaggle/working/bus_qc_review")
)
save_dir.mkdir(parents=True, exist_ok=True)
save_file = save_dir / "manual_review.csv"

fields = {
    "core_text": "内部：文字或测量标记",
    "rim_text": "边缘：文字或测量标记",
    "posterior_text": "后方：文字或测量标记",
    "reference_text": "参考区：文字或测量标记",
    "mask_bad": "轮廓：明显错位或漏圈/多圈",
    "orientation_bad": "方向：无法确认下方是声束深部",
}

source = (
    save_file if save_file.exists()
    else result_dir / "manual_review.csv"
)
review = pd.read_csv(source, dtype={"image_id": str})
assert review.image_id.is_unique
assert set(review.image_id) == set(meta.image_id), "质控表不匹配"
review = review.set_index("image_id").loc[meta.image_id].reset_index()

for col in fields:
    review[col] = pd.to_numeric(review[col], errors="raise")
    assert review[col].dropna().isin([0, 1]).all()

if "note" not in review:
    review["note"] = ""
review["note"] = review["note"].fillna("")

# ---------- 2. 找到原始图像 ----------
file_index = {}
for root in [
    Path("/kaggle/input"),
    Path("/kaggle/working/busbra_data"),
    Path("/kaggle/working/bus_recovery/images"),
]:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_file() and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"
            }:
                file_index.setdefault(p.name, set()).add(p)

def locate(saved_path):
    p = Path(saved_path)
    if p.is_file():
        return p
    found = sorted(file_index.get(p.name, set()))
    if len(found) != 1:
        raise FileNotFoundError(
            f"{p.name} 匹配到 {len(found)} 个文件。"
            "请确认原始 BUS-BRA 的 Images 和 Masks 已添加。"
            "V3 结果包本身不包含这些图像。"
        )
    return found[0]

# ---------- 3. 与原 V3 相同的区域定义 ----------
def make_regions(mask):
    y, x = np.where(mask)
    if len(y) < 9:
        raise ValueError("掩膜为空或过小。")

    h, w = mask.shape
    thickness = max(1, round(np.sqrt(mask.sum()) * 0.06))
    core = ndi.binary_erosion(mask, iterations=thickness)
    if core.sum() < 9:
        core = mask.copy()

    rim = ndi.binary_dilation(mask, iterations=thickness) & ~core
    posterior = np.zeros_like(mask)
    reference = np.zeros_like(mask)
    band = max(3, round((y.max() - y.min() + 1) * 0.5))

    for col in range(x.min(), x.max() + 1):
        yy = np.where(mask[:, col])[0]
        if len(yy):ﬁ
            posterior[
                yy.max() + 1:min(h, yy.max() + 1 + band), col
            ] = True

    posterior &= ~(mask | rim)

    reference[
        y.min():y.max() + 1, max(0, x.min() - band):x.min()
    ] = True
    reference[
        y.min():y.max() + 1,
        x.max() + 1:min(w, x.max() + 1 + band)
    ] = True
    reference &= ~(mask | rim | posterior)

    return [core, rim, posterior, reference]

# ---------- 4. 界面 ----------
unfinished = review[list(fields)].isna().any(axis=1)
pending = np.flatnonzero(unfinished.to_numpy())
position = int(pending[0]) if len(pending) else 0

heading = widgets.HTML()
canvas = widgets.Output()
message = widgets.HTML()

controls = {
    col: widgets.Dropdown(
        options=[
            ("不确定 / 尚未判断", -1),
            ("无明显问题", 0),
            ("有问题", 1),
        ],
        value=-1,
        description=label,
        style={"description_width": "280px"},
        layout=widgets.Layout(width="600px"),
    )
    for col, label in fields.items()
}
notes = widgets.Textarea(
    description="备注",
    layout=widgets.Layout(width="600px", height="60px")
)

previous_button = widgets.Button(description="上一张（不保存）")
save_button = widgets.Button(
    description="保存并下一张", button_style="success"
)
jump_number = widgets.BoundedIntText(
    value=position + 1, min=1, max=len(meta), description="跳到第"
)
jump_button = widgets.Button(description="跳转（不保存）")

def show_case():
    row = meta.iloc[position]

    for col, control in controls.items():
        value = review.at[position, col]
        control.value = -1 if pd.isna(value) else int(value)
    notes.value = str(review.at[position, "note"])
    jump_number.value = position + 1

    complete = int(review[list(fields)].notna().all(axis=1).sum())
    heading.value = (
        f"<b>{position + 1}/{len(meta)}：{row.image_id}</b>"
        f"　六项已明确：{complete}"
        "<br>绿：内部；橙：边缘；青：后方；紫：参考区。"
        "<br>没有把握请保留“不确定”，不要强行选无问题。"
    )

    with canvas:
        clear_output(wait=True)

        with Image.open(locate(row.image_path)) as im:
            original = np.asarray(im.convert("L"))
        with Image.open(locate(row.mask_path)) as im:
            a = np.asarray(im)

        mask = np.any(a[..., :3] > 0, axis=2) if a.ndim == 3 else a > 0
        if original.shape != mask.shape:
            raise ValueError("原图与掩膜尺寸不一致。")

        scale = 192 / max(original.shape)
        size = (
            max(1, round(original.shape[1] * scale)),
            max(1, round(original.shape[0] * scale)),
        )
        x = np.asarray(
            Image.fromarray(original.astype("float32")).resize(
                size, Image.Resampling.BILINEAR
            )
        )
        mask = np.asarray(
            Image.fromarray(mask.astype("uint8")).resize(
                size, Image.Resampling.NEAREST
            )
        ) > 0

        fig, axes = plt.subplots(1, 2, figsize=(12, 7))
        axes[0].imshow(original, cmap="gray", vmin=0, vmax=255)
        axes[0].set_title("Original image")
        axes[1].imshow(x, cmap="gray", vmin=0, vmax=255)
        axes[1].set_title("V3 regions")

        for region, color in zip(
            make_regions(mask),
            ["lime", "orange", "cyan", "magenta"]
        ):
            if region.any() and not region.all():
                axes[1].contour(
                    region, levels=[0.5],
                    colors=[color], linewidths=1
                )

        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close(fig)

def save_and_next(_):
    global position
    for col, control in controls.items():
        review.at[position, col] = (
            np.nan if control.value == -1 else control.value
        )
    review.at[position, "note"] = notes.value.strip()

    temp = save_file.with_suffix(".tmp")
    review.to_csv(temp, index=False)
    os.replace(temp, save_file)

    message.value = f"已保存到：{save_file}"
    position = min(position + 1, len(meta) - 1)
    show_case()

def previous_case(_):
    global position
    position = max(0, position - 1)
    message.value = "已切换；未保存的修改不会写入。"
    show_case()

def jump_case(_):
    global position
    position = jump_number.value - 1
    message.value = "已跳转；未保存的修改不会写入。"
    show_case()

save_button.on_click(save_and_next)
previous_button.on_click(previous_case)
jump_button.on_click(jump_case)

display(
    heading,
    canvas,
    widgets.VBox(list(controls.values())),
    notes,
    widgets.HBox([previous_button, save_button]),
    widgets.HBox([jump_number, jump_button]),
    message,
)

print("质控记录保存位置：", save_file)
print("结束本次检查前，请从 Output 下载 manual_review.csv 备份。")
show_case()

In [ ]:
from pathlib import Path
import hashlib

# 利用清单确定每条旧路径对应原图还是掩膜。
image_paths = set(meta["image_path"].astype(str))
mask_paths = set(meta["mask_path"].astype(str))

def path_kind(path):
    folders = {part.lower() for part in Path(path).parts[:-1]}
    if folders & {"masks", "mask", "segmentation", "segmentations"}:
        return "mask"
    if folders & {"images", "image", "imgs"}:
        return "image"
    return None

def file_digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def locate(saved_path):
    saved_path = str(saved_path)
    p = Path(saved_path)

    if p.is_file():
        return p

    is_image = saved_path in image_paths
    is_mask = saved_path in mask_paths

    if is_image and not is_mask:
        expected = "image"
    elif is_mask and not is_image:
        expected = "mask"
    else:
        expected = path_kind(p)

    candidates = sorted({
        Path(f).resolve()
        for f in file_index.get(p.name, set())
        if Path(f).is_file()
    })

    # 同名原图与掩膜必须分开，不能随便取第一份。
    if expected:
        candidates = [
            f for f in candidates
            if path_kind(f) == expected
        ]

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1:
        hashes = {file_digest(f) for f in candidates}
        if len(hashes) == 1:
            return candidates[0]

    details = "\n".join(str(f) for f in candidates)
    raise FileNotFoundError(
        f"{p.name}：类型={expected}，候选数={len(candidates)}。\n"
        f"{details}\n"
        "无法安全确定文件，请把这段输出发给我。"
    )

# 使用修复后的路径匹配函数重新显示当前图像。
show_case()Í

In [ ]:
# 自动数据检查：不训练、不删除图片、不使用人工质控记录
from pathlib import Path
from collections import defaultdict
from datetime import datetime
import os
import hashlib
import json
import shutil

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, FileLink

# ---------- 1. 读取已恢复的实验清单 ----------
if "RESULT_DIR" not in globals():
    raise RuntimeError("请先运行前面的 V3 结果恢复单元，再运行本段。")

source_dir = Path(RESULT_DIR)
manifest_file = source_dir / "manifest.csv"
if not manifest_file.is_file():
    raise FileNotFoundError(f"没有找到：{manifest_file}")

data = pd.read_csv(
    manifest_file,
    dtype={"image_id": str, "patient_id": str},
)

required = {"image_id", "patient_id", "image_path", "mask_path"}
missing = required - set(data.columns)
if missing:
    raise ValueError(f"清单缺少这些列：{sorted(missing)}")
if data[list(required)].isna().any().any():
    raise ValueError("清单的 ID 或文件路径存在空值，请先核查清单。")
if data["image_id"].duplicated().any():
    raise ValueError("清单出现重复 image_id，请先核查清单。")

# 每次新建检查目录，避免覆盖以前的结果或质控记录。
out = Path("/kaggle/working") / (
    "bus_auto_audit_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
out.mkdir(parents=True, exist_ok=False)

print("图像数：", len(data))
print("患者 ID 数：", data["patient_id"].nunique())
print("正在查找原始图像和掩膜……", flush=True)

# ---------- 2. 分开索引原图和掩膜 ----------
def folder_kind(path):
    parts = {x.lower() for x in Path(path).parts[:-1]}
    if parts & {"mask", "masks", "segmentation", "segmentations"}:
        return "mask"
    if parts & {"image", "images", "imgs"}:
        return "image"
    return None

index = defaultdict(set)
visited = set()
extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# 兼容 Input 挂载和之前解压到工作目录的原始数据。
roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working/busbra_data"),
    Path("/kaggle/working/bus_recovery/images"),
]

for root in roots:
    if not root.exists():
        continue
    for current, dirs, files in os.walk(root, followlinks=True):
        real = str(Path(current).resolve())
        if real in visited:
            dirs[:] = []
            continue
        visited.add(real)
        dirs[:] = [
            d for d in dirs
            if d.lower() not in {"notebooks", "__macosx", ".ipynb_checkpoints"}
        ]
        for name in files:
            p = Path(current) / name
            if p.suffix.lower() in extensions:
                kind = folder_kind(p)
                if kind:
                    index[(kind, p.name)].add(p.resolve())

def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def resolve_file(old_path, kind):
    old = Path(str(old_path))
    candidates = set(index.get((kind, old.name), set()))

    if old.is_file() and folder_kind(old) == kind:
        candidates.add(old.resolve())

    candidates = sorted(candidates)
    if not candidates:
        raise FileNotFoundError(f"{kind} 未找到：{old.name}")
    if len(candidates) > 1:
        # 只允许内容完全一致的重复副本。
        if len({sha_file(p) for p in candidates}) != 1:
            raise ValueError(
                f"{kind} 存在不同内容的同名文件：{old.name}\n"
                + "\n".join(map(str, candidates))
            )
    return candidates[0]

def pixel_hash(array):
    a = np.ascontiguousarray(array)
    h = hashlib.sha256()
    h.update(str((a.shape, str(a.dtype))).encode())
    h.update(a.tobytes())
    return h.hexdigest()

# ---------- 3. 逐图自动检查 ----------
records = []

for i, row in enumerate(data.itertuples(index=False), start=1):
    record = {
        "image_id": row.image_id,
        "patient_id": row.patient_id,
        "read_ok": False,
        "size_match": False,
        "mask_nonempty": False,
        "mask_full_image": False,
        "mask_touches_border": False,
        "error": "",
    }

    try:
        image_path = resolve_file(row.image_path, "image")
        mask_path = resolve_file(row.mask_path, "mask")
        record["resolved_image_path"] = str(image_path)
        record["resolved_mask_path"] = str(mask_path)

        with Image.open(image_path) as im:
            image_array = np.asarray(im).copy()
            image_size = im.size

        with Image.open(mask_path) as im:
            mask_array = np.asarray(im.convert("L")).copy()
            mask_size = im.size

        mask = mask_array > 0
        record.update({
            "read_ok": True,
            "size_match": image_size == mask_size,
            "width": image_size[0],
            "height": image_size[1],
            "mask_pixels": int(mask.sum()),
            "mask_fraction": float(mask.mean()),
            "mask_nonempty": bool(mask.any()),
            "mask_full_image": bool(mask.all()),
            "mask_value_count": int(np.unique(mask_array).size),
            "image_pixel_sha256": pixel_hash(image_array),
            "mask_pixel_sha256": pixel_hash(mask),
            "mask_touches_border": bool(
                mask[0, :].any() or mask[-1, :].any()
                or mask[:, 0].any() or mask[:, -1].any()
            ),
        })

    except Exception as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"

    records.append(record)

    if i % 250 == 0 or i == len(data):
        print(f"已检查 {i}/{len(data)}", flush=True)

audit = pd.DataFrame(records)

# 技术性阻断项；触边只作提示，不自动判定分割错误。
audit["technical_issue"] = (
    ~audit["read_ok"]
    | ~audit["size_match"]
    | ~audit["mask_nonempty"]
    | audit["mask_full_image"]
)

# ---------- 4. 检查完全相同的图像 ----------
audit["duplicate_image"] = False
audit["duplicate_across_patients"] = False

if "image_pixel_sha256" in audit.columns:
    valid = audit["image_pixel_sha256"].notna()
    hashes = audit.loc[valid, "image_pixel_sha256"]

    audit.loc[valid, "duplicate_image"] = hashes.duplicated(keep=False)

    patient_counts = (
        audit.loc[valid]
        .groupby("image_pixel_sha256")["patient_id"]
        .nunique()
    )
    audit.loc[valid, "duplicate_across_patients"] = (
        hashes.map(patient_counts).gt(1)
    )

audit.to_csv(out / "automatic_audit.csv", index=False)

attention = audit[
    audit["technical_issue"]
    | audit["mask_touches_border"]
    | audit["duplicate_image"]
]
attention.to_csv(out / "items_to_check.csv", index=False)

# 保留成功解析的路径，供后续代码读取，不改原始 manifest。
resolved = data.merge(
    audit[[
        "image_id", "resolved_image_path", "resolved_mask_path"
    ]] if {"resolved_image_path", "resolved_mask_path"}.issubset(audit.columns)
    else audit[["image_id"]],
    on="image_id",
    how="left",
    validate="one_to_one",
)
resolved.to_csv(out / "manifest_resolved.csv", index=False)

# 现有区域质量表原样带入检查包，不重新训练或覆盖特征。
quality_file = source_dir / "region_quality.csv"
if quality_file.is_file():
    shutil.copy2(quality_file, out / "region_quality_original.csv")

summary = {
    "images": int(len(audit)),
    "patient_ids": int(audit["patient_id"].nunique()),
    "read_failures": int((~audit["read_ok"]).sum()),
    "size_mismatches": int(
        (audit["read_ok"] & ~audit["size_match"]).sum()
    ),
    "empty_masks": int(
        (audit["read_ok"] & ~audit["mask_nonempty"]).sum()
    ),
    "full_image_masks": int(audit["mask_full_image"].sum()),
    "border_touching_masks": int(audit["mask_touches_border"].sum()),
    "images_in_duplicate_groups": int(audit["duplicate_image"].sum()),
    "images_in_cross_patient_duplicate_groups": int(
        audit["duplicate_across_patients"].sum()
    ),
    "technical_issue_images": int(audit["technical_issue"].sum()),
    "manual_review_used": False,
    "training_performed": False,
}

(out / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n========== 自动检查结果 ==========")
display(pd.DataFrame([
    ["图像数", summary["images"]],
    ["患者 ID 数", summary["patient_ids"]],
    ["读取失败", summary["read_failures"]],
    ["图像与掩膜尺寸不一致", summary["size_mismatches"]],
    ["空掩膜", summary["empty_masks"]],
    ["掩膜覆盖整幅图像", summary["full_image_masks"]],
    ["掩膜触及图像边缘（仅提示）", summary["border_touching_masks"]],
    ["完全相同图像涉及的图像数", summary["images_in_duplicate_groups"]],
    ["跨患者完全相同图像涉及的图像数",
     summary["images_in_cross_patient_duplicate_groups"]],
], columns=["检查项目", "数量"]))

if summary["technical_issue_images"]:
    print("\n存在技术问题，先处理下表，不建议直接进入下一轮分析。")
    display(audit.loc[
        audit["technical_issue"],
        ["image_id", "read_ok", "size_match", "mask_nonempty",
         "mask_full_image", "error"]
    ].head(20))
else:
    print("\n读取、尺寸和基本掩膜检查通过。")

print("触边或重复只记录，不自动删除；跨患者重复需要进一步核查。")
print("本检查不能证明轮廓医学准确，也没有检测文字或测量标记污染。")
print("本次没有训练模型，没有修改原始结果或人工记录。")

archive = shutil.make_archive(str(out), "zip", root_dir=out)
print("\n检查包：", archive)
display(FileLink(archive))

In [ ]:
from pathlib import Path
from datetime import datetime
import zipfile
from IPython.display import display, FileLink

# 使用前面已成功恢复的 V3 结果目录。
if "RESULT_DIR" not in globals():
    raise RuntimeError("请先运行 V3 结果恢复单元。")

root = Path(RESULT_DIR)

# 这些文件用于核对特征、患者划分和原有结果。
required = [
    "config.json",
    "split_audit.json",
    "manifest.csv",
    "features_raw.csv",
    "features_quality_masked.csv",
    "region_quality.csv",
    "oof_patients.csv",
    "threshold_audit.csv",
]

missing = [name for name in required if not (root / name).is_file()]
if missing:
    print("当前结果目录：", root)
    print("目录内文件：")
    for p in sorted(root.iterdir()):
        print(" ", p.name)
    raise FileNotFoundError(
        "缺少以下 V3 文件：\n" + "\n".join(missing)
    )

optional = [
    "oof_metrics.csv",
    "exploratory_paired_auc.csv",
]

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output = Path("/kaggle/working") / f"bus_v3_for_ablation_{stamp}.zip"

with zipfile.ZipFile(
    output, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for name in required + optional:
        p = root / name
        if p.is_file():
            z.write(p, arcname=name)

print("打包完成，没有训练或修改实验结果。")
print("文件：", output)
print(f"大小：{output.stat().st_size / 1024 / 1024:.2f} MB")
print("点击下方链接下载，然后上传到这里：")
display(FileLink(str(output)))

In [ ]:
from pathlib import Path
import json
import hashlib
import shutil

import numpy as np
import pandas as pd
import sklearn

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.metrics import roc_auc_score, brier_score_loss
from scipy.special import logit
from scipy.stats import rankdata
from threadpoolctl import threadpool_limits
from IPython.display import display, FileLink

if "RESULT_DIR" not in globals():
    raise RuntimeError("请先运行原来的 V3 结果恢复单元。")

SRC = Path(RESULT_DIR)

FILES = [
    "manifest.csv",
    "features_raw.csv",
    "features_quality_masked.csv",
    "config.json",
    "split_audit.json",
    "region_quality.csv",
]

for name in FILES:
    if not (SRC / name).is_file():
        raise FileNotFoundError(str(SRC / name))

M = pd.read_csv(
    SRC / "manifest.csv",
    dtype={"image_id": str, "patient_id": str, "group": str},
)

C = json.loads((SRC / "config.json").read_text())
SPLITS = json.loads((SRC / "split_audit.json").read_text())

F = {
    mode: pd.read_csv(SRC / f"features_{mode}.csv")
    for mode in ["raw", "quality_masked"]
}

assert len(M) == 1875
assert M.group.nunique() == 1064
assert M.image_id.is_unique
assert not M[["image_id", "group", "label"]].isna().any().any()
assert set(M.label) == {0, 1}
assert M.groupby("group").label.nunique().eq(1).all()

Y = M.groupby("group").label.first().astype(int).sort_index()

# 特征表没有 ID，必须保持原包中的行顺序。
# 不要单独排序 manifest 或特征表。
for mode, frame in F.items():
    assert len(frame) == len(M)
    assert frame.columns.is_unique
    assert list(frame.columns) == list(F["raw"].columns)
    F[mode] = frame.apply(
        pd.to_numeric, errors="raise"
    ).replace([np.inf, -np.inf], np.nan)

Q = pd.read_csv(
    SRC / "region_quality.csv"
).set_index("image_id")

assert Q.index.is_unique
assert set(Q.index) == set(M.image_id)

manual_columns = [
    "core_text", "rim_text", "posterior_text",
    "reference_text", "mask_bad", "orientation_bad",
]

# 本轮不使用后来手工填写的 manual_review.csv。
# 原包中的自动质量表应保持未加入人工标记的状态。
if Q[manual_columns].notna().any().any():
    raise ValueError(
        "原始质量表含人工标记，需先确认 quality_masked 的生成方式。"
    )

roles = ["predictor", "calibration", "error_gate", "outer_test"]
seen = []

for split in SPLITS:
    groups = [set(split[r]) for r in roles]

    assert all(
        len(split[r]) == len(groups[i])
        for i, r in enumerate(roles)
    )
    assert set.union(*groups) == set(Y.index)
    assert sum(map(len, groups)) == len(Y)

    for group_set in groups:
        assert Y.loc[sorted(group_set)].nunique() == 2

    seen.extend(split["outer_test"])

assert len(seen) == len(Y)
assert set(seen) == set(Y.index)

base = C["feature_sets"]["rim"]
full = C["feature_sets"]["full"]

post = [c for c in full if c.startswith("posterior__")]
ref = [c for c in full if c.startswith("reference__")]

SETS = {
    "base": base,
    "base_post": base + post,
    "base_ref": base + ref + ["core_reference__difference"],
    "both_no_cross": [
        c for c in full
        if c != "posterior_reference__difference"
    ],
    "full": full,
}

for columns in SETS.values():
    assert len(columns) == len(set(columns))
    assert set(columns) <= set(F["raw"].columns)

PARAMS = {
    "version": "V4-fixed-1",
    "seed": 42,
    "trees": 200,
    "iterations": 120,
    "target_sensitivity": 0.95,
    "bootstrap": 1000,
    "sklearn": sklearn.__version__,
    "feature_sets": SETS,
    "calibration": "patient mean; Platt on logit; C=1000",
    "weighting": "equal patient weight; equal class total weight",
    "note": (
        "New fixed model parameters; compare within V4, "
        "not to V3 as same models"
    ),
}

def make_models(seed):
    def prep(model, scale=False):
        steps = [
            SimpleImputer(
                strategy="median",
                keep_empty_features=True,
            )
        ]
        if scale:
            steps.append(StandardScaler())
        return make_pipeline(*steps, model)

    return {
        "LR": prep(
            LogisticRegression(
                C=1,
                max_iter=3000,
                solver="lbfgs",
                random_state=seed,
            ),
            scale=True,
        ),
        "ET": prep(
            ExtraTreesClassifier(
                n_estimators=200,
                min_samples_leaf=3,
                max_features=0.8,
                n_jobs=2,
                random_state=seed,
            )
        ),
        "HGB": prep(
            HistGradientBoostingClassifier(
                max_iter=120,
                learning_rate=0.05,
                max_leaf_nodes=15,
                min_samples_leaf=15,
                l2_regularization=1,
                early_stopping=False,
                random_state=seed,
            )
        ),
    }

PARAMS["models"] = {
    name: model.get_params()
    for name, model in make_models(42).items()
}

serialized = json.dumps(
    PARAMS,
    ensure_ascii=False,
    sort_keys=True,
    default=str,
)

h = hashlib.sha256(serialized.encode())
for name in FILES:
    h.update(name.encode())
    h.update((SRC / name).read_bytes())

OUT = Path("/kaggle/working") / (
    "bus_v4_regions_" + h.hexdigest()[:12]
)

OUT.mkdir(parents=True, exist_ok=True)
(OUT / "checkpoints").mkdir(exist_ok=True)

(OUT / "protocol.json").write_text(
    serialized, encoding="utf-8"
)

shutil.copy2(
    SRC / "split_audit.json",
    OUT / "split_audit.json",
)

print("数据检查通过；患者数：", len(Y))
print("特征数：", {k: len(v) for k, v in SETS.items()})
print("保存目录：", OUT)

In [ ]:
def patient_scores(pipeline, frame, indices):
    probs = pipeline.predict_proba(
        frame.iloc[indices]
    )[:, 1]

    return (
        pd.Series(
            probs,
            index=M.iloc[indices].group.to_numpy(),
        )
        .groupby(level=0)
        .mean()
        .sort_index()
    )

def calibrate(raw, fitted):
    x = logit(
        np.clip(raw.to_numpy(), 1e-6, 1 - 1e-6)
    ).reshape(-1, 1)

    return pd.Series(
        fitted.predict_proba(x)[:, 1],
        index=raw.index,
    )

def atomic_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False)
    temp.replace(path)

all_predictions = []
all_thresholds = []

with threadpool_limits(limits=2):
    for mode, frame in F.items():
        for feature_set, columns in SETS.items():
            X = frame[columns]

            for fold, split in enumerate(SPLITS):
                tag = f"{mode}_{feature_set}_fold{fold}"

                pf = OUT / "checkpoints" / f"{tag}_pred.csv"
                tf = OUT / "checkpoints" / f"{tag}_threshold.csv"

                if pf.exists() and tf.exists():
                    all_predictions.append(pd.read_csv(pf))
                    all_thresholds.append(pd.read_csv(tf))
                    print("已恢复：", tag, flush=True)
                    continue

                ix = {
                    role: np.flatnonzero(
                        M.group.isin(split[role]).to_numpy()
                    )
                    for role in roles
                }

                train = M.iloc[ix["predictor"]]

                # 同一患者有多张图时，避免其总训练权重更大。
                # 再令训练部分两类患者的总权重相等。
                counts = train.groupby("group").size()
                class_counts = (
                    Y.loc[split["predictor"]].value_counts()
                )

                weights = (
                    1 / train.group.map(counts).to_numpy()
                    / train.label.map(class_counts).to_numpy()
                )
                weights = weights / weights.mean()

                predictions = {
                    role: {} for role in roles[1:]
                }

                for name, pipeline in make_models(
                    42 + fold
                ).items():
                    last = pipeline.steps[-1][0]

                    pipeline.fit(
                        X.iloc[ix["predictor"]],
                        train.label.to_numpy(),
                        **{last + "__sample_weight": weights},
                    )

                    for role in roles[1:]:
                        predictions[role][name] = patient_scores(
                            pipeline, X, ix[role]
                        )

                # 集成：先平均三个模型的患者级原始概率，
                # 然后对集成概率单独进行校准。
                for role in roles[1:]:
                    predictions[role]["ensemble"] = pd.concat(
                        [
                            predictions[role][name]
                            for name in ["LR", "ET", "HGB"]
                        ],
                        axis=1,
                    ).mean(axis=1)

                rows = []
                threshold_rows = []

                for name in ["LR", "ensemble"]:
                    cal = predictions["calibration"][name]

                    platt = LogisticRegression(
                        C=1000,
                        solver="lbfgs",
                        max_iter=2000,
                    )

                    platt.fit(
                        logit(
                            np.clip(
                                cal.to_numpy(),
                                1e-6,
                                1 - 1e-6,
                            )
                        ).reshape(-1, 1),
                        Y.loc[cal.index],
                    )

                    dev = calibrate(
                        predictions["error_gate"][name],
                        platt,
                    )
                    test = calibrate(
                        predictions["outer_test"][name],
                        platt,
                    )

                    positives = np.sort(
                        dev.loc[
                            Y.loc[dev.index].eq(1)
                        ].to_numpy()
                    )

                    # 在阈值开发集上满足 >=95% 敏感度的最高阈值。
                    allowed = int(np.floor(
                        (1 - PARAMS["target_sensitivity"])
                        * len(positives) + 1e-9
                    ))
                    threshold = float(positives[allowed])

                    threshold_rows.append({
                        "mode": mode,
                        "feature_set": feature_set,
                        "fold": fold,
                        "model": name,
                        "threshold": threshold,
                        "development_n": len(dev),
                        "development_malignant": len(positives),
                        "development_sensitivity": float(
                            (positives >= threshold).mean()
                        ),
                        "calibration_slope": float(
                            platt.coef_[0, 0]
                        ),
                    })

                    rows.append(pd.DataFrame({
                        "group": test.index,
                        "label": Y.loc[test.index].to_numpy(),
                        "p": test.to_numpy(),
                        "prediction": (
                            test.to_numpy() >= threshold
                        ).astype(int),
                        "threshold": threshold,
                        "mode": mode,
                        "feature_set": feature_set,
                        "fold": fold,
                        "model": name,
                    }))

                pred = pd.concat(rows, ignore_index=True)
                th = pd.DataFrame(threshold_rows)

                atomic_csv(pred, pf)
                atomic_csv(th, tf)

                all_predictions.append(pred)
                all_thresholds.append(th)

                print("完成：", tag, flush=True)

OOF = pd.concat(all_predictions, ignore_index=True)
TH = pd.concat(all_thresholds, ignore_index=True)

assert len(OOF) == 2 * len(SETS) * 2 * len(Y)

for _, block in OOF.groupby(
    ["mode", "feature_set", "model"]
):
    assert block.group.is_unique
    assert set(block.group) == set(Y.index)
    assert np.isfinite(block.p).all()
    assert block.p.between(0, 1).all()
    assert np.array_equal(
        block.label.to_numpy(),
        Y.loc[block.group].to_numpy(),
    )

atomic_csv(OOF, OUT / "oof_patients.csv")
atomic_csv(TH, OUT / "threshold_audit.csv")

print("训练完成。下一段只统计和打包。")

In [ ]:
def metrics(block):
    y = block.label.to_numpy().astype(int)
    p = block.p.to_numpy()
    d = block.prediction.to_numpy()

    return {
        "n": len(y),
        "auc": roc_auc_score(y, p),
        "brier": brier_score_loss(y, p),
        "sensitivity": float(d[y == 1].mean()),
        "specificity": float((d[y == 0] == 0).mean()),
        "missed_malignant": int(
            ((y == 1) & (d == 0)).sum()
        ),
        "false_positive": int(
            ((y == 0) & (d == 1)).sum()
        ),
    }

rows = []

for (mode, feature_set, model), block in OOF.groupby(
    ["mode", "feature_set", "model"]
):
    rows.append({
        "mode": mode,
        "feature_set": feature_set,
        "model": model,
        **metrics(block),
    })

METRICS = pd.DataFrame(rows)
atomic_csv(METRICS, OUT / "metrics.csv")

fold_rows = []

for (mode, feature_set, model, fold), block in OOF.groupby(
    ["mode", "feature_set", "model", "fold"]
):
    fold_rows.append({
        "mode": mode,
        "feature_set": feature_set,
        "model": model,
        "fold": fold,
        **metrics(block),
    })

atomic_csv(
    pd.DataFrame(fold_rows),
    OUT / "fold_metrics.csv",
)

def fast_auc(y, p):
    n1 = int(y.sum())
    n0 = len(y) - n1
    return (
        rankdata(p)[y == 1].sum()
        - n1 * (n1 + 1) / 2
    ) / (n1 * n0)

PAIRS = [
    ("base_post", "base"),
    ("base_ref", "base"),
    ("both_no_cross", "base_ref"),
    ("both_no_cross", "base_post"),
    ("full", "both_no_cross"),
    ("full", "base"),
]

comparisons = []

for mode in F:
    for model in ["LR", "ensemble"]:
        subset = OOF[
            (OOF["mode"] == mode)
            & (OOF.model == model)
        ]

        tables = {
            name: (
                subset[subset.feature_set == name]
                .set_index("group")
                .sort_index()
            )
            for name in SETS
        }

        baseline = tables["base"]
        y = baseline.label.to_numpy().astype(int)
        fold = baseline.fold.to_numpy()

        # 同一患者在两个模型中的预测一起重采样。
        # 保持各折、各类别的样本数。
        strata = [
            np.flatnonzero((y == c) & (fold == f))
            for f in np.unique(fold)
            for c in [0, 1]
        ]

        for a_name, b_name in PAIRS:
            a = tables[a_name]
            b = tables[b_name]

            assert a.index.equals(b.index)
            assert np.array_equal(a.fold, b.fold)
            assert np.array_equal(a.label, b.label)

            pa = a.p.to_numpy()
            pb = b.p.to_numpy()

            rng = np.random.default_rng(20261013)
            differences = []

            for _ in range(PARAMS["bootstrap"]):
                ix = np.concatenate([
                    rng.choice(
                        s, size=len(s), replace=True
                    )
                    for s in strata
                ])

                differences.append(
                    fast_auc(y[ix], pa[ix])
                    - fast_auc(y[ix], pb[ix])
                )

            lo, hi = np.quantile(
                differences, [0.025, 0.975]
            )

            comparisons.append({
                "mode": mode,
                "model": model,
                "comparison": a_name + " minus " + b_name,
                "delta_auc": (
                    fast_auc(y, pa) - fast_auc(y, pb)
                ),
                "ci_low": lo,
                "ci_high": hi,
                "delta_brier": float(
                    np.mean((pa - y) ** 2)
                    - np.mean((pb - y) ** 2)
                ),
            })

COMPARE = pd.DataFrame(comparisons)

atomic_csv(
    COMPARE,
    OUT / "paired_auc_exploratory.csv",
)

print("\n患者级外层测试结果：")
print("各折阈值只由开发集确定，未使用外层测试标签。")

with pd.option_context(
    "display.max_columns", None,
    "display.max_rows", 100,
):
    display(METRICS.round(4))

print("\n配对 AUC 差值：正值代表前一组更高。")
with pd.option_context(
    "display.max_columns", None,
    "display.max_rows", 100,
):
    display(COMPARE.round(4))

print(
    "区间来自固定 OOF 预测的折内、类别内患者重采样，"
    "不包含重新训练的不确定性。"
)
print(
    "本轮属于探索性分析，未作多重比较校正；"
    "开发集目标95%不保证外层测试达到95%。"
)

if (TH.calibration_slope <= 0).any():
    print(
        "注意：存在非正校准斜率，"
        "请连同 threshold_audit.csv 一起核查。"
    )

archive = shutil.make_archive(
    str(OUT),
    "zip",
    root_dir=OUT,
)

print("\n请下载结果包：", archive)
display(FileLink(archive))

In [ ]:
# ============================================================
# BUS-BRA V5A：同深度后方测量与自动可用性检查
# CPU运行；不训练模型；不需要人工勾选
# 请完整复制到一个新的代码单元
# ============================================================

from pathlib import Path
from collections import defaultdict
import os
import json
import zipfile
import hashlib
import time
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, FileLink


# ---------- 1. 配置 ----------
# 默认留空，自动寻找。
# 如提示多个候选，再填写 /kaggle/input/... 下的实际路径。
RESULT_PATH = ""  # V3结果ZIP或含manifest.csv、config.json的目录
IMAGE_ROOT = ""   # 原始Images和Masks的共同上级目录，可留空

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

SIDE = 192
COVERAGE = 0.80

WORK.mkdir(parents=True, exist_ok=True)


# ---------- 2. 安全解压及寻找V3结果 ----------
def safe_extract(src, target):
    target = Path(target)
    target.mkdir(parents=True, exist_ok=True)
    root = target.resolve()

    with zipfile.ZipFile(src) as z:
        total = 0
        for item in z.infolist():
            dest = (root / item.filename).resolve()
            total += item.file_size

            if dest != root and root not in dest.parents:
                raise ValueError("压缩包包含异常路径。")

            if ((item.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError("压缩包包含符号链接。")

            if total > 4_000_000_000:
                raise ValueError("结果包解压后超过4GB，请确认文件。")

        z.extractall(root)


def find_result():
    if RESULT_PATH.strip():
        candidates = [Path(RESULT_PATH.strip())]
    else:
        candidates = sorted({
            p.parent.resolve()
            for p in INPUT.rglob("manifest.csv")
            if (p.parent / "config.json").is_file()
        })

        if not candidates:
            for p in INPUT.rglob("*.zip"):
                try:
                    with zipfile.ZipFile(p) as z:
                        names = {Path(n).name for n in z.namelist()}
                    required = {
                        "manifest.csv",
                        "config.json",
                        "features_raw.csv",
                    }
                    if required <= names:
                        candidates.append(p)
                except (OSError, zipfile.BadZipFile):
                    continue

    if len(candidates) != 1:
        print("符合条件的V3结果候选：")
        for p in candidates:
            print(p)

        raise RuntimeError(
            "需要指定一份V3结果。没有候选时，请添加V3结果包；"
            "多个候选时，请填写上方RESULT_PATH。"
            "V4预测结果包不能替代V3特征结果包。"
        )

    selected = candidates[0]

    if not selected.exists():
        raise FileNotFoundError(f"路径不存在：{selected}")

    if selected.is_file():
        if selected.suffix.lower() != ".zip":
            raise ValueError("RESULT_PATH应填写ZIP或结果目录。")

        digest = hashlib.sha256(
            selected.read_bytes()
        ).hexdigest()[:12]

        target = WORK / "v5a_cache" / digest
        safe_extract(selected, target)

        found = [
            p.parent
            for p in target.rglob("manifest.csv")
            if (p.parent / "config.json").is_file()
        ]

        if len(found) != 1:
            raise ValueError("结果ZIP内需要恰好一份完整图像清单。")

        return found[0]

    if not (selected / "manifest.csv").is_file():
        raise FileNotFoundError("所选目录中没有manifest.csv。")

    return selected


# ---------- 3. 测量函数 ----------
def measurement(x, mask, valid):
    """
    测量处理后B-mode图像的相对亮度。
    不等同于物理衰减系数，也不等同于已经验证的临床征象。
    """
    if x.shape != mask.shape or valid.shape != mask.shape:
        raise ValueError("图像、掩膜、有效像素尺寸不一致。")

    yy, xx = np.where(mask)
    if len(xx) == 0:
        raise ValueError("病灶掩膜为空。")

    height, width = mask.shape
    x0, x1 = int(xx.min()), int(xx.max())
    y0, y1 = int(yy.min()), int(yy.max())

    bbox_w = x1 - x0 + 1
    bbox_h = y1 - y0 + 1

    band_depth = max(3, int(round(0.5 * bbox_h)))
    ref_width = max(3, int(round(0.5 * bbox_w)))
    gap = max(1, int(round(0.05 * bbox_w)))

    posterior = np.zeros_like(mask, dtype=bool)
    planned_rows = set()
    desired_posterior_pixels = 0

    # 每列从病灶最深边界向下取固定长度。
    # 超出图像的部分仍计入“应有面积”，用于识别截断。
    for col in range(x0, x1 + 1):
        ys = np.flatnonzero(mask[:, col])
        if len(ys) == 0:
            continue

        start = int(ys.max()) + 1
        desired_posterior_pixels += band_depth
        planned_rows.update(range(start, start + band_depth))
        posterior[start:min(height, start + band_depth), col] = True

    def rectangle(rows, lo, hi):
        region = np.zeros_like(mask, dtype=bool)
        rows = [r for r in rows if 0 <= r < height]
        cols = list(range(max(0, lo), min(width, hi)))

        if rows and cols:
            region[np.ix_(rows, cols)] = True

        return region & valid & ~mask

    left = rectangle(
        planned_rows,
        x0 - gap - ref_width,
        x0 - gap,
    )
    right = rectangle(
        planned_rows,
        x1 + gap + 1,
        x1 + gap + 1 + ref_width,
    )

    posterior &= valid & ~mask

    desired_ref_pixels = len(planned_rows) * ref_width

    posterior_coverage = (
        posterior.sum() / max(1, desired_posterior_pixels)
    )
    left_coverage = left.sum() / max(1, desired_ref_pixels)
    right_coverage = right.sum() / max(1, desired_ref_pixels)

    differences = []
    left_differences = []
    right_differences = []
    disagreements = []
    reference_values = []

    # 同一行比较后方区与左右参考区，减少深度差异的混入。
    for row in sorted(planned_rows):
        if not 0 <= row < height:
            continue

        p = x[row, posterior[row]]
        l = x[row, left[row]]
        r = x[row, right[row]]

        if min(len(p), len(l), len(r)) < 3:
            continue

        pm = float(np.median(p))
        lm = float(np.median(l))
        rm = float(np.median(r))

        differences.append(pm - 0.5 * (lm + rm))
        left_differences.append(pm - lm)
        right_differences.append(pm - rm)
        disagreements.append(abs(lm - rm))

        reference_values.extend(l)
        reference_values.extend(r)

    matched_row_coverage = (
        len(differences) / max(1, len(planned_rows))
    )

    usable = bool(
        min(
            posterior_coverage,
            left_coverage,
            right_coverage,
            matched_row_coverage,
        ) >= COVERAGE
        and min(posterior.sum(), left.sum(), right.sum()) >= 9
        and len(differences) >= 3
    )

    reasons = []

    if posterior_coverage < COVERAGE:
        reasons.append("posterior_clipped_or_invalid")

    if min(left_coverage, right_coverage) < COVERAGE:
        reasons.append("bilateral_reference_unavailable")

    if matched_row_coverage < COVERAGE or len(differences) < 3:
        reasons.append("too_few_matched_rows")

    if min(posterior.sum(), left.sum(), right.sum()) < 9:
        reasons.append("too_few_pixels")

    # 几何对照：参考区位于病灶本身的深度。
    # 注意：这不是对旧V4特征的精确重实现。
    control_left = rectangle(
        range(y0, y1 + 1),
        x0 - gap - ref_width,
        x0 - gap,
    )
    control_right = rectangle(
        range(y0, y1 + 1),
        x1 + gap + 1,
        x1 + gap + 1 + ref_width,
    )

    control_ok = (
        min(control_left.sum(), control_right.sum())
        / max(1, bbox_h * ref_width)
    ) >= COVERAGE

    delta = float(np.median(differences)) if usable else np.nan

    reference_iqr = (
        float(
            np.percentile(reference_values, 75)
            - np.percentile(reference_values, 25)
        )
        if usable else np.nan
    )

    control_delta = np.nan

    if usable and control_ok:
        control_delta = float(
            np.median(x[posterior])
            - 0.5 * (
                np.median(x[control_left])
                + np.median(x[control_right])
            )
        )

    result = {
        "usable": usable,
        "reason": ";".join(reasons),
        "posterior_coverage": float(posterior_coverage),
        "left_coverage": float(left_coverage),
        "right_coverage": float(right_coverage),
        "matched_row_coverage": float(matched_row_coverage),
        "relative_brightness": delta,
        "normalized_relative_brightness": (
            delta / max(reference_iqr, 1 / 255)
            if usable else np.nan
        ),
        "bilateral_disagreement": (
            float(np.median(disagreements)) if usable else np.nan
        ),
        "left_relative_brightness": (
            float(np.median(left_differences)) if usable else np.nan
        ),
        "right_relative_brightness": (
            float(np.median(right_differences)) if usable else np.nan
        ),
        "lesion_depth_control": control_delta,
        "mask_area_pixels": int(mask.sum()),
        "bbox_width": bbox_w,
        "bbox_height": bbox_h,
    }

    return result, (posterior, left, right)


# ---------- 4. 主程序 ----------
def main():
    result_root = find_result()
    print("V3结果目录：", result_root)

    manifest = pd.read_csv(
        result_root / "manifest.csv",
        dtype={"image_id": str, "patient_id": str},
    )

    required_columns = {
        "image_id", "patient_id", "image_path", "mask_path"
    }

    if not required_columns <= set(manifest.columns):
        raise ValueError(
            f"图像清单缺少列：{required_columns - set(manifest.columns)}"
        )

    if (
        len(manifest) != 1875
        or manifest.patient_id.nunique() != 1064
        or not manifest.image_id.is_unique
        or manifest[list(required_columns)].isna().any().any()
    ):
        raise ValueError(
            "本代码要求完整BUS-BRA清单：1875幅图像、1064名患者，"
            "且图像ID唯一、路径及ID无缺失。"
        )

    # 测量过程不使用结局标签。
    manifest = manifest[
        ["image_id", "patient_id", "image_path", "mask_path"]
    ].copy()

    cfg = json.loads(
        (result_root / "config.json").read_text()
    )

    if int(cfg["cfg"]["side"]) != SIDE:
        raise ValueError("原配置side不是192，请先核对配置。")

    print("建立原始图像索引……")
    index = defaultdict(set)
    scan_root = Path(IMAGE_ROOT.strip()) if IMAGE_ROOT.strip() else INPUT

    if not scan_root.is_dir():
        raise FileNotFoundError(f"图像目录不存在：{scan_root}")

    for directory, _, files in os.walk(scan_root, followlinks=True):
        for name in files:
            if Path(name).suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"
            }:
                index[name].add((Path(directory) / name).resolve())

    def locate(saved_path, kind):
        saved = Path(saved_path)
        found = sorted(index.get(saved.name, set()))

        if saved.is_file():
            found = sorted(set(found + [saved.resolve()]))

        folder_names = (
            {"mask", "masks"} if kind == "mask"
            else {"image", "images"}
        )

        typed = [
            p for p in found
            if any(part.lower() in folder_names for part in p.parts)
        ]

        if typed:
            found = typed

        if len(found) > 1:
            hashes = {
                hashlib.sha256(p.read_bytes()).hexdigest()
                for p in found
            }
            # 只有内容完全相同时，才允许选择其中一份。
            if len(hashes) == 1:
                found = found[:1]

        if len(found) != 1:
            raise FileNotFoundError(
                f"{kind}: {saved.name}匹配到{len(found)}份文件。\n"
                f"候选：{found}\n"
                "请添加已解压的原始BUS-BRA Images/Masks，"
                "或填写IMAGE_ROOT。V3结果包不包含原始图像。"
            )

        return found[0]

    def load_image(row):
        image_path = locate(row.image_path, "image")
        mask_path = locate(row.mask_path, "mask")

        with Image.open(image_path) as im:
            image = np.asarray(
                im.convert("L"), dtype=np.float32
            ) / 255.0
            valid = np.asarray(im.convert("RGBA"))[:, :, 3] > 0

        with Image.open(mask_path) as im:
            raw_mask = np.asarray(im)

            if (
                raw_mask.ndim == 3
                and not np.all(
                    raw_mask[:, :, :3] == raw_mask[:, :, :1]
                )
            ):
                raise ValueError(
                    "发现彩色语义掩膜，不能按BUS-BRA二值掩膜处理。"
                )

            mask = np.asarray(im.convert("L")) > 0

        if image.shape != mask.shape:
            raise ValueError("原图和掩膜尺寸不一致。")

        scale = SIDE / max(image.shape)
        new_size = (
            max(1, round(image.shape[1] * scale)),
            max(1, round(image.shape[0] * scale)),
        )

        image = np.asarray(
            Image.fromarray(image).resize(
                new_size, Image.Resampling.BILINEAR
            )
        )

        mask = np.asarray(
            Image.fromarray(mask.astype("uint8")).resize(
                new_size, Image.Resampling.NEAREST
            )
        ) > 0

        valid = np.asarray(
            Image.fromarray(valid.astype("uint8")).resize(
                new_size, Image.Resampling.NEAREST
            )
        ) > 0

        return image, mask, valid, image_path, mask_path

    # 提前检查，避免缺原图还运行整轮。
    load_image(next(manifest.itertuples(index=False)))

    run_id = (
        time.strftime("%Y%m%d_%H%M%S")
        + "_"
        + str(time.time_ns() % 1_000_000)
    )

    out = WORK / ("bus_v5a_measurement_" + run_id)
    out.mkdir(parents=True, exist_ok=False)

    protocol = {
        "version": "V5A-measurement-1",
        "side": SIDE,
        "coverage_threshold": COVERAGE,
        "posterior_band_depth": "0.5*bbox_height; minimum 3 pixels",
        "reference_width": "0.5*bbox_width; minimum 3 pixels",
        "reference_gap": "0.05*bbox_width; minimum 1 pixel",
        "primary_measurement": (
            "Median across matched rows of posterior median "
            "minus mean of left and right medians"
        ),
        "same_depth_required": True,
        "outcomes_used": False,
        "model_training": False,
        "manual_qc_required": False,
        "text_or_caliper_removal": False,
        "orientation_assumption": (
            "Rows run from superficial to deep; "
            "not independently verified"
        ),
        "control": (
            "Lesion-depth geometric reference; "
            "not an exact reimplementation of V4"
        ),
        "manifest_sha256": hashlib.sha256(
            (result_root / "manifest.csv").read_bytes()
        ).hexdigest(),
        "interpretation": (
            "Relative brightness in processed B-mode images; "
            "not a physical attenuation coefficient "
            "or validated clinical sign"
        ),
    }

    (out / "protocol.json").write_text(
        json.dumps(protocol, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    records = []
    errors = []
    examples = []
    started = time.time()

    # 固定随机种子，仅用于生成区域示例，不需要人工勾选。
    preview_indices = set(
        np.random.default_rng(42).choice(
            len(manifest), size=12, replace=False
        )
    )

    print("开始测量，共1875幅图像。每100幅显示一次进度。")

    for i, row in enumerate(manifest.itertuples(index=False)):
        try:
            image, mask, valid, ip, mp = load_image(row)
            values, regions = measurement(image, mask, valid)

            records.append({
                "image_id": row.image_id,
                "patient_id": row.patient_id,
                "image_path": str(ip),
                "mask_path": str(mp),
                "image_sha256": hashlib.sha256(
                    ip.read_bytes()
                ).hexdigest(),
                "mask_sha256": hashlib.sha256(
                    mp.read_bytes()
                ).hexdigest(),
                **values,
            })

            if i in preview_indices:
                examples.append((
                    row.image_id,
                    image,
                    mask,
                    regions,
                    values["usable"],
                ))

        except Exception as exc:
            errors.append({
                "image_id": row.image_id,
                "patient_id": row.patient_id,
                "error": str(exc),
            })

        if (i + 1) % 100 == 0:
            pd.DataFrame(records).to_csv(
                out / "measurements.csv", index=False
            )
            pd.DataFrame(
                errors,
                columns=["image_id", "patient_id", "error"],
            ).to_csv(out / "errors.csv", index=False)

            print(
                f"已处理 {i + 1}/{len(manifest)}；"
                f"读取/测量错误 {len(errors)}；"
                f"耗时 {time.time() - started:.0f} 秒",
                flush=True,
            )

    measurements = pd.DataFrame(records)
    measurements.to_csv(out / "measurements.csv", index=False)

    pd.DataFrame(
        errors,
        columns=["image_id", "patient_id", "error"],
    ).to_csv(out / "errors.csv", index=False)

    # 患者级分母来自完整清单，读取失败的图像不会被误算为“全部可用”。
    patient_table = (
        manifest.groupby("patient_id")
        .size()
        .rename("expected_images")
        .to_frame()
    )

    if len(measurements):
        observed = measurements.groupby("patient_id")["usable"].agg(
            processed_images="size",
            usable_images="sum",
        )
        patient_table = patient_table.join(observed)
    else:
        patient_table["processed_images"] = 0
        patient_table["usable_images"] = 0

    for col in ["processed_images", "usable_images"]:
        patient_table[col] = patient_table[col].fillna(0).astype(int)

    patient_table["any_usable"] = patient_table["usable_images"] > 0
    patient_table["all_usable"] = (
        patient_table["usable_images"]
        == patient_table["expected_images"]
    )

    patient_table.to_csv(out / "patient_availability.csv")

    usable_images = (
        int(measurements["usable"].sum())
        if len(measurements) else 0
    )

    summary = {
        "expected_images": int(len(manifest)),
        "processed_images": int(len(measurements)),
        "errors": len(errors),
        "usable_images": usable_images,
        "usable_fraction_of_all_images": usable_images / len(manifest),
        "expected_patients": int(len(patient_table)),
        "patients_with_any_usable_image": int(
            patient_table["any_usable"].sum()
        ),
        "patients_with_all_images_usable": int(
            patient_table["all_usable"].sum()
        ),
        "seconds": round(time.time() - started, 1),
        "reason_counts_among_processed": (
            measurements["reason"]
            .replace("", "usable")
            .value_counts()
            .to_dict()
            if len(measurements) else {}
        ),
    }

    (out / "summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    if examples:
        fig, axes = plt.subplots(3, 4, figsize=(12, 10))
        axes = axes.ravel()

        for ax in axes:
            ax.axis("off")

        for ax, example in zip(axes, examples):
            name, image, mask, regions, usable = example
            ax.imshow(image, cmap="gray", vmin=0, vmax=1)

            for region, color in zip(
                (mask, *regions),
                ("yellow", "cyan", "magenta", "lime"),
            ):
                if region.any() and not region.all():
                    ax.contour(
                        region,
                        levels=[0.5],
                        colors=[color],
                        linewidths=0.6,
                    )

            ax.set_title(
                name + (" usable" if usable else " unavailable"),
                fontsize=8,
            )

        fig.tight_layout()
        fig.savefig(out / "regions_preview.png", dpi=130)
        plt.show()
        plt.close(fig)

    print("\n完成：")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    archive = Path(shutil.make_archive(str(out), "zip", out))

    print("\n结果目录：", out)
    print("结果压缩包：", archive)
    print("请下载这个ZIP发给我，不需要逐张检查示例图。")

    if errors:
        print("有读取或测量错误，先核查errors.csv，暂不进入模型训练。")

    display(FileLink(str(archive)))


main()

# ======================== 代码结束 ========================

In [ ]:
# ============================================================
# BrEaST 外部数据接入检查
# CPU运行；不训练；不调参；不需要人工勾选
#
# Input需要：
# 1. BrEaST原始图像与掩膜ZIP，或解压后的PNG
# 2. BrEaST临床标注XLSX
# ============================================================

from pathlib import Path
from collections import defaultdict
import os
import re
import json
import hashlib
import zipfile
import shutil
import time

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, FileLink

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# 一般留空。发现多个不同版本的临床表时，再填写实际路径。
TABLE_PATH = ""

CACHE = WORK / "breast_external_cache"
CACHE.mkdir(parents=True, exist_ok=True)

REQUIRED_COLUMNS = {
    "Case_ID",
    "Image_filename",
    "Mask_tumor_filename",
    "Classification",
    "Posterior_features",
}


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def read_tables(path):
    """兼容XLSX和由原表导出的CSV，不改写字段含义。"""
    if path.suffix.lower() == ".csv":
        return {"CSV": pd.read_csv(path, dtype=str)}
    return pd.read_excel(path, sheet_name=None, dtype=str)


def discover_table():
    paths = (
        [Path(TABLE_PATH.strip())]
        if TABLE_PATH.strip()
        else sorted(
            p for p in INPUT.rglob("*")
            if p.is_file() and p.suffix.lower() in {".xlsx", ".csv"}
        )
    )

    matches = []
    failures = []

    for path in paths:
        try:
            for sheet, df in read_tables(path).items():
                df.columns = [
                    str(c).strip().lstrip("\ufeff") for c in df.columns
                ]
                if REQUIRED_COLUMNS <= set(df.columns):
                    matches.append((path, sheet, df))
        except Exception as exc:
            failures.append((str(path), str(exc)))

    # 完全相同的表只保留一份，不随意选不同版本。
    unique = {}
    for path, sheet, df in matches:
        signature = hashlib.sha256(
            df.fillna("").to_csv(index=False).encode("utf-8")
        ).hexdigest()
        unique.setdefault(signature, (path, sheet, df))

    if len(unique) != 1:
        print("符合BrEaST字段要求的表：")
        for path, sheet, _ in matches:
            print(path, "工作表：", sheet)

        if not unique:
            print("\n表格读取失败示例：", failures[:5])

        raise RuntimeError(
            "需要一份BrEaST官方临床表。"
            "请上传Clinical data的XLSX；"
            "若有多个不同版本，在TABLE_PATH中指定一个。"
        )

    return next(iter(unique.values()))


def filenames(value):
    """仅解析明确写出的图像文件名；不猜测编号或扩展名。"""
    if pd.isna(value):
        return []

    text = str(value).strip()
    if text.lower() in {"", "nan", "none", "na", "n/a", "-"}:
        return []

    # 支持单个文件名及列表、分号分隔的文件名。
    matches = re.findall(
        r"""[^/\\\s,;'"()\[\]]+\.(?:png|jpg|jpeg|tif|tiff|bmp)""",
        text,
        flags=re.IGNORECASE,
    )
    return list(dict.fromkeys(matches))


def safe_extract(src, destination):
    root = destination.resolve()
    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(src) as z:
        total = 0
        for info in z.infolist():
            target = (root / info.filename).resolve()
            total += info.file_size

            if target != root and root not in target.parents:
                raise ValueError("ZIP存在异常路径。")

            if ((info.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError("ZIP包含符号链接。")

            if total > 2_000_000_000:
                raise ValueError("ZIP解压大小异常，请核对来源。")

        z.extractall(root)


def main():
    table_path, sheet, df = discover_table()
    df = df.dropna(how="all").copy()

    if df["Case_ID"].isna().any():
        raise ValueError("存在缺失Case_ID，请核对原始表。")

    df["Case_ID"] = df["Case_ID"].str.strip()

    if not df["Case_ID"].is_unique:
        raise ValueError("Case_ID重复，暂不能按独立病例处理。")

    print("临床表：", table_path)
    print("工作表：", sheet)
    print("病例数：", len(df))
    print("寻找图像及掩膜……")

    expected_images = set()
    for value in df["Image_filename"]:
        expected_images.update(filenames(value))

    if not expected_images:
        raise ValueError("无法从Image_filename读取原图文件名。")

    # 只解压确实含有该临床表所列原图的ZIP。
    extracted_roots = []
    for archive in INPUT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(archive) as z:
                names = {Path(n).name for n in z.namelist()}

            if not (names & expected_images):
                continue

            key = sha256_file(archive)[:16]
            target = CACHE / key
            marker = target / ".extraction_complete"

            if not marker.exists():
                print("解压：", archive.name, flush=True)
                safe_extract(archive, target)
                marker.touch()

            extracted_roots.append(target)

        except zipfile.BadZipFile:
            print("跳过无法读取的ZIP：", archive)

    index = defaultdict(set)
    for root in [INPUT, *extracted_roots]:
        for directory, _, files in os.walk(root, followlinks=True):
            for name in files:
                if Path(name).suffix.lower() in {
                    ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"
                }:
                    index[name].add(
                        (Path(directory) / name).resolve()
                    )

    digest_cache = {}

    def digest(path):
        if path not in digest_cache:
            digest_cache[path] = sha256_file(path)
        return digest_cache[path]

    def locate(name):
        found = sorted(index.get(name, set()))

        if len(found) > 1:
            if len({digest(p) for p in found}) == 1:
                found = found[:1]

        if len(found) != 1:
            raise ValueError(
                f"{name}匹配到{len(found)}份不同或缺失的文件：{found}"
            )

        return found[0]

    run_id = time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns()%1000000}"
    out = WORK / ("breast_external_audit_" + run_id)
    out.mkdir(parents=True)

    rows = []
    issues = []

    for _, row in df.iterrows():
        case_id = row["Case_ID"]
        record = {
            "case_id": case_id,
            "classification_raw": row["Classification"],
            "posterior_features_raw": row["Posterior_features"],
            "status": "error",
        }

        try:
            image_names = filenames(row["Image_filename"])
            tumor_names = filenames(row["Mask_tumor_filename"])

            if len(image_names) != 1:
                raise ValueError("每例应明确对应一张原图，当前无法唯一确定。")

            ip = locate(image_names[0])

            with Image.open(ip) as im:
                width, height = im.size
                gray = np.asarray(im.convert("L"))
                alpha = np.asarray(im.convert("RGBA"))[:, :, 3]

            record.update({
                "image_filename": ip.name,
                "image_path": str(ip),
                "image_sha256": digest(ip),
                "width": width,
                "height": height,
                "valid_pixel_fraction": float((alpha > 0).mean()),
                "partial_alpha_pixels": int(
                    ((alpha > 0) & (alpha < 255)).sum()
                ),
                "tumor_mask_count": len(tumor_names),
                "mask_paths": "[]",
                "mask_sha256": "[]",
            })

            classification = (
                "" if pd.isna(row["Classification"])
                else str(row["Classification"]).strip().lower()
            )

            # 只认可明确标签，不根据病理名称、BI-RADS猜测良恶性。
            label_map = {"benign": 0, "malignant": 1}
            record["label"] = label_map.get(classification, np.nan)

            if classification not in {"benign", "malignant", "normal"}:
                raise ValueError(
                    f"未识别的Classification值：{classification!r}"
                )

            if not tumor_names:
                if classification == "normal":
                    record["status"] = "normal_without_tumor_mask"
                else:
                    raise ValueError("良性/恶性病例缺少明确病灶掩膜。")
            else:
                if classification == "normal":
                    raise ValueError("normal病例带病灶掩膜，需要核对。")

                mask_paths = []
                mask_hashes = []
                union = np.zeros((height, width), dtype=bool)

                for name in tumor_names:
                    mp = locate(name)

                    with Image.open(mp) as im:
                        arr = np.asarray(im)
                        if (
                            arr.ndim == 3
                            and not np.all(
                                arr[:, :, :3] == arr[:, :, :1]
                            )
                        ):
                            raise ValueError(
                                f"{name}为彩色掩膜，需要单独核对编码。"
                            )

                        mask_gray = np.asarray(im.convert("L"))
                        mask_alpha = np.asarray(
                            im.convert("RGBA")
                        )[:, :, 3]

                    if mask_gray.shape != gray.shape:
                        raise ValueError(f"{name}与原图尺寸不一致。")

                    # 透明区域不算作病灶。
                    mask = (mask_gray > 0) & (mask_alpha > 0)

                    if not mask.any():
                        raise ValueError(f"{name}的非零前景为空。")

                    union |= mask
                    mask_paths.append(str(mp))
                    mask_hashes.append(digest(mp))

                record.update({
                    "mask_paths": json.dumps(mask_paths),
                    "mask_sha256": json.dumps(mask_hashes),
                    "tumor_pixel_count": int(union.sum()),
                    "tumor_fraction": float(union.mean()),
                    "tumor_pixels_on_invalid_image": int(
                        (union & (alpha == 0)).sum()
                    ),
                })

                if record["tumor_pixels_on_invalid_image"] > 0:
                    raise ValueError("病灶掩膜覆盖了原图透明无效区域。")

                # 多掩膜暂不自动合并用于特征提取。
                record["status"] = (
                    "ready_single_mask"
                    if len(tumor_names) == 1
                    else "needs_multiple_mask_policy"
                )

                if len(tumor_names) > 1:
                    issues.append({
                        "case_id": case_id,
                        "issue": "多个病灶掩膜；需固定处理规则后才能提特征。",
                    })

        except Exception as exc:
            record["status"] = "error"
            issues.append({"case_id": case_id, "issue": str(exc)})

        rows.append(record)

    audit = pd.DataFrame(rows)

    # 原表单独保留，不把诊断/专家征象误当作模型输入。
    df.to_csv(out / "clinical_metadata.csv", index=False)
    audit.to_csv(out / "external_manifest_audit.csv", index=False)

    pd.DataFrame(
        issues, columns=["case_id", "issue"]
    ).to_csv(out / "issues.csv", index=False)

    duplicates = pd.DataFrame()
    if "image_sha256" in audit:
        available = audit.dropna(subset=["image_sha256"])
        duplicates = available[
            available.duplicated("image_sha256", keep=False)
        ].copy()

    duplicates.to_csv(out / "duplicate_images.csv", index=False)

    # 只报告类别取值和缺失情况，不计算标签关联、AUC或选择阈值。
    distributions = {}
    for column in [
        "Classification", "Posterior_features", "Verification"
    ]:
        if column in df:
            distributions[column] = (
                df[column].fillna("<missing>").value_counts().to_dict()
            )

    summary = {
        "cases": int(len(df)),
        "matches_expected_256_cases": bool(len(df) == 256),
        "status_counts": audit["status"].value_counts().to_dict(),
        "issue_count": len(issues),
        "duplicate_image_rows": int(len(duplicates)),
        "field_distributions": distributions,
        "model_training": False,
        "performance_evaluation": False,
        "ready_for_feature_extraction": bool(
            len(df) == 256
            and len(issues) == 0
            and len(duplicates) == 0
        ),
    }

    protocol = {
        "stage": "BrEaST external input audit",
        "source": (
            "https://www.cancerimagingarchive.net/"
            "collection/breast-lesions-usg/"
        ),
        "paper_doi": "10.1038/s41597-024-02984-z",
        "clinical_table_sha256": sha256_file(table_path),
        "clinical_sheet": str(sheet),
        "tumor_masks": "Only Mask_tumor_filename",
        "other_object_masks": "Not used as tumor masks",
        "normal_cases": "Retained in audit; not included in benign/malignant task",
        "clinical_columns": "Metadata/targets only; not classifier inputs",
        "patient_independence": (
            "Case IDs retained; cross-dataset patient overlap "
            "not established by this audit"
        ),
        "limitations": (
            "No manual contour validation; no clinical-sign validation; "
            "no source/external image-overlap check yet"
        ),
    }

    for name, content in [
        ("summary.json", summary),
        ("protocol.json", protocol),
    ]:
        (out / name).write_text(
            json.dumps(content, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    archive = Path(shutil.make_archive(str(out), "zip", out))

    print("\n检查完成：")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print("\n结果包：", archive)
    print("请下载结果包发给我。此单元没有训练模型或计算外部AUC。")

    display(FileLink(str(archive)))


main()

# ======================== 代码结束 ========================

In [ ]:
from pathlib import Path
import os
import zipfile
import pandas as pd

root = Path("/kaggle/input")
tables, archives = [], []

# 跟随挂载目录，兼容不同的Input目录结构。
for directory, _, files in os.walk(root, followlinks=True):
    for name in files:
        p = Path(directory) / name
        if p.suffix.lower() in {".xlsx", ".xls", ".csv"}:
            tables.append(p)
        elif p.suffix.lower() == ".zip":
            archives.append(p)

print("=== Input目录 ===")
for p in sorted(root.iterdir()):
    print(p)

print(f"\n=== 找到 {len(tables)} 个表格 ===")
for p in sorted(set(tables)):
    print("\n文件：", p)
    try:
        if p.suffix.lower() == ".csv":
            df = pd.read_csv(p, nrows=0)
            print("列名：", df.columns.tolist())
        else:
            with pd.ExcelFile(p) as book:
                print("工作表：", book.sheet_names)
                for sheet in book.sheet_names:
                    preview = pd.read_excel(
                        book, sheet_name=sheet,
                        header=None, nrows=4,
                    ).fillna("")
                    print(f"工作表 {sheet!r} 的前4行：")
                    print(preview.to_string(index=False, header=False))
    except Exception as e:
        print("读取失败：", type(e).__name__, str(e))

print("\n=== ZIP内是否包含临床表 ===")
for p in sorted(set(archives)):
    try:
        with zipfile.ZipFile(p) as z:
            names = [
                n for n in z.namelist()
                if Path(n).suffix.lower() in {".xlsx", ".xls", ".csv"}
            ]
        print("\n压缩包：", p)
        print("内部表格：", names[:30] if names else "没有表格")
    except Exception as e:
        print("ZIP读取失败：", p, str(e))

print("\n检查结束，请把以上输出发给我。")

In [ ]:
from pathlib import Path
import pandas as pd

# 使用你输出中已经确认存在的文件
TABLE_PATH = (
    "/kaggle/input/datasets/kokomixueyue/breast/"
    "BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx"
)

def discover_table():
    path = Path(TABLE_PATH)

    if not path.is_file():
        raise FileNotFoundError(f"找不到临床表：{path}")

    sheet = "BrEaST-Lesions-USG clinical dat"
    df = pd.read_excel(path, sheet_name=sheet, dtype=str)

    df.columns = [
        str(c).strip().lstrip("\ufeff")
        for c in df.columns
    ]

    # 官方数据表用CaseID，原代码内部用Case_ID
    if "CaseID" in df.columns and "Case_ID" not in df.columns:
        df = df.rename(columns={"CaseID": "Case_ID"})

    required = {
        "Case_ID",
        "Image_filename",
        "Mask_tumor_filename",
        "Classification",
        "Posterior_features",
    }

    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"临床表缺少字段：{sorted(missing)}")

    print("临床表识别成功：CaseID → Case_ID")
    print("病例数：", len(df))

    return path, sheet, df


# 调用前面已定义的BrEaST检查主程序
main()

In [ ]:
# ============================================================
# V5B：固定后方测量与BrEaST专家征象的一致性验证
# CPU；不训练模型；不调参数；不需要人工勾选
#
# 必须已运行BrEaST外部数据检查，且原图仍在Input。
# ============================================================

from pathlib import Path
from collections import defaultdict
import os
import json
import hashlib
import zipfile
import time
import shutil

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score
from IPython.display import display, FileLink

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# 留空：读取本会话最新生成的BrEaST检查结果。
# 也可填写之前下载、重新上传的检查结果ZIP路径。
AUDIT_PATH = ""

SIDE = 192
COVERAGE = 0.80
BOOTSTRAPS = 2000
SEED = 20261013


def read_audit():
    if AUDIT_PATH.strip():
        p = Path(AUDIT_PATH.strip())
    else:
        candidates = list(
            WORK.glob(
                "breast_external_audit_*/external_manifest_audit.csv"
            )
        )
        if candidates:
            p = max(candidates, key=lambda x: x.stat().st_mtime)
        else:
            candidates = list(
                INPUT.rglob("breast_external_audit_*.zip")
            )
            if len(candidates) != 1:
                raise RuntimeError(
                    "未找到唯一的外部检查结果。"
                    "请填写AUDIT_PATH，或先运行BrEaST检查单元。"
                )
            p = candidates[0]

    if p.is_dir():
        p = p / "external_manifest_audit.csv"

    if p.suffix.lower() == ".zip":
        with zipfile.ZipFile(p) as z:
            names = [
                n for n in z.namelist()
                if Path(n).name == "external_manifest_audit.csv"
            ]
            if len(names) != 1:
                raise ValueError("ZIP中的外部清单不唯一。")
            with z.open(names[0]) as f:
                df = pd.read_csv(f, dtype={"case_id": str})
    else:
        df = pd.read_csv(p, dtype={"case_id": str})

    print("使用外部清单：", p)

    if len(df) != 256 or not df.case_id.is_unique:
        raise ValueError("需要256例、Case ID唯一的完整BrEaST清单。")

    cohort = df[df.status == "ready_single_mask"].copy()

    if len(cohort) != 252:
        raise ValueError("预期252例具有单个病灶掩膜，请先核查清单。")

    return cohort, str(p)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


# 与V5A相同的后方带、两侧参考区、覆盖率和逐行测量规则。
# 此处只提取预先确定的原始相对亮度，不使用归一化版本选优。
def fixed_measurement(image, mask, valid):
    yy, xx = np.where(mask)
    if len(xx) == 0:
        raise ValueError("空病灶掩膜。")

    h, w = mask.shape
    x0, x1 = int(xx.min()), int(xx.max())
    y0, y1 = int(yy.min()), int(yy.max())

    bw, bh = x1 - x0 + 1, y1 - y0 + 1
    depth = max(3, int(round(0.5 * bh)))
    ref_width = max(3, int(round(0.5 * bw)))
    gap = max(1, int(round(0.05 * bw)))

    post = np.zeros_like(mask, bool)
    planned_rows = set()
    desired_post = 0

    for col in range(x0, x1 + 1):
        ys = np.flatnonzero(mask[:, col])
        if not len(ys):
            continue
        start = int(ys.max()) + 1
        desired_post += depth
        planned_rows.update(range(start, start + depth))
        post[start:min(h, start + depth), col] = True

    def rectangle(rows, lo, hi):
        out = np.zeros_like(mask, bool)
        rr = [r for r in rows if 0 <= r < h]
        cc = list(range(max(0, lo), min(w, hi)))
        if rr and cc:
            out[np.ix_(rr, cc)] = True
        return out & valid & ~mask

    left = rectangle(
        planned_rows, x0 - gap - ref_width, x0 - gap
    )
    right = rectangle(
        planned_rows, x1 + gap + 1, x1 + gap + 1 + ref_width
    )
    post &= valid & ~mask

    desired_ref = len(planned_rows) * ref_width
    cp = post.sum() / max(1, desired_post)
    cl = left.sum() / max(1, desired_ref)
    cr = right.sum() / max(1, desired_ref)

    differences = []
    disagreements = []

    for row in sorted(planned_rows):
        if not 0 <= row < h:
            continue

        p = image[row, post[row]]
        l = image[row, left[row]]
        r = image[row, right[row]]

        if min(len(p), len(l), len(r)) < 3:
            continue

        pm, lm, rm = np.median(p), np.median(l), np.median(r)
        differences.append(float(pm - 0.5 * (lm + rm)))
        disagreements.append(float(abs(lm - rm)))

    matched = len(differences) / max(1, len(planned_rows))

    usable = bool(
        min(cp, cl, cr, matched) >= COVERAGE
        and min(post.sum(), left.sum(), right.sum()) >= 9
        and len(differences) >= 3
    )

    reasons = []
    if cp < COVERAGE:
        reasons.append("posterior_clipped_or_invalid")
    if min(cl, cr) < COVERAGE:
        reasons.append("bilateral_reference_unavailable")
    if matched < COVERAGE or len(differences) < 3:
        reasons.append("too_few_matched_rows")
    if min(post.sum(), left.sum(), right.sum()) < 9:
        reasons.append("too_few_pixels")

    return {
        "usable": usable,
        "reason": ";".join(reasons),
        "relative_brightness": (
            float(np.median(differences)) if usable else np.nan
        ),
        "bilateral_disagreement": (
            float(np.median(disagreements)) if usable else np.nan
        ),
        "posterior_coverage": float(cp),
        "left_coverage": float(cl),
        "right_coverage": float(cr),
        "matched_row_coverage": float(matched),
    }


def main_v5b():
    cohort, source = read_audit()

    index = defaultdict(set)
    for directory, _, files in os.walk(INPUT, followlinks=True):
        for name in files:
            if Path(name).suffix.lower() == ".png":
                index[name].add(
                    (Path(directory) / name).resolve()
                )

    def locate(saved, expected_hash):
        p = Path(saved)
        candidates = set(index.get(p.name, set()))
        if p.is_file():
            candidates.add(p.resolve())

        # 必须与上一步检查的文件哈希相同。
        matches = [
            p for p in sorted(candidates)
            if sha256(p) == expected_hash
        ]
        if not matches:
            raise FileNotFoundError(
                f"找不到与检查结果一致的文件：{saved}"
            )
        return matches[0]

    records = []
    errors = []

    print("开始固定规则测量，252例……")

    for i, row in enumerate(cohort.itertuples(index=False)):
        try:
            paths = json.loads(row.mask_paths)
            hashes = json.loads(row.mask_sha256)

            if len(paths) != 1 or len(hashes) != 1:
                raise ValueError("此阶段只接受单个病灶掩膜。")

            ip = locate(row.image_path, row.image_sha256)
            mp = locate(paths[0], hashes[0])

            with Image.open(ip) as im:
                image = np.asarray(
                    im.convert("L"), dtype=np.float32
                ) / 255.0
                valid = np.asarray(im.convert("RGBA"))[:, :, 3] > 0

            with Image.open(mp) as im:
                mask = (
                    (np.asarray(im.convert("L")) > 0)
                    & (np.asarray(im.convert("RGBA"))[:, :, 3] > 0)
                )

            if image.shape != mask.shape:
                raise ValueError("图像和掩膜尺寸不一致。")

            scale = SIDE / max(image.shape)
            size = (
                max(1, round(image.shape[1] * scale)),
                max(1, round(image.shape[0] * scale)),
            )

            image = np.asarray(
                Image.fromarray(image).resize(
                    size, Image.Resampling.BILINEAR
                )
            )
            mask = np.asarray(
                Image.fromarray(mask.astype("uint8")).resize(
                    size, Image.Resampling.NEAREST
                )
            ) > 0
            valid = np.asarray(
                Image.fromarray(valid.astype("uint8")).resize(
                    size, Image.Resampling.NEAREST
                )
            ) > 0

            values = fixed_measurement(image, mask, valid)

            # 测量完成后才附加专家标注，不用标注改变区域。
            records.append({
                "case_id": row.case_id,
                "posterior_features": str(
                    row.posterior_features_raw
                ).strip().lower(),
                **values,
            })

        except Exception as exc:
            errors.append({
                "case_id": row.case_id,
                "error": str(exc),
            })

        if (i + 1) % 50 == 0:
            print(f"已处理 {i + 1}/252", flush=True)

    run = time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns()%1000000}"
    out = WORK / ("breast_v5b_semantics_" + run)
    out.mkdir(parents=True)

    measured = pd.DataFrame(records)
    measured.to_csv(out / "measurements.csv", index=False)
    pd.DataFrame(
        errors, columns=["case_id", "error"]
    ).to_csv(out / "errors.csv", index=False)

    # 从全部252例建立分母，读取失败也保留在可用性统计内。
    availability = cohort[["case_id", "posterior_features_raw"]].copy()
    availability["posterior_features"] = (
        availability.posterior_features_raw.astype(str)
        .str.strip().str.lower()
    )

    if len(measured):
        availability = availability.merge(
            measured[["case_id", "usable"]],
            on="case_id", how="left", validate="one_to_one",
        )
    else:
        availability["usable"] = np.nan

    availability["processed"] = availability.usable.notna()
    availability["usable"] = availability.usable.eq(True)

    grouped = availability.groupby("posterior_features").agg(
        total=("case_id", "size"),
        processed=("processed", "sum"),
        usable=("usable", "sum"),
    )
    grouped["usable_fraction"] = grouped.usable / grouped.total
    grouped.to_csv(out / "availability_by_sign.csv")

    comparisons = []

    # 固定方向：更亮对应增强；更暗对应声影。
    # combined不强行并入任何二分类。
    planned = [
        ("primary_enhancement_vs_shadowing",
         "enhancement", "shadowing", 1),
        ("secondary_enhancement_vs_no",
         "enhancement", "no", 1),
        ("secondary_shadowing_vs_no",
         "shadowing", "no", -1),
    ]

    # 有读取错误时先交回核查，不生成不完整的推断结果。
    if not errors and len(measured):
        usable = measured[measured.usable].copy()

        usable.groupby("posterior_features")[
            "relative_brightness"
        ].describe().to_csv(out / "brightness_by_sign.csv")

        for name, positive, negative, direction in planned:
            subset = usable[
                usable.posterior_features.isin([positive, negative])
            ]

            y = (
                subset.posterior_features == positive
            ).astype(int).to_numpy()
            score = (
                direction * subset.relative_brightness.to_numpy()
            )

            n_pos, n_neg = int(y.sum()), int((y == 0).sum())
            result = {
                "comparison": name,
                "positive_sign": positive,
                "negative_sign": negative,
                "positive_n": n_pos,
                "negative_n": n_neg,
                "score_direction": direction,
            }

            if min(n_pos, n_neg) < 2:
                result["status"] = "insufficient_cases"
            else:
                result["status"] = "computed"
                result["auc"] = float(roc_auc_score(y, score))

                # 病例级、类别内有放回抽样。
                rng = np.random.default_rng(SEED)
                pos = np.flatnonzero(y == 1)
                neg = np.flatnonzero(y == 0)
                boot = []

                for _ in range(BOOTSTRAPS):
                    idx = np.concatenate([
                        rng.choice(pos, len(pos), replace=True),
                        rng.choice(neg, len(neg), replace=True),
                    ])
                    boot.append(roc_auc_score(y[idx], score[idx]))

                result["ci_low"] = float(np.percentile(boot, 2.5))
                result["ci_high"] = float(np.percentile(boot, 97.5))

            comparisons.append(result)

    pd.DataFrame(comparisons).to_csv(
        out / "fixed_sign_comparisons.csv", index=False
    )

    protocol = {
        "stage": "V5B fixed posterior-sign association",
        "source_audit": source,
        "side": SIDE,
        "coverage": COVERAGE,
        "posterior_depth": "0.5*bbox_height, minimum 3",
        "reference_width": "0.5*bbox_width, minimum 3",
        "reference_gap": "0.05*bbox_width, minimum 1",
        "measurement": "V5A raw matched-row relative brightness",
        "primary": "enhancement versus shadowing, higher score positive",
        "secondary": [
            "enhancement versus no, higher score positive",
            "shadowing versus no, lower score positive",
        ],
        "combined_sign": "Descriptive only",
        "missing_measurements": "Not imputed; availability reported by sign",
        "bootstrap": "2000 case-level stratified replicates",
        "multiplicity": "Secondary intervals are descriptive, unadjusted",
        "training_or_threshold_search": False,
        "interpretation": (
            "Expert-sign association among measurable cases; "
            "not benign/malignant classification performance"
        ),
        "limitations": [
            "Image orientation assumed superficial-to-deep",
            "No text/caliper removal",
            "Availability may introduce selection",
            "Cross-dataset image-overlap check remains outstanding",
            "Do not modify rules using these results and still call this confirmatory",
        ],
    }

    summary = {
        "expected_lesion_cases": 252,
        "processed_cases": len(records),
        "errors": len(errors),
        "usable_cases": int(availability.usable.sum()),
        "usable_fraction": float(availability.usable.mean()),
        "sign_comparisons_generated": bool(comparisons),
    }

    for name, data in [
        ("protocol.json", protocol),
        ("summary.json", summary),
    ]:
        (out / name).write_text(
            json.dumps(data, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    archive = Path(shutil.make_archive(str(out), "zip", out))

    print("\n可用性：")
    print(grouped.to_string())
    print("\n摘要：")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    if comparisons:
        print("\n固定方向的征象区分结果：")
        print(pd.DataFrame(comparisons).to_string(index=False))

    print("\n结果包：", archive)
    print("请下载ZIP发回。这里的AUC不是良恶性诊断AUC。")
    display(FileLink(str(archive)))


main_v5b()

# ======================== 代码结束 ========================

In [ ]:
# ============================================================
# V6：统一、可追溯的特征提取
# CPU；不训练；不需要人工勾选
# 依赖：numpy pandas scipy scikit-image Pillow（Kaggle常见环境已有）
# ============================================================

from pathlib import Path
from collections import defaultdict
import os
import json
import time
import hashlib
import shutil
import zipfile

import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi
from skimage.measure import perimeter
from IPython.display import display, FileLink

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# 默认路径来自你已经成功读取的数据。
V3_DIR = (
    "/kaggle/input/datasets/kokomixueyue/bus-v3"
)

# 留空时找当前working中最新的BrEaST检查结果。
# 如果重启后找不到，可填重新上传的检查结果ZIP路径。
BREAST_AUDIT = ""

SIDE = 192
LEVELS = 16
COVERAGE = 0.80

# 固定规则，不根据BrEaST结果调整。
# 相同列名只是便于核对，不表示等价于旧V3实现。


def file_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def read_external():
    if BREAST_AUDIT.strip():
        p = Path(BREAST_AUDIT.strip())
    else:
        found = list(
            WORK.glob(
                "breast_external_audit_*/external_manifest_audit.csv"
            )
        )
        if found:
            p = max(found, key=lambda x: x.stat().st_mtime)
        else:
            found = list(INPUT.rglob("breast_external_audit_*.zip"))
            if len(found) != 1:
                raise RuntimeError(
                    "找不到唯一的BrEaST检查结果。"
                    "请填写BREAST_AUDIT，或重跑之前的外部数据检查。"
                )
            p = found[0]

    if p.is_dir():
        p = p / "external_manifest_audit.csv"

    if p.suffix.lower() == ".zip":
        with zipfile.ZipFile(p) as z:
            names = [
                n for n in z.namelist()
                if Path(n).name == "external_manifest_audit.csv"
            ]
            if len(names) != 1:
                raise ValueError("外部结果ZIP中的清单不唯一。")
            with z.open(names[0]) as f:
                df = pd.read_csv(f, dtype={"case_id": str})
    else:
        df = pd.read_csv(p, dtype={"case_id": str})

    if len(df) != 256 or not df.case_id.is_unique:
        raise ValueError("BrEaST清单不是256个唯一病例。")

    df = df[df.status == "ready_single_mask"].copy()
    if len(df) != 252:
        raise ValueError("预期252例带单个病灶掩膜的BrEaST病例。")

    rows = []
    for r in df.itertuples(index=False):
        masks = json.loads(r.mask_paths)
        hashes = json.loads(r.mask_sha256)
        if len(masks) != 1 or len(hashes) != 1:
            raise ValueError("发现非单个病灶掩膜。")

        rows.append({
            "dataset": "BrEaST",
            "image_id": str(r.case_id),
            "patient_id": str(r.case_id),
            "image_path": r.image_path,
            "mask_path": masks[0],
            "expected_image_sha": r.image_sha256,
            "expected_mask_sha": hashes[0],
            "label": int(float(r.label)),
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# 特征实现保存为独立源码，避免再次只留下CSV。
# ------------------------------------------------------------
EXTRACTOR_SOURCE = r'''
import numpy as np
from scipy import ndimage as ndi
from skimage.measure import perimeter

SIDE = 192
LEVELS = 16
COVERAGE = 0.80

def texture_features(x, region, valid):
    r = region & valid
    v = x[r]
    names = [
        "mean", "std", "p10", "median", "p90",
        "entropy", "edge", "contrast", "homogeneity", "energy"
    ]
    if len(v) < 9:
        return {**{k: np.nan for k in names}, "missing": 1.0}

    hist = np.histogram(v, bins=LEVELS, range=(0, 1))[0].astype(float)
    probabilities = hist[hist > 0] / hist.sum()
    entropy = -np.sum(probabilities * np.log2(probabilities))

    # 梯度只统计邻域完全有效且位于该区域内的像素。
    interior = ndi.binary_erosion(
        r, structure=np.ones((3, 3), bool), border_value=0
    )
    gx = ndi.sobel(x, axis=1, mode="reflect") / 8.0
    gy = ndi.sobel(x, axis=0, mode="reflect") / 8.0
    edge = (
        float(np.hypot(gx, gy)[interior].mean())
        if interior.any() else np.nan
    )

    # 四个方向、距离1；仅使用两个端点都位于区域内的像素对。
    q = np.clip((x * LEVELS).astype(int), 0, LEVELS - 1)
    h, w = x.shape
    counts = np.zeros((LEVELS, LEVELS), dtype=float)

    for dy, dx in [(0, 1), (1, 0), (1, 1), (1, -1)]:
        ya, yb = max(0, -dy), min(h, h - dy)
        xa, xb = max(0, -dx), min(w, w - dx)
        a = (slice(ya, yb), slice(xa, xb))
        b = (slice(ya + dy, yb + dy), slice(xa + dx, xb + dx))
        keep = r[a] & r[b]
        if keep.any():
            codes = q[a][keep] * LEVELS + q[b][keep]
            counts += np.bincount(
                codes, minlength=LEVELS * LEVELS
            ).reshape(LEVELS, LEVELS)

    counts = counts + counts.T
    if counts.sum() > 0:
        p = counts / counts.sum()
        ii, jj = np.indices(p.shape)
        contrast = float(np.sum(p * (ii - jj) ** 2))
        homogeneity = float(np.sum(p / (1 + (ii - jj) ** 2)))
        energy = float(np.sqrt(np.sum(p ** 2)))
    else:
        contrast = homogeneity = energy = np.nan

    return {
        "mean": float(v.mean()),
        "std": float(v.std(ddof=0)),
        "p10": float(np.percentile(v, 10)),
        "median": float(np.median(v)),
        "p90": float(np.percentile(v, 90)),
        "entropy": float(entropy),
        "edge": edge,
        "contrast": contrast,
        "homogeneity": homogeneity,
        "energy": energy,
        "missing": 0.0,
    }


def posterior_measurement(x, m, valid):
    yy, xx = np.where(m)
    h, w = m.shape
    x0, x1 = int(xx.min()), int(xx.max())
    y0, y1 = int(yy.min()), int(yy.max())
    bw, bh = x1 - x0 + 1, y1 - y0 + 1

    depth = max(3, round(0.5 * bh))
    width = max(3, round(0.5 * bw))
    gap = max(1, round(0.05 * bw))

    post = np.zeros_like(m)
    planned_rows = set()
    desired_post = 0

    for col in range(x0, x1 + 1):
        ys = np.flatnonzero(m[:, col])
        if len(ys):
            start = int(ys.max()) + 1
            desired_post += depth
            planned_rows.update(range(start, start + depth))
            post[start:min(h, start + depth), col] = True

    def rect(lo, hi):
        r = np.zeros_like(m)
        rows = [y for y in planned_rows if 0 <= y < h]
        cols = list(range(max(0, lo), min(w, hi)))
        if rows and cols:
            r[np.ix_(rows, cols)] = True
        return r & valid & ~m

    left = rect(x0 - gap - width, x0 - gap)
    right = rect(x1 + gap + 1, x1 + gap + 1 + width)
    post &= valid & ~m

    cp = post.sum() / max(1, desired_post)
    cl = left.sum() / max(1, len(planned_rows) * width)
    cr = right.sum() / max(1, len(planned_rows) * width)

    values = []
    for row in sorted(planned_rows):
        if not 0 <= row < h:
            continue
        p, l, r = x[row, post[row]], x[row, left[row]], x[row, right[row]]
        if min(len(p), len(l), len(r)) >= 3:
            values.append(
                float(np.median(p) - 0.5 * (np.median(l) + np.median(r)))
            )

    cm = len(values) / max(1, len(planned_rows))
    ok = bool(
        min(cp, cl, cr, cm) >= COVERAGE
        and min(post.sum(), left.sum(), right.sum()) >= 9
        and len(values) >= 3
    )
    return (
        float(np.median(values)) if ok else np.nan,
        {
            "posterior_usable": ok,
            "posterior_coverage": float(cp),
            "left_coverage": float(cl),
            "right_coverage": float(cr),
            "matched_row_coverage": float(cm),
        }
    )


def extract_features(x, m, valid):
    if x.shape != m.shape or m.shape != valid.shape:
        raise ValueError("Shape mismatch")
    if m.sum() < 9:
        raise ValueError("Lesion mask has fewer than 9 pixels")
    if (m & ~valid).any():
        raise ValueError("Lesion overlaps invalid image pixels")

    thickness = max(1, round(np.sqrt(m.sum()) * 0.06))
    core = ndi.binary_erosion(m, iterations=thickness)
    if core.sum() < 9:
        core = m.copy()
    rim = ndi.binary_dilation(m, iterations=thickness) & ~core

    result = {}
    for name, region in [("core", core), ("rim", rim)]:
        result.update({
            name + "__" + k: v
            for k, v in texture_features(x, region, valid).items()
        })

    yy, xx = np.where(m)
    bw = int(xx.max() - xx.min() + 1)
    bh = int(yy.max() - yy.min() + 1)
    area = float(m.sum())
    perim = float(perimeter(m, neighborhood=4))
    coordinates = np.column_stack([yy, xx]).astype(float)
    eig = np.linalg.eigvalsh(
        np.cov(coordinates, rowvar=False, bias=True)
    )

    result.update({
        # 这些是V6明确定义，不声称与旧V3一致。
        "shape__area": area / m.size,
        "shape__aspect": bw / bh,
        "shape__fill": area / (bw * bh),
        "shape__roughness": perim ** 2 / (4 * np.pi * area),
        "shape__elongation": float(
            np.sqrt(eig[-1] / max(eig[0], 1e-8))
        ),
        "rim_core__difference": (
            result["rim__mean"] - result["core__mean"]
        ),
    })

    delta, quality = posterior_measurement(x, m, valid)
    result["postdepth__delta"] = delta
    result["postdepth__missing"] = float(not quality["posterior_usable"])

    quality.update({
        "core_valid_fraction": float(
            (core & valid).sum() / max(1, core.sum())
        ),
        "rim_valid_fraction": float(
            (rim & valid).sum() / max(1, rim.sum())
        ),
    })
    return result, quality
'''

namespace = {}
exec(compile(EXTRACTOR_SOURCE, "v6_extractor.py", "exec"), namespace)
extract_features = namespace["extract_features"]


def main_v6():
    source = Path(V3_DIR)
    if not (source / "manifest.csv").is_file():
        raise FileNotFoundError(
            "找不到V3目录，请修改V3_DIR为包含manifest.csv的目录。"
        )

    bus = pd.read_csv(
        source / "manifest.csv",
        dtype={"image_id": str, "patient_id": str},
    )

    if (
        len(bus) != 1875
        or bus.patient_id.nunique() != 1064
        or not bus.image_id.is_unique
    ):
        raise ValueError("BUS-BRA清单数量或ID不符合预期。")

    old = pd.read_csv(source / "features_raw.csv")
    if len(old) != len(bus):
        raise ValueError("旧特征与原始清单行数不同。")

    # 在任何排序或合并之前，按原结果包行顺序绑定ID。
    old.index = bus.image_id.to_numpy()

    bus = bus[
        ["image_id", "patient_id", "image_path", "mask_path", "label"]
    ].copy()
    bus["dataset"] = "BUS-BRA"
    bus["expected_image_sha"] = ""
    bus["expected_mask_sha"] = ""

    ext = read_external()
    cohort = pd.concat([bus, ext], ignore_index=True)

    if cohort[["patient_id", "image_id", "label"]].isna().any().any():
        raise ValueError("清单存在缺失ID或标签。")

    cohort["patient_key"] = (
        cohort.dataset + ":" + cohort.patient_id.astype(str)
    )
    cohort["uid"] = cohort.dataset + ":" + cohort.image_id.astype(str)
    if not cohort.uid.is_unique:
        raise ValueError("联合清单UID重复。")

    index = defaultdict(set)
    print("建立图像索引……", flush=True)
    for directory, _, files in os.walk(INPUT, followlinks=True):
        for name in files:
            if Path(name).suffix.lower() in {".png", ".jpg", ".jpeg"}:
                index[name].add(
                    (Path(directory) / name).resolve()
                )

    hash_cache = {}

    def digest(p):
        if p not in hash_cache:
            hash_cache[p] = file_sha(p)
        return hash_cache[p]

    def locate(saved, expected, kind):
        p = Path(saved)
        candidates = set(index.get(p.name, set()))
        if p.is_file():
            candidates.add(p.resolve())

        candidates = sorted(candidates)
        if isinstance(expected, str) and expected:
            candidates = [
                q for q in candidates if digest(q) == expected
            ]
        else:
            folders = {"images", "image"} if kind == "image" else {"masks", "mask"}
            typed = [
                q for q in candidates
                if any(part.lower() in folders for part in q.parts)
            ]
            if typed:
                candidates = typed

        if len(candidates) > 1:
            if len({digest(q) for q in candidates}) == 1:
                candidates = candidates[:1]

        if len(candidates) != 1:
            raise ValueError(
                f"{kind} {p.name}: 无法唯一匹配，候选={candidates}"
            )
        return candidates[0]

    run = time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns()%1000000}"
    out = WORK / ("bus_v6_features_" + run)
    out.mkdir(parents=True)

    (out / "v6_extractor.py").write_text(
        EXTRACTOR_SOURCE, encoding="utf-8"
    )

    cohort.to_csv(out / "cohort.csv", index=False)

    feature_rows, quality_rows, file_rows, errors = [], [], [], []
    started = time.time()

    for i, row in enumerate(cohort.itertuples(index=False)):
        try:
            ip = locate(row.image_path, row.expected_image_sha, "image")
            mp = locate(row.mask_path, row.expected_mask_sha, "mask")

            with Image.open(ip) as im:
                gray = np.asarray(im.convert("L"), dtype=np.uint8)
                valid = np.asarray(im.convert("RGBA"))[:, :, 3] > 0

            # 哈希使用原始尺寸的解码灰度像素，不受文件压缩方式影响。
            ph = hashlib.sha256()
            ph.update(str(gray.shape).encode())
            ph.update(gray.tobytes())
            pixel_hash = ph.hexdigest()

            with Image.open(mp) as im:
                arr = np.asarray(im)
                if (
                    arr.ndim == 3
                    and not np.all(arr[:, :, :3] == arr[:, :, :1])
                ):
                    raise ValueError("彩色掩膜编码尚未定义。")
                mask = (
                    (np.asarray(im.convert("L")) > 0)
                    & (np.asarray(im.convert("RGBA"))[:, :, 3] > 0)
                )

            if gray.shape != mask.shape:
                raise ValueError("原图与掩膜尺寸不同。")
            if (mask & ~valid).any():
                raise ValueError("原始掩膜覆盖透明区域。")

            scale = SIDE / max(gray.shape)
            size = (
                max(1, round(gray.shape[1] * scale)),
                max(1, round(gray.shape[0] * scale)),
            )

            x = np.asarray(
                Image.fromarray(
                    gray.astype(np.float32) / 255.0
                ).resize(size, Image.Resampling.BILINEAR)
            )
            m = np.asarray(
                Image.fromarray(mask.astype("uint8")).resize(
                    size, Image.Resampling.NEAREST
                )
            ) > 0
            v = np.asarray(
                Image.fromarray(valid.astype("uint8")).resize(
                    size, Image.Resampling.NEAREST
                )
            ) > 0

            # 仅将像素、掩膜、有效区域传入，不传入任何诊断标签。
            features, quality = extract_features(x, m, v)

            feature_rows.append({"uid": row.uid, **features})
            quality_rows.append({"uid": row.uid, **quality})
            file_rows.append({
                "uid": row.uid,
                "dataset": row.dataset,
                "image_id": row.image_id,
                "image_path": str(ip),
                "mask_path": str(mp),
                "image_sha256": digest(ip),
                "mask_sha256": digest(mp),
                "decoded_gray_sha256": pixel_hash,
            })

        except Exception as exc:
            errors.append({"uid": row.uid, "error": str(exc)})

        if (i + 1) % 100 == 0:
            pd.DataFrame(feature_rows).to_csv(
                out / "features.csv", index=False
            )
            print(
                f"{i+1}/{len(cohort)}，错误={len(errors)}，"
                f"耗时={time.time()-started:.0f}秒",
                flush=True,
            )

    features = pd.DataFrame(feature_rows)
    quality = pd.DataFrame(quality_rows)
    files = pd.DataFrame(file_rows)

    features.to_csv(out / "features.csv", index=False)
    quality.to_csv(out / "quality.csv", index=False)
    files.to_csv(out / "resolved_files.csv", index=False)
    pd.DataFrame(
        errors, columns=["uid", "error"]
    ).to_csv(out / "errors.csv", index=False)

    # 旧V3与新版同名特征核对，容差固定，不据此自动宣称等价。
    comparison = []
    if len(features):
        new_bus = features[
            features.uid.str.startswith("BUS-BRA:")
        ].copy()
        new_bus.index = new_bus.uid.str.split(":", n=1).str[1]
        ids = old.index.intersection(new_bus.index)

        for col in sorted(set(old.columns) & set(new_bus.columns)):
            a = pd.to_numeric(old.loc[ids, col], errors="coerce").to_numpy()
            b = pd.to_numeric(new_bus.loc[ids, col], errors="coerce").to_numpy()
            finite = np.isfinite(a) & np.isfinite(b)
            matching = np.isclose(
                a, b, rtol=1e-5, atol=1e-7, equal_nan=True
            )

            comparison.append({
                "feature": col,
                "compared_images": len(ids),
                "finite_pairs": int(finite.sum()),
                "missing_pattern_differences": int(
                    (np.isnan(a) != np.isnan(b)).sum()
                ),
                "fraction_matching_tolerance": float(matching.mean()),
                "median_absolute_difference": (
                    float(np.median(np.abs(a[finite] - b[finite])))
                    if finite.any() else np.nan
                ),
                "max_absolute_difference": (
                    float(np.max(np.abs(a[finite] - b[finite])))
                    if finite.any() else np.nan
                ),
            })

    pd.DataFrame(comparison).to_csv(
        out / "comparison_with_v3.csv", index=False
    )

    # 完全相同的解码灰度图；不能代替近重复/患者重叠检查。
    overlap = pd.DataFrame()
    if len(files):
        cross_hashes = files.groupby(
            "decoded_gray_sha256"
        ).dataset.nunique()
        cross_hashes = cross_hashes[cross_hashes > 1].index
        overlap = files[
            files.decoded_gray_sha256.isin(cross_hashes)
        ]

    overlap.to_csv(out / "exact_cross_dataset_overlap.csv", index=False)

    base_columns = [
        c for c in features.columns
        if c != "uid" and not c.startswith("postdepth__")
    ]

    protocol = {
        "version": "V6-unified-features-1",
        "extractor_sha256": hashlib.sha256(
            EXTRACTOR_SOURCE.encode("utf-8")
        ).hexdigest(),
        "side": SIDE,
        "grayscale_range": [0, 1],
        "resize": "Preserve aspect ratio; bilinear image; nearest masks",
        "glcm": "16 levels; distance 1; four directions pooled; symmetric; both pixels inside region",
        "feature_sets": {
            "base": base_columns,
            "base_plus_posterior": base_columns + [
                "postdepth__delta", "postdepth__missing"
            ],
        },
        "labels_used_in_extractor": False,
        "feature_selection_or_training": False,
        "missing_values": "Retained as NaN; future imputation fitted on training data only",
        "relationship_to_v3": "New explicit implementation; equivalence not assumed",
        "posterior": "Fixed V5A raw measurement; no changes using BrEaST signs",
        "limitations": [
            "No text/caliper removal",
            "Depth direction assumed from image rows",
            "No assertion of clinical contour correctness",
            "Exact grayscale overlap check does not detect all near duplicates",
            "BrEaST has already been used for posterior-sign analysis; disclose this history",
        ],
    }

    summary = {
        "expected_images": int(len(cohort)),
        "processed_images": int(len(features)),
        "errors": len(errors),
        "feature_count": max(0, len(features.columns) - 1),
        "exact_cross_dataset_overlap_rows": int(len(overlap)),
        "seconds": round(time.time() - started, 1),
        "ready_for_review": bool(
            len(features) == len(cohort)
            and not errors
            and len(overlap) == 0
        ),
    }

    if len(quality):
        q = quality.merge(
            cohort[["uid", "dataset"]], on="uid", validate="one_to_one"
        )
        summary["posterior_availability"] = (
            q.groupby("dataset").posterior_usable
            .agg(["count", "sum", "mean"])
            .to_dict(orient="index")
        )

    for name, data in [
        ("protocol.json", protocol),
        ("summary.json", summary),
    ]:
        (out / name).write_text(
            json.dumps(data, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    # 同时保存本次实际执行单元，包含读文件和预处理代码。
    try:
        history = get_ipython().history_manager.input_hist_raw
        this_cell = history[-1]
        (out / "executed_cell.py").write_text(
            this_cell, encoding="utf-8"
        )
    except Exception as exc:
        (out / "source_export_note.txt").write_text(
            "单元历史导出失败，请另存Notebook：" + str(exc),
            encoding="utf-8",
        )

    archive = Path(shutil.make_archive(str(out), "zip", out))

    print("\n完成：")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print("\n与V3同名特征的核对：")
    if comparison:
        display(pd.DataFrame(comparison))

    print("\n结果包：", archive)
    print("请下载ZIP发给我；现在不要自行删除缺失值或开始训练。")
    display(FileLink(str(archive)))


main_v6()

# ======================== 代码结束 ========================

In [ ]:
# ============================================================
# V6：固定训练、校准、阈值开发与BrEaST外部评估
# CPU；3组特征 × LR/集成；不搜索参数
# ============================================================

from pathlib import Path
import os
import json
import time
import hashlib
import zipfile
import shutil
import platform
import importlib.metadata

import numpy as np
import pandas as pd
import joblib

from scipy.special import logit
from scipy.stats import rankdata
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.metrics import roc_auc_score, brier_score_loss
from threadpoolctl import threadpool_limits
from IPython.display import display, FileLink

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# 多份V6结果时，填写要用的ZIP或目录。
FEATURE_PATH = ""

SEED = 42
TARGET_SENSITIVITY = 0.95
BOOTSTRAPS = 2000
THREADS = 2


# ---------- 1. 输入与安全解压 ----------
def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def safe_extract(src, target):
    target.mkdir(parents=True, exist_ok=True)
    root = target.resolve()

    with zipfile.ZipFile(src) as z:
        total = 0
        for item in z.infolist():
            p = (root / item.filename).resolve()
            total += item.file_size

            if p != root and root not in p.parents:
                raise ValueError("ZIP包含异常路径。")
            if ((item.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError("ZIP包含符号链接。")
            if total > 2_000_000_000:
                raise ValueError("V6特征结果包异常大。")

        z.extractall(root)


def find_features():
    required = {
        "features.csv", "cohort.csv", "protocol.json", "summary.json"
    }

    def valid(p):
        return p.is_dir() and all((p / n).is_file() for n in required)

    if FEATURE_PATH.strip():
        selected = Path(FEATURE_PATH.strip())
    else:
        candidates = [
            p for p in WORK.glob("bus_v6_features_*")
            if valid(p)
        ]

        if not candidates:
            candidates = [
                p.parent for p in INPUT.rglob("features.csv")
                if valid(p.parent)
                and (p.parent / "v6_extractor.py").is_file()
            ]

        candidates = sorted(set(candidates))

        if not candidates:
            candidates = sorted(INPUT.rglob("bus_v6_features_*.zip"))

        if len(candidates) != 1:
            print("找到的候选：")
            for p in candidates:
                print(p)
            raise RuntimeError(
                "请在FEATURE_PATH填写唯一的V6结果ZIP或目录。"
            )

        selected = candidates[0]

    if selected.is_file():
        if selected.suffix.lower() != ".zip":
            raise ValueError("FEATURE_PATH应为ZIP或结果目录。")

        target = WORK / "v6_train_cache" / sha_file(selected)[:16]
        safe_extract(selected, target)

        found = [
            p.parent for p in target.rglob("features.csv")
            if valid(p.parent)
        ]
        if len(found) != 1:
            raise ValueError("ZIP内的V6结果目录不唯一。")

        selected = found[0]

    if not valid(selected):
        raise ValueError("所选目录缺少V6必要文件。")

    return selected


# ---------- 2. 固定模型 ----------
def make_models():
    def prep(model, scale=False):
        steps = [
            SimpleImputer(
                strategy="median",
                keep_empty_features=True,
            )
        ]
        if scale:
            steps.append(StandardScaler())
        return make_pipeline(*steps, model)

    return {
        "LR": prep(
            LogisticRegression(
                C=1,
                max_iter=3000,
                solver="lbfgs",
                random_state=SEED,
            ),
            scale=True,
        ),
        "ET": prep(
            ExtraTreesClassifier(
                n_estimators=200,
                min_samples_leaf=3,
                max_features=0.8,
                random_state=SEED,
                n_jobs=THREADS,
            )
        ),
        "HGB": prep(
            HistGradientBoostingClassifier(
                max_iter=120,
                learning_rate=0.05,
                max_leaf_nodes=15,
                min_samples_leaf=15,
                l2_regularization=1,
                early_stopping=False,
                random_state=SEED,
            )
        ),
    }


def patient_probability(pipeline, frame, columns):
    values = pipeline.predict_proba(frame[columns])[:, 1]
    return (
        pd.Series(values, index=frame.patient_key.to_numpy())
        .groupby(level=0)
        .mean()
        .sort_index()
    )


def calibrated_probability(raw, calibrator):
    x = logit(
        np.clip(raw.to_numpy(), 1e-6, 1 - 1e-6)
    ).reshape(-1, 1)

    return pd.Series(
        calibrator.predict_proba(x)[:, 1],
        index=raw.index,
    )


def fast_auc(y, score):
    y = np.asarray(y, dtype=int)
    n1 = int(y.sum())
    n0 = len(y) - n1
    ranks = rankdata(score, method="average")
    return float(
        (ranks[y == 1].sum() - n1 * (n1 + 1) / 2)
        / (n1 * n0)
    )


def wilson(successes, total):
    if not total:
        return np.nan, np.nan
    z = 1.959963984540054
    p = successes / total
    den = 1 + z * z / total
    center = (p + z * z / (2 * total)) / den
    radius = (
        z * np.sqrt(
            p * (1 - p) / total + z * z / (4 * total * total)
        ) / den
    )
    return float(center - radius), float(center + radius)


def main_train():
    src = find_features()
    print("使用特征结果：", src, flush=True)

    feature_protocol = json.loads(
        (src / "protocol.json").read_text()
    )
    summary = json.loads((src / "summary.json").read_text())

    if feature_protocol.get("version") != "V6-unified-features-1":
        raise ValueError("此训练代码只接受指定V6特征版本。")

    if (
        summary.get("errors") != 0
        or summary.get("processed_images") != 2127
        or summary.get("exact_cross_dataset_overlap_rows") != 0
    ):
        raise ValueError("V6提取或重复检查未通过，先处理问题。")

    extractor = src / "v6_extractor.py"
    if not extractor.is_file():
        raise FileNotFoundError("缺少V6提取源码。")

    if sha_file(extractor) != feature_protocol["extractor_sha256"]:
        raise ValueError("提取源码与协议哈希不符。")

    meta = pd.read_csv(
        src / "cohort.csv",
        dtype={
            "uid": str,
            "image_id": str,
            "patient_id": str,
            "patient_key": str,
        },
    )
    features = pd.read_csv(src / "features.csv", dtype={"uid": str})

    if (
        not meta.uid.is_unique
        or not features.uid.is_unique
        or set(meta.uid) != set(features.uid)
    ):
        raise ValueError("特征和清单ID不匹配。")

    data = meta.merge(features, on="uid", validate="one_to_one")
    data["label"] = pd.to_numeric(data.label, errors="raise").astype(int)

    if set(data.label) != {0, 1}:
        raise ValueError("诊断标签必须为0/1。")

    if not data.groupby("patient_key").label.nunique().eq(1).all():
        raise ValueError("同一患者存在冲突诊断标签。")

    base = list(feature_protocol["feature_sets"]["base"])
    sets = {
        "base": base,
        "base_missing": base + ["postdepth__missing"],
        "base_posterior": base + [
            "postdepth__missing", "postdepth__delta"
        ],
    }

    all_columns = list(dict.fromkeys(
        c for columns in sets.values() for c in columns
    ))

    for columns in sets.values():
        if len(columns) != len(set(columns)):
            raise ValueError("特征列表存在重复列。")

    for col in all_columns:
        data[col] = pd.to_numeric(data[col], errors="raise")

    if np.isinf(data[all_columns].to_numpy()).any():
        raise ValueError("特征中存在无穷值。")

    if not data["postdepth__missing"].isin([0, 1]).all():
        raise ValueError("后方缺失标记不是0/1。")

    if not np.array_equal(
        data["postdepth__delta"].isna().to_numpy(),
        data["postdepth__missing"].eq(1).to_numpy(),
    ):
        raise ValueError("后方测量缺失值与标记不一致。")

    bus = data[data.dataset == "BUS-BRA"].copy()
    ext = data[data.dataset == "BrEaST"].copy()

    if (
        len(bus) != 1875
        or bus.patient_key.nunique() != 1064
        or len(ext) != 252
        or ext.patient_key.nunique() != 252
    ):
        raise ValueError("数据量不符合预期。")

    # 标签不进入特征列表。
    prohibited = {
        "label", "patient_id", "patient_key", "uid",
        "Classification", "Posterior_features",
    }
    if prohibited & set(all_columns):
        raise ValueError("预测特征中混入标签或ID。")

    source_y = bus.groupby("patient_key").label.first().sort_index()

    # 固定一次患者级分层划分。大约60%/20%/20%。
    train_ids, remaining_ids = train_test_split(
        source_y.index.to_numpy(),
        test_size=0.40,
        stratify=source_y.to_numpy(),
        random_state=SEED,
    )
    cal_ids, gate_ids = train_test_split(
        remaining_ids,
        test_size=0.50,
        stratify=source_y.loc[remaining_ids].to_numpy(),
        random_state=SEED + 1,
    )

    roles = {
        "train": sorted(train_ids),
        "calibration": sorted(cal_ids),
        "threshold": sorted(gate_ids),
    }

    role_sets = [set(v) for v in roles.values()]
    if (
        set.union(*role_sets) != set(source_y.index)
        or sum(map(len, role_sets)) != len(source_y)
    ):
        raise ValueError("来源数据划分有重叠或遗漏。")

    frames = {
        role: bus[bus.patient_key.isin(ids)].copy()
        for role, ids in roles.items()
    }

    train = frames["train"]

    # 每位患者总训练权重相同，再使两类总权重相同。
    image_counts = train.groupby("patient_key").size()
    class_counts = source_y.loc[roles["train"]].value_counts()

    weights = (
        1.0 / train.patient_key.map(image_counts).to_numpy()
        / train.label.map(class_counts).to_numpy()
    )
    weights /= weights.mean()

    run = time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns()%1000000}"
    out = WORK / ("bus_v6_external_training_" + run)
    out.mkdir(parents=True)
    (out / "models").mkdir()

    split_table = pd.DataFrame([
        {
            "patient_key": patient,
            "role": role,
            "label": int(source_y.loc[patient]),
        }
        for role, ids in roles.items()
        for patient in ids
    ])
    split_table.to_csv(out / "source_patient_split.csv", index=False)

    versions = {"python": platform.python_version()}
    for package in ["numpy", "pandas", "scipy", "scikit-learn", "joblib"]:
        versions[package] = importlib.metadata.version(package)

    protocol = {
        "version": "V6-external-fixed-1",
        "seed": SEED,
        "feature_sets": sets,
        "source_split": "Patient-stratified 60% train, 20% calibration, 20% threshold",
        "imputation": "Training-only median, keep_empty_features=True",
        "scaling": "Training-only StandardScaler for LR",
        "weighting": "Equal patient weight, equal class total weight",
        "ensemble": "Mean LR/ET/HGB patient probabilities, then separate Platt calibration",
        "calibration": "Unweighted LR(C=1000) on clipped logit of patient probabilities",
        "threshold": "Highest threshold attaining >=95% empirical sensitivity in source threshold subset",
        "target_sensitivity": TARGET_SENSITIVITY,
        "external_recalibration": False,
        "parameter_search": False,
        "primary_model": "LR",
        "primary_increment": "base_posterior minus base_missing",
        "secondary_model": "ensemble",
        "bootstrap_replicates": BOOTSTRAPS,
        "ci_note": "Conditional on fitted models; does not include training-split uncertainty",
        "multiplicity": "Secondary intervals are descriptive and unadjusted",
        "prior_external_use": (
            "BrEaST posterior-sign labels were previously examined in V5B; "
            "disclose this development history"
        ),
        "feature_protocol": feature_protocol,
        "input_hashes": {
            name: sha_file(src / name)
            for name in ["features.csv", "cohort.csv", "v6_extractor.py"]
        },
        "versions": versions,
        "model_parameters": {
            name: {
                k: str(v) if not isinstance(
                    v, (str, int, float, bool, type(None))
                ) else v
                for k, v in model.get_params().items()
            }
            for name, model in make_models().items()
        },
    }

    (out / "protocol.json").write_text(
        json.dumps(protocol, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    shutil.copy2(extractor, out / "v6_extractor.py")

    try:
        cell = get_ipython().history_manager.input_hist_raw[-1]
        (out / "training_cell.py").write_text(cell, encoding="utf-8")
    except Exception:
        pass

    print("\n患者划分：")
    print(split_table.groupby(["role", "label"]).size().unstack(fill_value=0))
    print("\n开始训练，共9次基础模型拟合。", flush=True)

    external_predictions = []
    source_predictions = []
    thresholds = []
    started = time.time()

    with threadpool_limits(limits=THREADS):
        for set_name, columns in sets.items():
            models = make_models()

            raw = {
                "calibration": {},
                "threshold": {},
                "external": {},
            }

            for model_name, pipeline in models.items():
                print(f"训练 {set_name} / {model_name}", flush=True)
                final_step = pipeline.steps[-1][0]

                pipeline.fit(
                    train[columns],
                    train.label.to_numpy(),
                    **{final_step + "__sample_weight": weights},
                )

                for role in ["calibration", "threshold"]:
                    raw[role][model_name] = patient_probability(
                        pipeline, frames[role], columns
                    )

                # 外部标签不用于预测、拟合或校准。
                raw["external"][model_name] = patient_probability(
                    pipeline, ext, columns
                )

            for role in raw:
                raw[role]["ensemble"] = pd.concat(
                    [raw[role][name] for name in ["LR", "ET", "HGB"]],
                    axis=1,
                ).mean(axis=1)

            for model_name in ["LR", "ensemble"]:
                cal_raw = raw["calibration"][model_name]

                calibrator = LogisticRegression(
                    C=1000,
                    solver="lbfgs",
                    max_iter=2000,
                )
                calibrator.fit(
                    logit(
                        np.clip(cal_raw.to_numpy(), 1e-6, 1 - 1e-6)
                    ).reshape(-1, 1),
                    source_y.loc[cal_raw.index].to_numpy(),
                )

                gate = calibrated_probability(
                    raw["threshold"][model_name], calibrator
                )
                external = calibrated_probability(
                    raw["external"][model_name], calibrator
                )

                positives = np.sort(
                    gate.loc[
                        source_y.loc[gate.index].eq(1)
                    ].to_numpy()
                )
                if len(positives) == 0:
                    raise ValueError("阈值开发集没有恶性患者。")

                allowed_misses = int(np.floor(
                    (1 - TARGET_SENSITIVITY) * len(positives) + 1e-9
                ))
                threshold = float(positives[allowed_misses])

                thresholds.append({
                    "feature_set": set_name,
                    "model": model_name,
                    "threshold": threshold,
                    "threshold_patients": len(gate),
                    "threshold_malignant": len(positives),
                    "development_sensitivity": float(
                        (positives >= threshold).mean()
                    ),
                    "calibration_slope": float(calibrator.coef_[0, 0]),
                    "calibration_intercept": float(calibrator.intercept_[0]),
                })

                external_predictions.append(pd.DataFrame({
                    "patient_key": external.index,
                    "feature_set": set_name,
                    "model": model_name,
                    "raw_p": raw["external"][model_name].loc[
                        external.index
                    ].to_numpy(),
                    "p": external.to_numpy(),
                    "threshold": threshold,
                    "prediction": (
                        external.to_numpy() >= threshold
                    ).astype(int),
                }))

                source_predictions.append(pd.DataFrame({
                    "patient_key": gate.index,
                    "feature_set": set_name,
                    "model": model_name,
                    "role": "threshold_development",
                    "label": source_y.loc[gate.index].to_numpy(),
                    "p": gate.to_numpy(),
                    "threshold": threshold,
                    "prediction": (
                        gate.to_numpy() >= threshold
                    ).astype(int),
                }))

                used_models = (
                    {"LR": models["LR"]}
                    if model_name == "LR" else models
                )
                joblib.dump(
                    {
                        "feature_columns": columns,
                        "models": used_models,
                        "model_type": model_name,
                        "calibrator": calibrator,
                        "threshold": threshold,
                        "patient_aggregation": "mean",
                        "feature_version": "V6-unified-features-1",
                    },
                    out / "models" / f"{set_name}_{model_name}.joblib",
                    compress=3,
                )

            pd.DataFrame(thresholds).to_csv(
                out / "threshold_audit.csv", index=False
            )
            print(f"完成 {set_name}", flush=True)

    # 先保存全部固定预测及模型，再合入外部结局计算成绩。
    pred = pd.concat(external_predictions, ignore_index=True)
    pred.to_csv(out / "external_predictions_frozen.csv", index=False)
    pd.concat(source_predictions, ignore_index=True).to_csv(
        out / "source_threshold_predictions.csv", index=False
    )

    model_hashes = {
        p.name: sha_file(p)
        for p in sorted((out / "models").glob("*.joblib"))
    }
    (out / "model_hashes.json").write_text(
        json.dumps(model_hashes, indent=2), encoding="utf-8"
    )

    external_y = ext.groupby("patient_key").label.first().sort_index()
    pred["label"] = pred.patient_key.map(external_y)

    if pred.label.isna().any():
        raise ValueError("外部预测与结局ID不匹配。")

    pred["label"] = pred.label.astype(int)
    pred.to_csv(out / "external_predictions.csv", index=False)

    # 所有模型使用同一批病例级Bootstrap抽样索引，便于配对比较。
    ids = external_y.index
    y = external_y.to_numpy().astype(int)
    pos = np.flatnonzero(y == 1)
    neg = np.flatnonzero(y == 0)

    rng = np.random.default_rng(SEED + 100)
    bootstrap_indices = [
        np.concatenate([
            rng.choice(pos, len(pos), replace=True),
            rng.choice(neg, len(neg), replace=True),
        ])
        for _ in range(BOOTSTRAPS)
    ]

    metric_rows = []
    auc_boot = {}
    probability_arrays = {}

    print("\n模型已固定，开始外部评估及置信区间计算……", flush=True)

    for (set_name, model_name), block in pred.groupby(
        ["feature_set", "model"]
    ):
        if not block.patient_key.is_unique:
            raise ValueError("同一配置的外部患者预测重复。")

        block = block.set_index("patient_key").loc[ids]
        p = block.p.to_numpy()
        d = block.prediction.to_numpy().astype(int)

        if not np.isfinite(p).all() or not ((p >= 0) & (p <= 1)).all():
            raise ValueError("外部概率异常。")

        tp = int(((d == 1) & (y == 1)).sum())
        fn = int(((d == 0) & (y == 1)).sum())
        tn = int(((d == 0) & (y == 0)).sum())
        fp = int(((d == 1) & (y == 0)).sum())

        sens_ci = wilson(tp, tp + fn)
        spec_ci = wilson(tn, tn + fp)

        draws = np.array([
            fast_auc(y[idx], p[idx]) for idx in bootstrap_indices
        ])

        key = (set_name, model_name)
        auc_boot[key] = draws
        probability_arrays[key] = p

        metric_rows.append({
            "feature_set": set_name,
            "model": model_name,
            "n": len(y),
            "malignant": int(y.sum()),
            "auc": float(roc_auc_score(y, p)),
            "auc_ci_low": float(np.percentile(draws, 2.5)),
            "auc_ci_high": float(np.percentile(draws, 97.5)),
            "brier": float(brier_score_loss(y, p)),
            "sensitivity": tp / (tp + fn),
            "sensitivity_ci_low": sens_ci[0],
            "sensitivity_ci_high": sens_ci[1],
            "specificity": tn / (tn + fp),
            "specificity_ci_low": spec_ci[0],
            "specificity_ci_high": spec_ci[1],
            "missed_malignant": fn,
            "false_positive": fp,
            "threshold": float(block.threshold.iloc[0]),
        })

    metrics = pd.DataFrame(metric_rows)
    metrics.to_csv(out / "external_metrics.csv", index=False)

    # 配对比较：正值表示A组AUC更高。
    paired_rows = []

    comparisons = [
        ("base_posterior", "base_missing", "measurement_increment"),
        ("base_missing", "base", "missingness_increment"),
        ("base_posterior", "base", "total_increment"),
    ]

    for model_name in ["LR", "ensemble"]:
        for a, b, meaning in comparisons:
            difference = (
                fast_auc(y, probability_arrays[(a, model_name)])
                - fast_auc(y, probability_arrays[(b, model_name)])
            )
            draws = (
                auc_boot[(a, model_name)] - auc_boot[(b, model_name)]
            )
            paired_rows.append({
                "model": model_name,
                "comparison": meaning,
                "a": a,
                "b": b,
                "auc_difference_a_minus_b": difference,
                "ci_low": float(np.percentile(draws, 2.5)),
                "ci_high": float(np.percentile(draws, 97.5)),
                "primary": bool(
                    model_name == "LR"
                    and meaning == "measurement_increment"
                ),
            })

    paired = pd.DataFrame(paired_rows)
    paired.to_csv(out / "paired_auc_differences.csv", index=False)

    # 校准曲线的固定等宽分箱，只描述，不据此重新校准。
    calibration_rows = []
    for (set_name, model_name), block in pred.groupby(
        ["feature_set", "model"]
    ):
        bins = np.minimum((block.p.to_numpy() * 10).astype(int), 9)
        for bin_id in range(10):
            b = block.iloc[np.flatnonzero(bins == bin_id)]
            if len(b):
                calibration_rows.append({
                    "feature_set": set_name,
                    "model": model_name,
                    "bin": bin_id,
                    "n": len(b),
                    "mean_probability": float(b.p.mean()),
                    "observed_malignant_fraction": float(b.label.mean()),
                })

    pd.DataFrame(calibration_rows).to_csv(
        out / "external_calibration_bins.csv", index=False
    )

    (out / "run_summary.json").write_text(
        json.dumps({
            "source_patients": int(len(source_y)),
            "external_patients": int(len(external_y)),
            "comparisons": len(metrics),
            "seconds": round(time.time() - started, 1),
            "completed": True,
            "note": (
                "External sensitivity is evaluated, not guaranteed to be 95%. "
                "Source threshold-set performance is developmental, not a test result."
            ),
        }, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    archive = Path(shutil.make_archive(str(out), "zip", out))

    print("\n外部评估：")
    display(metrics.round(4))

    print("\n配对AUC增量：")
    display(paired.round(4))

    print("\n完成。模型、阈值、患者划分、预测和源码均已保存。")
    print("结果包：", archive)
    print("请下载ZIP发给我。不要按外部成绩重新挑种子或阈值。")
    display(FileLink(str(archive)))


main_train()

# ======================== 代码结束 ========================

In [ ]:
# ============================================================
# V6稳定性实验：只改变来源患者划分
# 固定5次：42、43、44、45、46
# 模型随机种子固定42；不搜索参数、不选择最优重复
# ============================================================

from pathlib import Path
import json
import time
import hashlib
import zipfile
import shutil
import platform
import importlib.metadata

import numpy as np
import pandas as pd
import joblib

from scipy.special import logit
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.metrics import roc_auc_score, brier_score_loss
from threadpoolctl import threadpool_limits
from IPython.display import display, FileLink

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# 留空自动寻找。若提示多个候选，再填ZIP或目录路径。
FEATURE_PATH = ""
PREVIOUS_RUN_PATH = ""

SPLIT_SEEDS = [42, 43, 44, 45, 46]
MODEL_SEED = 42
THREADS = 2
TARGET_SENSITIVITY = 0.95


def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def extract_zip(src, dest):
    dest.mkdir(parents=True, exist_ok=True)
    root = dest.resolve()

    with zipfile.ZipFile(src) as z:
        total = 0
        for info in z.infolist():
            p = (root / info.filename).resolve()
            total += info.file_size

            if p != root and root not in p.parents:
                raise ValueError("ZIP路径异常。")
            if ((info.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError("ZIP含符号链接。")
            if total > 2_000_000_000:
                raise ValueError("结果包大小异常。")

        z.extractall(root)


def locate_package(explicit, prefix, marker, exact_name):
    if explicit.strip():
        selected = Path(explicit.strip())
    else:
        exact_dir = WORK / exact_name
        if (exact_dir / marker).is_file():
            selected = exact_dir
        else:
            exact_zips = sorted(INPUT.rglob(exact_name + ".zip"))
            if len(exact_zips) == 1:
                selected = exact_zips[0]
            else:
                directories = [
                    p for p in WORK.glob(prefix + "*")
                    if p.is_dir() and (p / marker).is_file()
                ]
                directories += [
                    p.parent for p in INPUT.rglob(marker)
                    if (p.parent / "protocol.json").is_file()
                    and p.parent.name.startswith(prefix)
                ]
                candidates = sorted(set(directories))

                if not candidates:
                    candidates = sorted(INPUT.rglob(prefix + "*.zip"))

                if len(candidates) != 1:
                    print("候选：")
                    for p in candidates:
                        print(p)
                    raise RuntimeError(
                        f"请手动填写{prefix}对应的输入路径。"
                    )
                selected = candidates[0]

    if selected.is_file():
        if selected.suffix.lower() != ".zip":
            raise ValueError("请选择ZIP或结果目录。")

        cache = WORK / "v6_stability_cache" / sha_file(selected)[:16]
        extract_zip(selected, cache)

        found = [
            p.parent for p in cache.rglob(marker)
            if (p.parent / "protocol.json").is_file()
        ]
        if len(found) != 1:
            raise ValueError("ZIP中的结果目录不唯一。")
        selected = found[0]

    if not (selected / marker).is_file():
        raise FileNotFoundError(f"缺少{marker}：{selected}")

    return selected


def fixed_models():
    def prep(model, scale=False):
        steps = [
            SimpleImputer(
                strategy="median",
                keep_empty_features=True,
            )
        ]
        if scale:
            steps.append(StandardScaler())
        return make_pipeline(*steps, model)

    return {
        "LR": prep(
            LogisticRegression(
                C=1,
                max_iter=3000,
                solver="lbfgs",
                random_state=MODEL_SEED,
            ),
            scale=True,
        ),
        "ET": prep(
            ExtraTreesClassifier(
                n_estimators=200,
                min_samples_leaf=3,
                max_features=0.8,
                n_jobs=THREADS,
                random_state=MODEL_SEED,
            )
        ),
        "HGB": prep(
            HistGradientBoostingClassifier(
                max_iter=120,
                learning_rate=0.05,
                max_leaf_nodes=15,
                min_samples_leaf=15,
                l2_regularization=1,
                early_stopping=False,
                random_state=MODEL_SEED,
            )
        ),
    }


def patient_score(pipeline, frame, columns):
    p = pipeline.predict_proba(frame[columns])[:, 1]
    return (
        pd.Series(p, index=frame.patient_key.to_numpy())
        .groupby(level=0).mean().sort_index()
    )


def apply_calibration(raw, model):
    values = logit(
        np.clip(raw.to_numpy(), 1e-6, 1 - 1e-6)
    ).reshape(-1, 1)
    return pd.Series(
        model.predict_proba(values)[:, 1],
        index=raw.index,
    )


def main_stability():
    src = locate_package(
        FEATURE_PATH,
        "bus_v6_features_",
        "features.csv",
        "bus_v6_features_20261013_120420_94529",
    )
    previous = locate_package(
        PREVIOUS_RUN_PATH,
        "bus_v6_external_training_",
        "external_predictions.csv",
        "bus_v6_external_training_20261013_121259_911015",
    )

    print("特征目录：", src)
    print("原训练目录：", previous)

    old_protocol = json.loads(
        (previous / "protocol.json").read_text()
    )

    if old_protocol.get("version") != "V6-external-fixed-1":
        raise ValueError("原训练协议版本不匹配。")

    # 必须与上一轮使用完全相同的特征及清单。
    for name, expected in old_protocol["input_hashes"].items():
        if sha_file(src / name) != expected:
            raise ValueError(f"{name}与上一轮输入不一致。")

    sets = old_protocol["feature_sets"]
    if set(sets) != {"base", "base_missing", "base_posterior"}:
        raise ValueError("特征组不符合预期。")

    versions = {"python": platform.python_version()}
    for name in ["numpy", "pandas", "scipy", "scikit-learn", "joblib"]:
        versions[name] = importlib.metadata.version(name)

    changed = {
        name: (old_protocol["versions"].get(name), version)
        for name, version in versions.items()
        if old_protocol["versions"].get(name) != version
    }

    if changed:
        print("运行环境与上一轮不同：", changed)
        print("后面将严格检查seed=42能否复现。")

    meta = pd.read_csv(
        src / "cohort.csv",
        dtype={"uid": str, "patient_key": str},
    )
    feature = pd.read_csv(src / "features.csv", dtype={"uid": str})

    if (
        not meta.uid.is_unique
        or not feature.uid.is_unique
        or set(meta.uid) != set(feature.uid)
    ):
        raise ValueError("清单与特征ID不匹配。")

    data = meta.merge(feature, on="uid", validate="one_to_one")
    data["label"] = pd.to_numeric(
        data.label, errors="raise"
    ).astype(int)

    columns_all = list(dict.fromkeys(
        col for columns in sets.values() for col in columns
    ))
    for col in columns_all:
        data[col] = pd.to_numeric(data[col], errors="raise")

    if np.isinf(data[columns_all].to_numpy()).any():
        raise ValueError("特征存在无穷值。")
    if not data.groupby("patient_key").label.nunique().eq(1).all():
        raise ValueError("患者标签冲突。")

    bus = data[data.dataset == "BUS-BRA"].copy()
    ext = data[data.dataset == "BrEaST"].copy()

    if (
        len(bus) != 1875
        or bus.patient_key.nunique() != 1064
        or len(ext) != 252
        or ext.patient_key.nunique() != 252
    ):
        raise ValueError("数据量不符合预期。")

    y_source = bus.groupby("patient_key").label.first().sort_index()
    y_external = ext.groupby("patient_key").label.first().sort_index()

    old_predictions = pd.read_csv(
        previous / "external_predictions.csv",
        dtype={"patient_key": str},
    )

    run = time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns()%1000000}"
    out = WORK / ("bus_v6_stability_" + run)
    out.mkdir(parents=True)
    (out / "models").mkdir()

    protocol = {
        "version": "V6-split-stability-1",
        "split_seeds": SPLIT_SEEDS,
        "model_seed": MODEL_SEED,
        "split": "Source patient-stratified 60/20/20",
        "feature_sets": sets,
        "target_development_sensitivity": TARGET_SENSITIVITY,
        "parameter_search": False,
        "best_seed_selection": False,
        "primary_comparison": "LR: base_posterior minus base_missing",
        "external_cohort": "Same 252 BrEaST cases in every repetition",
        "interpretation": (
            "Descriptive sensitivity to source splitting, "
            "not five independent external cohorts or a new sample size."
        ),
        "ci_policy": (
            "Report across-split mean, SD and range descriptively; "
            "do not use SD/sqrt(5) as an external-performance confidence interval."
        ),
        "prior_protocol_sha256": sha_file(previous / "protocol.json"),
        "prior_input_hashes": old_protocol["input_hashes"],
        "versions": versions,
    }

    (out / "protocol.json").write_text(
        json.dumps(protocol, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    shutil.copy2(src / "v6_extractor.py", out / "v6_extractor.py")

    try:
        cell = get_ipython().history_manager.input_hist_raw[-1]
        (out / "executed_cell.py").write_text(cell, encoding="utf-8")
    except Exception:
        pass

    all_predictions = []
    all_splits = []
    all_thresholds = []
    reproducibility = []
    started = time.time()

    with threadpool_limits(limits=THREADS):
        for split_seed in SPLIT_SEEDS:
            print(
                f"\n开始患者划分seed={split_seed}；模型seed固定42",
                flush=True,
            )

            train_ids, remaining = train_test_split(
                y_source.index.to_numpy(),
                test_size=0.40,
                stratify=y_source.to_numpy(),
                random_state=split_seed,
            )
            cal_ids, gate_ids = train_test_split(
                remaining,
                test_size=0.50,
                stratify=y_source.loc[remaining].to_numpy(),
                random_state=split_seed + 1,
            )

            roles = {
                "train": sorted(train_ids),
                "calibration": sorted(cal_ids),
                "threshold": sorted(gate_ids),
            }
            groups = [set(x) for x in roles.values()]
            if (
                set.union(*groups) != set(y_source.index)
                or sum(map(len, groups)) != len(y_source)
            ):
                raise ValueError("患者划分有重叠或遗漏。")

            for role, ids in roles.items():
                for patient in ids:
                    all_splits.append({
                        "split_seed": split_seed,
                        "patient_key": patient,
                        "role": role,
                        "label": int(y_source.loc[patient]),
                    })

            frames = {
                role: bus[bus.patient_key.isin(ids)].copy()
                for role, ids in roles.items()
            }
            train = frames["train"]

            image_counts = train.groupby("patient_key").size()
            class_counts = y_source.loc[roles["train"]].value_counts()
            weights = (
                1.0 / train.patient_key.map(image_counts).to_numpy()
                / train.label.map(class_counts).to_numpy()
            )
            weights /= weights.mean()

            seed_predictions = []

            for set_name, columns in sets.items():
                models = fixed_models()
                raw = {
                    "calibration": {},
                    "threshold": {},
                    "external": {},
                }

                for name, pipeline in models.items():
                    final_step = pipeline.steps[-1][0]
                    pipeline.fit(
                        train[columns],
                        train.label.to_numpy(),
                        **{final_step + "__sample_weight": weights},
                    )

                    for role in ["calibration", "threshold"]:
                        raw[role][name] = patient_score(
                            pipeline, frames[role], columns
                        )
                    raw["external"][name] = patient_score(
                        pipeline, ext, columns
                    )

                for role in raw:
                    raw[role]["ensemble"] = pd.concat(
                        [raw[role][n] for n in ["LR", "ET", "HGB"]],
                        axis=1,
                    ).mean(axis=1)

                calibrators = {}
                thresholds = {}

                for name in ["LR", "ensemble"]:
                    cal_raw = raw["calibration"][name]

                    platt = LogisticRegression(
                        C=1000, solver="lbfgs", max_iter=2000
                    )
                    platt.fit(
                        logit(
                            np.clip(
                                cal_raw.to_numpy(), 1e-6, 1 - 1e-6
                            )
                        ).reshape(-1, 1),
                        y_source.loc[cal_raw.index].to_numpy(),
                    )

                    gate = apply_calibration(
                        raw["threshold"][name], platt
                    )
                    test = apply_calibration(
                        raw["external"][name], platt
                    )

                    positives = np.sort(
                        gate.loc[
                            y_source.loc[gate.index].eq(1)
                        ].to_numpy()
                    )
                    allowed = int(np.floor(
                        (1 - TARGET_SENSITIVITY)
                        * len(positives) + 1e-9
                    ))
                    threshold = float(positives[allowed])

                    calibrators[name] = platt
                    thresholds[name] = threshold

                    all_thresholds.append({
                        "split_seed": split_seed,
                        "feature_set": set_name,
                        "model": name,
                        "threshold": threshold,
                        "development_malignant": len(positives),
                        "development_sensitivity": float(
                            (positives >= threshold).mean()
                        ),
                        "calibration_slope": float(platt.coef_[0, 0]),
                    })

                    seed_predictions.append(pd.DataFrame({
                        "split_seed": split_seed,
                        "patient_key": test.index,
                        "feature_set": set_name,
                        "model": name,
                        "p": test.to_numpy(),
                        "threshold": threshold,
                        "prediction": (
                            test.to_numpy() >= threshold
                        ).astype(int),
                    }))

                joblib.dump(
                    {
                        "split_seed": split_seed,
                        "model_seed": MODEL_SEED,
                        "columns": columns,
                        "models": models,
                        "calibrators": calibrators,
                        "thresholds": thresholds,
                    },
                    out / "models" / f"seed{split_seed}_{set_name}.joblib",
                    compress=3,
                )
                print(f"完成 {set_name}", flush=True)

            current = pd.concat(seed_predictions, ignore_index=True)
            current.to_csv(
                out / f"predictions_frozen_seed{split_seed}.csv",
                index=False,
            )

            # 第一轮必须复现原先seed42，避免环境或实现变化混入。
            if split_seed == 42:
                keys = ["patient_key", "feature_set", "model"]
                matched = current.merge(
                    old_predictions[keys + ["p", "threshold", "prediction"]],
                    on=keys,
                    suffixes=("_new", "_old"),
                    validate="one_to_one",
                )

                ok = (
                    len(matched) == 252 * 6
                    and np.allclose(
                        matched.p_new, matched.p_old,
                        rtol=1e-7, atol=1e-9,
                    )
                    and np.allclose(
                        matched.threshold_new, matched.threshold_old,
                        rtol=1e-7, atol=1e-9,
                    )
                    and np.array_equal(
                        matched.prediction_new, matched.prediction_old
                    )
                )

                reproducibility.append({
                    "split_seed": 42,
                    "matched_rows": len(matched),
                    "max_probability_difference": float(
                        (matched.p_new - matched.p_old).abs().max()
                    ),
                    "passed": bool(ok),
                })
                pd.DataFrame(reproducibility).to_csv(
                    out / "reproducibility_check.csv", index=False
                )

                if not ok:
                    raise RuntimeError(
                        f"seed42未复现上一轮，已停止后续重复。"
                        f"请发回{out}/reproducibility_check.csv，"
                        "不要放宽容差继续跑。"
                    )

                print("seed42复现检查通过。", flush=True)

            all_predictions.append(current)

            pd.concat(all_predictions, ignore_index=True).to_csv(
                out / "external_predictions_frozen.csv", index=False
            )
            pd.DataFrame(all_thresholds).to_csv(
                out / "threshold_audit.csv", index=False
            )
            pd.DataFrame(all_splits).to_csv(
                out / "patient_splits.csv", index=False
            )

    predictions = pd.concat(all_predictions, ignore_index=True)
    predictions["label"] = predictions.patient_key.map(y_external)

    if predictions.label.isna().any():
        raise ValueError("外部病例标签匹配失败。")

    predictions["label"] = predictions.label.astype(int)
    predictions.to_csv(
        out / "external_predictions.csv", index=False
    )

    metric_rows = []

    for (seed, feature_set, model), b in predictions.groupby(
        ["split_seed", "feature_set", "model"]
    ):
        if len(b) != 252 or not b.patient_key.is_unique:
            raise ValueError("每个配置应有252例唯一外部病例。")

        y = b.label.to_numpy()
        p = b.p.to_numpy()
        d = b.prediction.to_numpy()

        tp = int(((y == 1) & (d == 1)).sum())
        fn = int(((y == 1) & (d == 0)).sum())
        tn = int(((y == 0) & (d == 0)).sum())
        fp = int(((y == 0) & (d == 1)).sum())

        metric_rows.append({
            "split_seed": seed,
            "feature_set": feature_set,
            "model": model,
            "n": len(b),
            "auc": float(roc_auc_score(y, p)),
            "brier": float(brier_score_loss(y, p)),
            "sensitivity": tp / (tp + fn),
            "specificity": tn / (tn + fp),
            "missed_malignant": fn,
            "false_positive": fp,
        })

    metrics = pd.DataFrame(metric_rows)
    metrics.to_csv(out / "metrics_by_split.csv", index=False)

    descriptive = metrics.groupby(
        ["feature_set", "model"]
    )[["auc", "brier", "sensitivity", "specificity"]].agg(
        ["mean", "std", "min", "max"]
    )
    descriptive.columns = [
        f"{metric}_{stat}" for metric, stat in descriptive.columns
    ]
    descriptive.reset_index().to_csv(
        out / "descriptive_stability.csv", index=False
    )

    # 每个划分内部配对比较，不能把5次重复当作独立外部样本。
    delta_rows = []
    indexed = metrics.set_index(
        ["split_seed", "feature_set", "model"]
    )

    for seed in SPLIT_SEEDS:
        for model in ["LR", "ensemble"]:
            for a, b, name in [
                ("base_posterior", "base_missing", "measurement_increment"),
                ("base_missing", "base", "missingness_increment"),
                ("base_posterior", "base", "total_increment"),
            ]:
                aa = indexed.loc[(seed, a, model)]
                bb = indexed.loc[(seed, b, model)]
                delta_rows.append({
                    "split_seed": seed,
                    "model": model,
                    "comparison": name,
                    "auc_difference": float(aa.auc - bb.auc),
                    # 正值表示Brier降低、得到改善。
                    "brier_improvement": float(bb.brier - aa.brier),
                    "sensitivity_difference": float(
                        aa.sensitivity - bb.sensitivity
                    ),
                    "specificity_difference": float(
                        aa.specificity - bb.specificity
                    ),
                })

    deltas = pd.DataFrame(delta_rows)
    deltas.to_csv(out / "paired_differences_by_split.csv", index=False)

    delta_summary = deltas.groupby(
        ["model", "comparison"]
    ).agg(
        auc_delta_mean=("auc_difference", "mean"),
        auc_delta_min=("auc_difference", "min"),
        auc_delta_max=("auc_difference", "max"),
        positive_auc_repeats=(
            "auc_difference", lambda x: int((x > 0).sum())
        ),
        brier_improvement_mean=("brier_improvement", "mean"),
    )
    delta_summary.to_csv(out / "paired_difference_summary.csv")

    model_hashes = {
        p.name: sha_file(p)
        for p in sorted((out / "models").glob("*.joblib"))
    }
    (out / "model_hashes.json").write_text(
        json.dumps(model_hashes, indent=2), encoding="utf-8"
    )

    (out / "run_summary.json").write_text(
        json.dumps({
            "completed": True,
            "source_patients": len(y_source),
            "external_patients": len(y_external),
            "repetitions": len(SPLIT_SEEDS),
            "configurations_per_repetition": 6,
            "seed42_reproduced": True,
            "seconds": round(time.time() - started, 1),
            "no_best_seed_selected": True,
        }, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    archive = Path(shutil.make_archive(str(out), "zip", out))

    print("\n各配置的划分稳定性（描述性统计）：")
    display(descriptive.round(4))

    print("\n配对增量：")
    display(delta_summary.round(5))

    print("\n完成。未选择最优种子，也未调整外部阈值。")
    print("结果包：", archive)
    display(FileLink(str(archive)))


main_stability()

# ======================== 代码结束 ========================

In [ ]:
# ============================================================
# V6：固定seed42的校准、分层及失败病例分析
# 不重新训练诊断模型，不修改预测、模型或阈值
# ============================================================

from pathlib import Path
import json
import time
import hashlib
import zipfile
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.special import expit, logit
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score, brier_score_loss
from IPython.display import display, FileLink

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# 一般留空；有多个候选时填写ZIP或结果目录。
FEATURE_PATH = ""
TRAINING_PATH = ""

BOOTSTRAPS = 2000
SEED = 20261013


def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def unpack(src, target):
    target.mkdir(parents=True, exist_ok=True)
    root = target.resolve()

    with zipfile.ZipFile(src) as z:
        total = 0
        for info in z.infolist():
            p = (root / info.filename).resolve()
            total += info.file_size
            if p != root and root not in p.parents:
                raise ValueError("ZIP路径异常。")
            if ((info.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError("ZIP包含符号链接。")
            if total > 2_000_000_000:
                raise ValueError("结果ZIP异常大。")
        z.extractall(root)


def locate_package(explicit, exact_name, prefix, marker):
    if explicit.strip():
        selected = Path(explicit.strip())
    else:
        p = WORK / exact_name
        if (p / marker).is_file():
            selected = p
        else:
            found = sorted(INPUT.rglob(exact_name + ".zip"))
            if len(found) == 1:
                selected = found[0]
            else:
                candidates = [
                    p for p in WORK.glob(prefix + "*")
                    if p.is_dir() and (p / marker).is_file()
                ]
                candidates += [
                    p.parent for p in INPUT.rglob(marker)
                    if p.parent.name.startswith(prefix)
                    and (p.parent / "protocol.json").is_file()
                ]
                candidates = sorted(set(candidates))
                if not candidates:
                    candidates = sorted(INPUT.rglob(prefix + "*.zip"))
                if len(candidates) != 1:
                    print("候选：", candidates)
                    raise RuntimeError("请填写对应的输入路径。")
                selected = candidates[0]

    if selected.is_file():
        if selected.suffix.lower() != ".zip":
            raise ValueError("输入应为ZIP或结果目录。")
        target = WORK / "v6_analysis_cache" / sha_file(selected)[:16]
        unpack(selected, target)
        found = [
            p.parent for p in target.rglob(marker)
            if (p.parent / "protocol.json").is_file()
        ]
        if len(found) != 1:
            raise ValueError("结果ZIP中的目录不唯一。")
        selected = found[0]

    if not (selected / marker).is_file():
        raise FileNotFoundError(f"缺少 {marker}：{selected}")
    return selected


def wilson(k, n):
    if n == 0:
        return np.nan, np.nan
    z = 1.959963984540054
    p = k / n
    den = 1 + z * z / n
    center = (p + z * z / (2 * n)) / den
    radius = z * np.sqrt(
        p * (1 - p) / n + z * z / (4 * n * n)
    ) / den
    return float(center - radius), float(center + radius)


def auc_from_scores(y, p):
    """采用排序计算，妥善处理并列分数。"""
    y = np.asarray(y, dtype=int)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if not n1 or not n0:
        return np.nan
    ranks = pd.Series(p).rank(method="average").to_numpy()
    return float(
        (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
    )


def auc_interval(y, p):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    pos = np.flatnonzero(y == 1)
    neg = np.flatnonzero(y == 0)

    if min(len(pos), len(neg)) < 2:
        return np.nan, np.nan

    rng = np.random.default_rng(SEED)
    draws = []
    for _ in range(BOOTSTRAPS):
        ix = np.concatenate([
            rng.choice(pos, len(pos), replace=True),
            rng.choice(neg, len(neg), replace=True),
        ])
        draws.append(auc_from_scores(y[ix], p[ix]))

    return (
        float(np.percentile(draws, 2.5)),
        float(np.percentile(draws, 97.5)),
    )


def diagnostic_calibration(y, p):
    """
    仅估计校准诊断量，绝不将拟合结果用于改写预测。
    CITL：斜率固定1时估计截距。
    slope：同时估计截距与斜率时的斜率。
    """
    y = np.asarray(y, dtype=float)
    x = logit(np.clip(np.asarray(p), 1e-6, 1 - 1e-6))

    def fit(design, offset, initial):
        def objective(beta):
            eta = offset + design @ beta
            loss = np.mean(np.logaddexp(0, eta) - y * eta)
            grad = design.T @ (expit(eta) - y) / len(y)
            return loss, grad

        res = minimize(
            objective,
            np.asarray(initial, dtype=float),
            method="BFGS",
            jac=True,
            options={"maxiter": 2000, "gtol": 1e-8},
        )

        _, grad = objective(res.x)
        ok = (
            np.isfinite(res.x).all()
            and np.max(np.abs(grad)) < 1e-5
        )
        return res.x, bool(ok), str(res.message)

    intercept, ok1, message1 = fit(
        np.ones((len(y), 1)), x, [0.0]
    )
    joint, ok2, message2 = fit(
        np.column_stack([np.ones(len(y)), x]),
        np.zeros(len(y)),
        [0.0, 1.0],
    )

    return {
        "calibration_in_large": float(intercept[0]) if ok1 else np.nan,
        "joint_intercept": float(joint[0]) if ok2 else np.nan,
        "calibration_slope": float(joint[1]) if ok2 else np.nan,
        "citl_fit_ok": ok1,
        "slope_fit_ok": ok2,
        "citl_fit_message": message1,
        "slope_fit_message": message2,
    }


def main_analysis():
    feature_root = locate_package(
        FEATURE_PATH,
        "bus_v6_features_20261013_120420_94529",
        "bus_v6_features_",
        "features.csv",
    )
    training_root = locate_package(
        TRAINING_PATH,
        "bus_v6_external_training_20261013_121259_911015",
        "bus_v6_external_training_",
        "external_predictions.csv",
    )

    print("特征目录：", feature_root)
    print("固定训练结果：", training_root)

    training_protocol = json.loads(
        (training_root / "protocol.json").read_text()
    )
    if (
        training_protocol.get("version") != "V6-external-fixed-1"
        or training_protocol.get("seed") != 42
    ):
        raise ValueError("必须使用最初固定的V6 seed42结果。")

    for name, expected in training_protocol["input_hashes"].items():
        if sha_file(feature_root / name) != expected:
            raise ValueError(f"{name}与原训练输入不一致。")

    pred = pd.read_csv(
        training_root / "external_predictions.csv",
        dtype={"patient_key": str},
    )
    frozen = pd.read_csv(
        training_root / "external_predictions_frozen.csv",
        dtype={"patient_key": str},
    )
    keys = ["patient_key", "feature_set", "model"]

    if pred.duplicated(keys).any() or frozen.duplicated(keys).any():
        raise ValueError("发现重复预测。")

    check = pred.merge(
        frozen,
        on=keys,
        suffixes=("_eval", "_frozen"),
        validate="one_to_one",
    )

    if len(check) != 1512:
        raise ValueError("应有6组×252例预测。")

    for col in ["p", "threshold", "prediction"]:
        if not np.array_equal(
            check[col + "_eval"].to_numpy(),
            check[col + "_frozen"].to_numpy(),
        ):
            raise ValueError(f"{col}与冻结预测不一致。")

    meta = pd.read_csv(
        feature_root / "cohort.csv",
        dtype={"uid": str, "patient_key": str},
    )
    feature = pd.read_csv(
        feature_root / "features.csv", dtype={"uid": str}
    )

    external = meta[meta.dataset == "BrEaST"].copy()
    external = external.merge(
        feature[["uid", "postdepth__missing", "postdepth__delta"]],
        on="uid",
        validate="one_to_one",
    )

    if len(external) != 252 or not external.patient_key.is_unique:
        raise ValueError("外部清单不是252例唯一病例。")

    external["availability"] = np.where(
        external.postdepth__missing.eq(0),
        "measurable",
        "unmeasurable",
    )

    pred = pred.merge(
        external[
            ["patient_key", "uid", "label", "availability",
             "postdepth__delta"]
        ].rename(columns={"label": "manifest_label"}),
        on="patient_key",
        validate="many_to_one",
    )

    if not np.array_equal(pred.label, pred.manifest_label):
        raise ValueError("预测标签与清单不一致。")

    if not np.array_equal(
        pred.prediction.to_numpy(),
        (pred.p >= pred.threshold).astype(int).to_numpy(),
    ):
        raise ValueError("分类结果与固定阈值不一致。")

    run = time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns()%1000000}"
    out = WORK / ("bus_v6_calibration_failures_" + run)
    out.mkdir(parents=True)

    try:
        cell = get_ipython().history_manager.input_hist_raw[-1]
        (out / "executed_cell.py").write_text(cell, encoding="utf-8")
    except Exception:
        pass

    # ---------- 1. 可测量性及病例构成 ----------
    availability = external.groupby("availability").agg(
        n=("patient_key", "size"),
        malignant=("label", "sum"),
    )
    availability["benign"] = availability.n - availability.malignant
    availability["malignant_fraction"] = (
        availability.malignant / availability.n
    )
    availability.to_csv(out / "availability_composition.csv")

    # ---------- 2. 全体及分层表现，沿用原阈值 ----------
    metric_rows = []

    for (feature_set, model), block in pred.groupby(
        ["feature_set", "model"]
    ):
        groups = [("all", block)]
        groups += list(block.groupby("availability"))

        for subgroup, b in groups:
            y = b.label.to_numpy().astype(int)
            p = b.p.to_numpy()
            d = b.prediction.to_numpy().astype(int)

            tp = int(((y == 1) & (d == 1)).sum())
            fn = int(((y == 1) & (d == 0)).sum())
            tn = int(((y == 0) & (d == 0)).sum())
            fp = int(((y == 0) & (d == 1)).sum())

            sens_ci = wilson(tp, tp + fn)
            spec_ci = wilson(tn, tn + fp)
            auc_ci = auc_interval(y, p)

            metric_rows.append({
                "feature_set": feature_set,
                "model": model,
                "subgroup": subgroup,
                "n": len(b),
                "malignant": int(y.sum()),
                "auc": auc_from_scores(y, p),
                "auc_ci_low": auc_ci[0],
                "auc_ci_high": auc_ci[1],
                "brier": float(brier_score_loss(y, p)),
                "sensitivity": tp / (tp + fn) if tp + fn else np.nan,
                "sensitivity_ci_low": sens_ci[0],
                "sensitivity_ci_high": sens_ci[1],
                "specificity": tn / (tn + fp) if tn + fp else np.nan,
                "specificity_ci_low": spec_ci[0],
                "specificity_ci_high": spec_ci[1],
                "missed_malignant": fn,
                "false_positive": fp,
                "mean_probability": float(p.mean()),
                "observed_malignant_fraction": float(y.mean()),
            })

    metrics = pd.DataFrame(metric_rows)
    metrics.to_csv(out / "overall_and_subgroup_metrics.csv", index=False)

    # ---------- 3. 校准诊断：不应用回模型 ----------
    calibration_rows = []
    bin_rows = []
    figure, axes = plt.subplots(2, 3, figsize=(13, 8))

    configs = [
        (fs, model)
        for model in ["LR", "ensemble"]
        for fs in ["base", "base_missing", "base_posterior"]
    ]

    for ax, (feature_set, model) in zip(axes.ravel(), configs):
        b = pred[
            (pred.feature_set == feature_set)
            & (pred.model == model)
        ].copy()
        y = b.label.to_numpy().astype(int)
        p = b.p.to_numpy()

        fitted = diagnostic_calibration(y, p)

        # 固定10个等宽分箱，ECE只作描述性指标。
        bin_id = np.minimum((p * 10).astype(int), 9)
        ece = 0.0
        plot_rows = []

        for k in range(10):
            selected = bin_id == k
            n = int(selected.sum())
            if n == 0:
                continue

            mean_p = float(p[selected].mean())
            observed = float(y[selected].mean())
            low, high = wilson(int(y[selected].sum()), n)
            ece += n / len(y) * abs(mean_p - observed)

            row = {
                "feature_set": feature_set,
                "model": model,
                "bin": k,
                "n": n,
                "mean_probability": mean_p,
                "observed_fraction": observed,
                "observed_ci_low": low,
                "observed_ci_high": high,
            }
            bin_rows.append(row)
            plot_rows.append(row)

        calibration_rows.append({
            "feature_set": feature_set,
            "model": model,
            "n": len(y),
            "mean_probability": float(p.mean()),
            "observed_malignant_fraction": float(y.mean()),
            # 正值表示平均预测概率偏高。
            "mean_prediction_minus_observed": float(p.mean() - y.mean()),
            "brier": float(brier_score_loss(y, p)),
            "ece_10_equal_width_bins": float(ece),
            **fitted,
        })

        plotted = pd.DataFrame(plot_rows)
        ax.plot([0, 1], [0, 1], "--", color="gray")
        ax.errorbar(
            plotted.mean_probability,
            plotted.observed_fraction,
            yerr=np.vstack([
                np.maximum(
                    0,
                    plotted.observed_fraction - plotted.observed_ci_low,
                ),
                np.maximum(
                    0,
                    plotted.observed_ci_high - plotted.observed_fraction,
                ),
            ]),
            fmt="o-",
            markersize=4,
            capsize=2,
        )
        ax.set(
            xlim=(0, 1), ylim=(0, 1),
            xlabel="Mean predicted probability",
            ylabel="Observed malignant fraction",
            title=f"{feature_set} / {model}",
        )
        ax.grid(alpha=0.2)

    figure.tight_layout()
    figure.savefig(out / "calibration_curves.png", dpi=180)
    plt.show()
    plt.close(figure)

    calibration = pd.DataFrame(calibration_rows)
    calibration.to_csv(out / "calibration_diagnostics.csv", index=False)
    pd.DataFrame(bin_rows).to_csv(
        out / "calibration_bins.csv", index=False
    )

    # ---------- 4. 病例级错误表 ----------
    pred["error_type"] = np.select(
        [
            (pred.label == 1) & (pred.prediction == 0),
            (pred.label == 0) & (pred.prediction == 1),
        ],
        ["false_negative", "false_positive"],
        default="correct",
    )
    pred.to_csv(out / "all_case_predictions.csv", index=False)

    pred[pred.error_type != "correct"].to_csv(
        out / "errors_by_case.csv", index=False
    )

    pred[
        (pred.label == 1) & (pred.prediction == 0)
    ].to_csv(out / "missed_malignant_cases.csv", index=False)

    # ---------- 5. 配对错误变化 ----------
    # 所有方案使用各自在BUS-BRA上确定的阈值，
    # 所以变化同时反映概率与来源阈值差异，不能归因于单一特征。
    transition_rows = []
    transition_cases = []

    comparisons = [
        ("base_missing", "base_posterior", "measurement_addition"),
        ("base", "base_posterior", "total_addition"),
    ]

    for model in ["LR", "ensemble"]:
        for before, after, comparison in comparisons:
            a = pred[
                (pred.model == model) & (pred.feature_set == before)
            ][
                ["patient_key", "label", "availability", "p",
                 "threshold", "prediction"]
            ]
            b = pred[
                (pred.model == model) & (pred.feature_set == after)
            ][["patient_key", "p", "threshold", "prediction"]]

            paired = a.merge(
                b, on="patient_key", suffixes=("_before", "_after"),
                validate="one_to_one",
            )

            for label, label_name in [(1, "malignant"), (0, "benign")]:
                subset = paired[paired.label == label].copy()
                was_wrong = subset.prediction_before != label
                now_wrong = subset.prediction_after != label

                corrected = int((was_wrong & ~now_wrong).sum())
                introduced = int((~was_wrong & now_wrong).sum())
                discordant = corrected + introduced

                # 精确配对检验，仅作探索性结果，不作多重比较校正。
                p_value = (
                    float(
                        binomtest(
                            corrected, discordant, p=0.5,
                            alternative="two-sided",
                        ).pvalue
                    )
                    if discordant else 1.0
                )

                transition_rows.append({
                    "model": model,
                    "comparison": comparison,
                    "before": before,
                    "after": after,
                    "class": label_name,
                    "n": len(subset),
                    "corrected_errors": corrected,
                    "introduced_errors": introduced,
                    "wrong_in_both": int((was_wrong & now_wrong).sum()),
                    "correct_in_both": int((~was_wrong & ~now_wrong).sum()),
                    "exploratory_exact_p_unadjusted": p_value,
                })

                subset["transition"] = np.select(
                    [
                        was_wrong & ~now_wrong,
                        ~was_wrong & now_wrong,
                        was_wrong & now_wrong,
                    ],
                    ["corrected", "introduced", "wrong_in_both"],
                    default="correct_in_both",
                )
                subset["model"] = model
                subset["comparison"] = comparison
                transition_cases.append(subset)

    transitions = pd.DataFrame(transition_rows)
    transitions.to_csv(out / "paired_error_transitions.csv", index=False)
    pd.concat(transition_cases, ignore_index=True).to_csv(
        out / "paired_case_transitions.csv", index=False
    )

    protocol = {
        "version": "V6-calibration-failure-analysis-1",
        "source_prediction_seed": 42,
        "diagnostic_model_retraining": False,
        "threshold_changes": False,
        "calibration_diagnostic_fit": (
            "Fit intercept/slope only to describe calibration; "
            "never apply them to predictions"
        ),
        "availability_definition": "V6 postdepth__missing == 0",
        "subgroups": "Exploratory measurable/unmeasurable comparisons",
        "auc_ci": "2000 class-stratified case bootstrap replicates",
        "sensitivity_specificity_ci": "Wilson 95%",
        "calibration_curve": "Fixed 10 equal-width bins; Wilson intervals",
        "paired_error_test": "Exploratory exact McNemar/binomial test, unadjusted",
        "important_limits": [
            "Subgroups may differ in case mix; not causal comparisons",
            "Repeated configurations use the same external patients",
            "Calibration slope/intercept are point estimates",
            "ECE depends on binning and is descriptive",
            "Error transitions use model-specific source-developed thresholds",
            "No model selection or tuning from these results",
        ],
        "input_hashes": {
            "external_predictions.csv": sha_file(
                training_root / "external_predictions.csv"
            ),
            "features.csv": sha_file(feature_root / "features.csv"),
            "cohort.csv": sha_file(feature_root / "cohort.csv"),
        },
    }

    (out / "protocol.json").write_text(
        json.dumps(protocol, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    archive = Path(shutil.make_archive(str(out), "zip", out))

    print("\n可测量性与病例构成：")
    display(availability)

    print("\n校准诊断：")
    display(calibration[
        [
            "feature_set", "model",
            "mean_prediction_minus_observed",
            "brier", "calibration_in_large",
            "calibration_slope",
            "citl_fit_ok", "slope_fit_ok",
        ]
    ].round(4))

    print("\n分层表现：")
    display(metrics[
        metrics.subgroup != "all"
    ][
        [
            "feature_set", "model", "subgroup", "n", "malignant",
            "auc", "sensitivity", "specificity",
            "missed_malignant",
        ]
    ].round(4))

    print("\n配对错误变化：")
    display(transitions)

    print("\n完成。未重新训练诊断模型，也未修改任何阈值。")
    print("结果包：", archive)
    display(FileLink(str(archive)))


main_analysis()

# ======================== 代码结束 ========================

In [ ]:
# ============================================================
# BUS-UCLM：第二外部数据集接入审计
# 不训练、不预测、不改变冻结模型
# ============================================================

from pathlib import Path
import os
import re
import json
import time
import hashlib
import shutil

import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi
from IPython.display import display, FileLink

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# 一般留空，自动寻找包含images和masks的BUS-UCLM目录。
# 有多个候选时，填写其中一份完整数据的目录。
UCLM_ROOT = ""

# 一般留空，自动寻找之前V6提取保存的resolved_files.csv。
# 用于与BUS-BRA和BrEaST检查完全重复的灰度图。
V6_RESOLVED_PATH = ""


def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def normalized(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


def find_uclm():
    start = Path(UCLM_ROOT.strip()) if UCLM_ROOT.strip() else INPUT
    candidates = []

    for directory, dirs, _ in os.walk(start, followlinks=True):
        root = Path(directory)
        if (
            not UCLM_ROOT.strip()
            and "busuclm" not in normalized(str(root))
        ):
            continue

        mapping = {name.lower(): name for name in dirs}
        if "images" in mapping and "masks" in mapping:
            image_dir = root / mapping["images"]
            mask_dir = root / mapping["masks"]
            n = len([
                p for p in image_dir.iterdir()
                if p.is_file() and p.suffix.lower() == ".png"
            ])
            candidates.append((root, image_dir, mask_dir, n))

    complete = [item for item in candidates if item[3] == 683]

    if len(complete) == 1:
        return complete[0]

    print("找到的候选目录及原图数量：")
    for root, _, _, n in candidates:
        print(root, "PNG原图：", n)

    raise RuntimeError(
        "需要一份包含683张原图的BUS-UCLM完整目录。"
        "请确认下载已完成；多个完整目录时填写UCLM_ROOT。"
        "若上传的是未解压ZIP，请先解压或把目录列表发给我。"
    )


def find_v6_reference():
    if V6_RESOLVED_PATH.strip():
        paths = [Path(V6_RESOLVED_PATH.strip())]
    else:
        paths = list(
            WORK.glob("bus_v6_features_*/resolved_files.csv")
        )
        paths += [
            p for p in INPUT.rglob("resolved_files.csv")
            if (p.parent / "v6_extractor.py").is_file()
        ]

    unique = {}
    for path in paths:
        if path.is_file():
            unique.setdefault(sha_file(path), path)

    if len(unique) == 1:
        return next(iter(unique.values()))

    if len(unique) > 1:
        print("存在不同版本的V6图像清单：")
        for path in unique.values():
            print(path)

    return None


def main_uclm_audit():
    root, image_dir, mask_dir, _ = find_uclm()
    print("BUS-UCLM目录：", root)

    images = {
        p.name: p
        for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() == ".png"
    }
    masks = {
        p.name: p
        for p in mask_dir.iterdir()
        if p.is_file() and p.suffix.lower() == ".png"
    }

    if len(images) != 683 or set(images) != set(masks):
        print("原图数：", len(images), "掩膜数：", len(masks))
        print("缺失掩膜：", sorted(set(images) - set(masks))[:20])
        print("多余掩膜：", sorted(set(masks) - set(images))[:20])
        raise RuntimeError("原图与掩膜不完整或不同名，先停止。")

    run = time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns()%1000000}"
    out = WORK / ("bus_uclm_audit_" + run)
    out.mkdir(parents=True)

    records = []
    errors = []
    color_counts = {}

    for i, name in enumerate(sorted(images)):
        ip, mp = images[name], masks[name]
        record = {
            "image_id": ip.stem,
            "image_path": str(ip),
            "mask_path": str(mp),
        }

        try:
            # 依据作者代码采用前4字符患者ID，同时核对命名格式。
            if not re.fullmatch(r"[A-Za-z]{4}_\d+", ip.stem):
                raise ValueError(
                    f"文件名不符合四字母患者ID_图像编号格式：{name}"
                )

            record["patient_id"] = ip.stem[:4]

            with Image.open(ip) as im:
                gray = np.asarray(im.convert("L"), dtype=np.uint8)
                alpha = np.asarray(im.convert("RGBA"))[:, :, 3]

            with Image.open(mp) as im:
                rgb = np.asarray(im.convert("RGB"))
                mask_alpha = np.asarray(im.convert("RGBA"))[:, :, 3]

            if rgb.shape[:2] != gray.shape:
                raise ValueError("原图与掩膜尺寸不一致。")

            colors, counts = np.unique(
                rgb.reshape(-1, 3), axis=0, return_counts=True
            )
            for color, count in zip(colors, counts):
                key = ",".join(map(str, color.tolist()))
                color_counts[key] = color_counts.get(key, 0) + int(count)

            # 严格核对RGB语义，不把颜色作为预测特征。
            black = np.all(rgb == [0, 0, 0], axis=2)
            green = np.all(rgb == [0, 255, 0], axis=2)
            red = np.all(rgb == [255, 0, 0], axis=2)
            unknown = ~(black | green | red)

            if unknown.any():
                raise ValueError(
                    f"掩膜含非黑/纯绿/纯红像素：{int(unknown.sum())}个。"
                    "保留检查，不猜测颜色映射。"
                )

            if np.any((green | red) & (mask_alpha == 0)):
                raise ValueError("彩色病灶像素位于透明掩膜区域。")

            lesion = green | red
            if np.any(lesion & (alpha == 0)):
                raise ValueError("病灶掩膜覆盖原图透明区域。")

            has_benign = bool(green.any())
            has_malignant = bool(red.any())

            if has_benign and has_malignant:
                category = "mixed"
                label = np.nan
            elif has_benign:
                category = "benign"
                label = 0
            elif has_malignant:
                category = "malignant"
                label = 1
            else:
                category = "normal"
                label = np.nan

            # 8邻域连通域仅作结构审计，不自行拆成独立患者。
            _, components = ndi.label(
                lesion, structure=np.ones((3, 3), dtype=int)
            )

            ph = hashlib.sha256()
            ph.update(str(gray.shape).encode())
            ph.update(gray.tobytes())

            record.update({
                "category": category,
                "label": label,
                "width": int(gray.shape[1]),
                "height": int(gray.shape[0]),
                "benign_pixels": int(green.sum()),
                "malignant_pixels": int(red.sum()),
                "lesion_components": int(components),
                "valid_pixel_fraction": float((alpha > 0).mean()),
                "image_sha256": sha_file(ip),
                "mask_sha256": sha_file(mp),
                "decoded_gray_sha256": ph.hexdigest(),
                "status": "checked",
            })

        except Exception as exc:
            record["status"] = "error"
            errors.append({"image_id": ip.stem, "error": str(exc)})

        records.append(record)

        if (i + 1) % 100 == 0:
            print(f"已检查 {i+1}/683", flush=True)

    manifest = pd.DataFrame(records)
    manifest.to_csv(out / "uclm_manifest.csv", index=False)

    pd.DataFrame(
        errors, columns=["image_id", "error"]
    ).to_csv(out / "errors.csv", index=False)

    (out / "mask_color_counts.json").write_text(
        json.dumps(color_counts, indent=2), encoding="utf-8"
    )

    good = manifest[manifest.status == "checked"].copy()

    if len(good):
        patient_summary = pd.crosstab(
            good.patient_id, good.category
        ).reindex(
            columns=["normal", "benign", "malignant", "mixed"],
            fill_value=0,
        )
        patient_summary["has_both_benign_and_malignant_images"] = (
            (patient_summary.benign > 0)
            & (patient_summary.malignant > 0)
        )
        patient_summary.to_csv(out / "patient_label_structure.csv")

        duplicate_rows = good[
            good.duplicated("decoded_gray_sha256", keep=False)
        ].copy()
    else:
        patient_summary = pd.DataFrame()
        duplicate_rows = pd.DataFrame()

    duplicate_rows.to_csv(
        out / "within_dataset_exact_duplicates.csv", index=False
    )

    # 保存元数据表信息，不凭未知列名自动推断患者标签。
    metadata_report = []
    metadata_dir = out / "metadata"
    metadata_dir.mkdir()

    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in {
            ".csv", ".xlsx", ".txt"
        }:
            continue

        item = {
            "path": str(p),
            "sha256": sha_file(p),
            "bytes": p.stat().st_size,
        }
        try:
            if p.suffix.lower() == ".csv":
                table = pd.read_csv(p, sep=None, engine="python")
                item["columns"] = [str(c) for c in table.columns]
                item["rows"] = len(table)
            elif p.suffix.lower() == ".xlsx":
                book = pd.read_excel(p, sheet_name=None)
                item["sheets"] = {
                    str(k): {
                        "rows": len(v),
                        "columns": [str(c) for c in v.columns],
                    }
                    for k, v in book.items()
                }

            # 小型元数据原样保留，方便后续核实多普勒/拼接标记。
            if p.stat().st_size <= 5_000_000:
                target = metadata_dir / (
                    item["sha256"][:8] + "_" + p.name
                )
                shutil.copy2(p, target)

        except Exception as exc:
            item["read_error"] = str(exc)

        metadata_report.append(item)

    (out / "metadata_inventory.json").write_text(
        json.dumps(metadata_report, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    reference = find_v6_reference()
    cross = pd.DataFrame()
    overlap_checked = False

    if reference is not None and len(good):
        old = pd.read_csv(reference)
        required = {"decoded_gray_sha256", "dataset", "image_id"}
        if not required <= set(old.columns):
            raise ValueError("V6原图清单缺少跨数据集核对列。")

        cross = good.merge(
            old,
            on="decoded_gray_sha256",
            suffixes=("_uclm", "_previous"),
        )
        overlap_checked = True

    cross.to_csv(out / "cross_dataset_exact_duplicates.csv", index=False)

    actual_counts = (
        good.category.value_counts().to_dict() if len(good) else {}
    )
    expected_counts = {"normal": 419, "benign": 174, "malignant": 90}

    counts_match = all(
        actual_counts.get(k, 0) == v
        for k, v in expected_counts.items()
    ) and actual_counts.get("mixed", 0) == 0

    summary = {
        "expected_images": 683,
        "checked_images": len(good),
        "errors": len(errors),
        "unique_patient_ids": (
            int(good.patient_id.nunique()) if len(good) else 0
        ),
        "image_category_counts": actual_counts,
        "matches_published_image_counts": counts_match,
        "patients_with_lesion_images": (
            int(
                good[good.category.isin(["benign", "malignant", "mixed"])]
                .patient_id.nunique()
            ) if len(good) else 0
        ),
        "patients_with_both_benign_and_malignant_images": (
            int(
                patient_summary[
                    "has_both_benign_and_malignant_images"
                ].sum()
            ) if len(patient_summary) else 0
        ),
        "images_with_multiple_lesion_components": (
            int((good.lesion_components > 1).sum())
            if len(good) else 0
        ),
        "within_dataset_duplicate_rows": len(duplicate_rows),
        "cross_dataset_overlap_checked": overlap_checked,
        "cross_dataset_overlap_rows": len(cross),
        "reference_file": str(reference) if reference else None,
        "training_or_prediction_performed": False,
        "basic_structure_passed": bool(
            len(good) == 683
            and not errors
            and good.patient_id.nunique() == 38
            and counts_match
        ),
        "note": (
            "Structure audit only. Multi-lesion handling, metadata and "
            "duplicates must be reviewed before prediction."
        ),
    }

    protocol = {
        "dataset": "BUS-UCLM",
        "intended_version": 3,
        "version_verified_from_archive_checksum": False,
        "patient_id_rule": "First four filename characters, per author code",
        "pairing": "Same filename in images and masks",
        "mask_colors": {
            "0,0,0": "background",
            "0,255,0": "benign",
            "255,0,0": "malignant",
        },
        "normal_images": "Retained in audit, not benign/malignant classification",
        "mixed_images": "Flagged, not assigned a single guessed label",
        "multiple_components": "Flagged; no independent-patient assumption",
        "duplicate_check": "Exact decoded grayscale pixels and dimensions",
        "limitations": [
            "Exact hashes do not detect all near duplicates",
            "Metadata has not yet been used to select images",
            "Filename/structure checks alone do not prove dataset version",
        ],
    }

    for name, content in [
        ("summary.json", summary),
        ("protocol.json", protocol),
    ]:
        (out / name).write_text(
            json.dumps(content, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    try:
        cell = get_ipython().history_manager.input_hist_raw[-1]
        (out / "executed_cell.py").write_text(cell, encoding="utf-8")
    except Exception:
        pass

    archive = Path(shutil.make_archive(str(out), "zip", out))

    print("\n检查结果：")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print("\n元数据表信息：")
    print(json.dumps(metadata_report, ensure_ascii=False, indent=2))

    print("\n结果包：", archive)
    print("请下载ZIP发回来。此阶段未运行模型，不需要人工逐张检查。")
    display(FileLink(str(archive)))


main_uclm_audit()

# ======================== 代码结束 ========================

In [ ]:
# ============================================================
# BUS-UCLM：针对性异常检查
# 不修改数据、不删除病例、不训练、不预测
# ============================================================

from pathlib import Path
import os
import json
import time
import hashlib
import zipfile
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage as ndi
from IPython.display import display, FileLink

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# 一般留空。也可填写之前的bus_uclm_audit_*.zip或结果目录。
AUDIT_PATH = ""


def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def safe_extract(src, target):
    target.mkdir(parents=True, exist_ok=True)
    root = target.resolve()

    with zipfile.ZipFile(src) as z:
        total = 0
        for info in z.infolist():
            dest = (root / info.filename).resolve()
            total += info.file_size

            if dest != root and root not in dest.parents:
                raise ValueError("ZIP包含异常路径。")

            if ((info.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError("ZIP包含符号链接。")

            if total > 1_000_000_000:
                raise ValueError("检查结果ZIP异常大。")

        z.extractall(root)


def find_audit():
    if AUDIT_PATH.strip():
        p = Path(AUDIT_PATH.strip())
    else:
        exact = WORK / "bus_uclm_audit_20261013_124907_539742"
        if (exact / "uclm_manifest.csv").is_file():
            p = exact
        else:
            directories = [
                x for x in WORK.glob("bus_uclm_audit_*")
                if x.is_dir() and (x / "uclm_manifest.csv").is_file()
            ]

            if len(directories) == 1:
                p = directories[0]
            else:
                archives = list(INPUT.rglob("bus_uclm_audit_*.zip"))
                if len(archives) != 1:
                    print("目录候选：", directories)
                    print("ZIP候选：", archives)
                    raise RuntimeError(
                        "请在AUDIT_PATH指定上一轮检查结果。"
                    )
                p = archives[0]

    if p.is_file():
        if p.suffix.lower() != ".zip":
            raise ValueError("AUDIT_PATH应为ZIP或结果目录。")

        target = WORK / "uclm_detail_cache" / sha_file(p)[:16]
        safe_extract(p, target)

        found = list(target.rglob("uclm_manifest.csv"))
        if len(found) != 1:
            raise ValueError("结果包中的清单不唯一。")

        p = found[0].parent

    if not (p / "uclm_manifest.csv").is_file():
        raise FileNotFoundError("找不到uclm_manifest.csv。")

    return p


def main_uclm_detail():
    audit = find_audit()
    print("使用审计结果：", audit)

    manifest = pd.read_csv(
        audit / "uclm_manifest.csv",
        dtype={"image_id": str, "patient_id": str},
    )

    if len(manifest) != 683 or not manifest.image_id.is_unique:
        raise ValueError("需要683张图像、ID唯一的原始检查清单。")

    # 建立索引，仅在旧路径失效时使用。
    index = {}
    for directory, _, files in os.walk(INPUT, followlinks=True):
        for name in files:
            if name.lower().endswith(".png"):
                index.setdefault(name, []).append(
                    (Path(directory) / name).resolve()
                )

    def locate(saved, kind, expected_hash=None):
        p = Path(saved)

        if p.is_file():
            candidates = [p]
        else:
            candidates = sorted(set(index.get(p.name, [])))
            candidates = [
                q for q in candidates
                if q.parent.name.lower() == kind
            ]

        if (
            expected_hash is not None
            and pd.notna(expected_hash)
            and str(expected_hash).strip()
        ):
            candidates = [
                q for q in candidates
                if sha_file(q) == str(expected_hash)
            ]

        if len(candidates) > 1:
            if len({sha_file(q) for q in candidates}) == 1:
                candidates = candidates[:1]

        if len(candidates) != 1:
            raise FileNotFoundError(
                f"{kind}/{p.name}无法唯一匹配：{candidates}"
            )

        return candidates[0]

    def load_case(row):
        ip = locate(
            row["image_path"], "images",
            row.get("image_sha256"),
        )
        mp = locate(
            row["mask_path"], "masks",
            row.get("mask_sha256"),
        )

        with Image.open(ip) as im:
            image = np.asarray(im.convert("RGB"))
            image_info = {
                "image_width": im.width,
                "image_height": im.height,
                "image_mode": im.mode,
                "image_exif_orientation": im.getexif().get(274),
            }

        with Image.open(mp) as im:
            mask = np.asarray(im.convert("RGB"))
            mask_info = {
                "mask_width": im.width,
                "mask_height": im.height,
                "mask_mode": im.mode,
                "mask_exif_orientation": im.getexif().get(274),
            }

        return image, mask, ip, mp, {**image_info, **mask_info}

    # 使用审计结果中已经保存的INFO.csv。
    tables = list((audit / "metadata").glob("*INFO.csv"))
    if len(tables) != 1:
        raise RuntimeError("需要审计结果包中唯一的INFO.csv。")

    info = pd.read_csv(tables[0], sep=None, engine="python")
    info.columns = [
        str(c).lstrip("\ufeff").strip() for c in info.columns
    ]

    required = {"Image", "Label", "Doppler", "Marks", "Combined"}
    if not required <= set(info.columns):
        raise ValueError(f"INFO.csv缺少列：{required-set(info.columns)}")

    info["image_id"] = info["Image"].astype(str).map(
        lambda x: Path(x).stem
    )

    if not info.image_id.is_unique:
        raise ValueError("INFO.csv存在重复图像ID。")

    if set(info.image_id) != set(manifest.image_id):
        raise ValueError("INFO.csv与图像清单ID不一致。")

    def parse_flag(value):
        v = str(value).strip().lower()
        if v in {"yes", "y"}:
            return True
        if v in {"no", "n"}:
            return False
        raise ValueError(f"未识别的元数据标记：{value!r}")

    for column in ["Doppler", "Marks", "Combined"]:
        info[column + "_flag"] = info[column].map(parse_flag)

    info["metadata_category"] = (
        info["Label"].astype(str).str.strip().str.lower()
    )

    data = manifest.merge(
        info[
            [
                "image_id", "metadata_category",
                "Doppler_flag", "Marks_flag", "Combined_flag",
            ]
        ],
        on="image_id",
        validate="one_to_one",
    )

    run = time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns()%1000000}"
    out = WORK / ("bus_uclm_targeted_check_" + run)
    out.mkdir(parents=True)
    (out / "previews").mkdir()

    info.to_csv(out / "normalized_metadata.csv", index=False)

    # ========================================================
    # 1. HESN：只记录尺寸和原始掩膜，不做拉伸或裁剪
    # ========================================================
    hesn = data[data.image_id.str.startswith("HESN_")].copy()
    geometry_rows = []
    inspection_errors = []
    hesn_cache = []

    for _, row in hesn.iterrows():
        try:
            image, mask, ip, mp, geometry = load_case(row)

            green = np.all(mask == [0, 255, 0], axis=2)
            red = np.all(mask == [255, 0, 0], axis=2)
            lesion = green | red
            yy, xx = np.where(lesion)

            if green.any() and red.any():
                category = "mixed"
            elif green.any():
                category = "benign"
            elif red.any():
                category = "malignant"
            else:
                category = "normal"

            colors = np.unique(mask.reshape(-1, 3), axis=0)

            geometry_rows.append({
                "image_id": row.image_id,
                "metadata_category": row.metadata_category,
                "mask_category": category,
                **geometry,
                "width_ratio_mask_to_image": (
                    geometry["mask_width"] / geometry["image_width"]
                ),
                "height_ratio_mask_to_image": (
                    geometry["mask_height"] / geometry["image_height"]
                ),
                "lesion_pixels": int(lesion.sum()),
                "mask_bbox_xmin": int(xx.min()) if len(xx) else np.nan,
                "mask_bbox_xmax": int(xx.max()) if len(xx) else np.nan,
                "mask_bbox_ymin": int(yy.min()) if len(yy) else np.nan,
                "mask_bbox_ymax": int(yy.max()) if len(yy) else np.nan,
                "mask_colors": json.dumps(colors.tolist()),
                "image_sha256": sha_file(ip),
                "mask_sha256": sha_file(mp),
            })

            hesn_cache.append((row.image_id, image, mask, category))

        except Exception as exc:
            inspection_errors.append({
                "image_id": row.image_id,
                "stage": "HESN",
                "error": str(exc),
            })

    pd.DataFrame(geometry_rows).to_csv(
        out / "hesn_geometry.csv", index=False
    )

    # 原图和掩膜分别显示，绝不强行叠加。
    for start in range(0, len(hesn_cache), 4):
        batch = hesn_cache[start:start + 4]
        fig, axes = plt.subplots(
            len(batch), 2, figsize=(10, 3.4 * len(batch)),
            squeeze=False,
        )

        for axes_row, (name, image, mask, category) in zip(axes, batch):
            axes_row[0].imshow(image)
            axes_row[0].set_title(
                f"{name} image {image.shape[1]}x{image.shape[0]}"
            )
            axes_row[1].imshow(mask)
            axes_row[1].set_title(
                f"{category} mask {mask.shape[1]}x{mask.shape[0]}"
            )
            for ax in axes_row:
                ax.axis("off")

        fig.tight_layout()
        fig.savefig(
            out / "previews" / f"hesn_native_{start:02d}.png",
            dpi=160,
        )
        plt.close(fig)

    # ========================================================
    # 2. 相同灰度图：比较掩膜，而不是自动合并患者
    # ========================================================
    checked = data[data.status == "checked"].copy()
    dup = checked[
        checked.duplicated("decoded_gray_sha256", keep=False)
    ]

    duplicate_pairs = []

    for group_number, (_, block) in enumerate(
        dup.groupby("decoded_gray_sha256")
    ):
        entries = list(block.iterrows())

        for a_index in range(len(entries)):
            for b_index in range(a_index + 1, len(entries)):
                a = entries[a_index][1]
                b = entries[b_index][1]

                try:
                    ia, ma, _, _, _ = load_case(a)
                    ib, mb, _, _, _ = load_case(b)

                    ga = np.asarray(Image.fromarray(ia).convert("L"))
                    gb = np.asarray(Image.fromarray(ib).convert("L"))

                    same_gray = (
                        ga.shape == gb.shape
                        and np.array_equal(ga, gb)
                    )

                    ba = (
                        np.all(ma == [0, 255, 0], axis=2)
                        | np.all(ma == [255, 0, 0], axis=2)
                    )
                    bb = (
                        np.all(mb == [0, 255, 0], axis=2)
                        | np.all(mb == [255, 0, 0], axis=2)
                    )

                    same_shape = ba.shape == bb.shape
                    intersection = (
                        int((ba & bb).sum()) if same_shape else 0
                    )
                    union = (
                        int((ba | bb).sum()) if same_shape else 0
                    )
                    total = int(ba.sum() + bb.sum())

                    # 两张都为空掩膜时，不把Dice=1当作病灶重合证据。
                    dice = (
                        2 * intersection / total
                        if same_shape and total else np.nan
                    )
                    iou = (
                        intersection / union
                        if same_shape and union else np.nan
                    )

                    duplicate_pairs.append({
                        "image_a": a.image_id,
                        "image_b": b.image_id,
                        "patient_a": a.patient_id,
                        "patient_b": b.patient_id,
                        "category_a": a.category,
                        "category_b": b.category,
                        "gray_pixels_identical": same_gray,
                        "rgb_pixels_identical": bool(
                            ia.shape == ib.shape
                            and np.array_equal(ia, ib)
                        ),
                        "mask_pixels_identical": bool(
                            ma.shape == mb.shape
                            and np.array_equal(ma, mb)
                        ),
                        "both_masks_empty": bool(total == 0),
                        "lesion_dice": dice,
                        "lesion_iou": iou,
                        "lesion_pixels_a": int(ba.sum()),
                        "lesion_pixels_b": int(bb.sum()),
                    })

                    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
                    for k, (name, image, mask) in enumerate([
                        (a.image_id, ia, ma),
                        (b.image_id, ib, mb),
                    ]):
                        axes[k, 0].imshow(image)
                        axes[k, 0].set_title(name + " original")
                        axes[k, 1].imshow(mask)
                        axes[k, 1].set_title(name + " mask")
                        axes[k, 0].axis("off")
                        axes[k, 1].axis("off")

                    fig.tight_layout()
                    fig.savefig(
                        out / "previews" /
                        f"duplicate_{group_number}_{a_index}_{b_index}.png",
                        dpi=160,
                    )
                    plt.close(fig)

                except Exception as exc:
                    inspection_errors.append({
                        "image_id": f"{a.image_id}|{b.image_id}",
                        "stage": "duplicates",
                        "error": str(exc),
                    })

    pd.DataFrame(duplicate_pairs).to_csv(
        out / "duplicate_pair_comparison.csv", index=False
    )

    # ========================================================
    # 3. 多连通区域：记录各区域大小、位置及元数据
    # ========================================================
    multi = checked[checked.lesion_components > 1].copy()
    multi.to_csv(out / "multiple_components_with_metadata.csv", index=False)

    component_rows = []

    for _, row in multi.iterrows():
        try:
            image, mask, _, _, _ = load_case(row)
            lesion = (
                np.all(mask == [0, 255, 0], axis=2)
                | np.all(mask == [255, 0, 0], axis=2)
            )

            labeled, n = ndi.label(
                lesion, structure=np.ones((3, 3), dtype=int)
            )
            lesion_total = int(lesion.sum())

            for component in range(1, n + 1):
                yy, xx = np.where(labeled == component)
                component_rows.append({
                    "image_id": row.image_id,
                    "patient_id": row.patient_id,
                    "category": row.category,
                    "component": component,
                    "pixels": len(xx),
                    "fraction_of_all_lesion_pixels": len(xx) / lesion_total,
                    "xmin": int(xx.min()),
                    "xmax": int(xx.max()),
                    "ymin": int(yy.min()),
                    "ymax": int(yy.max()),
                    "Doppler": bool(row.Doppler_flag),
                    "Marks": bool(row.Marks_flag),
                    "Combined": bool(row.Combined_flag),
                })

            fig, axes = plt.subplots(1, 2, figsize=(11, 5))
            axes[0].imshow(image)
            axes[0].set_title(row.image_id + " original")
            axes[1].imshow(image)
            for component in range(1, n + 1):
                region = labeled == component
                if region.any() and not region.all():
                    axes[1].contour(
                        region, levels=[0.5], linewidths=1
                    )
                yy, xx = np.where(region)
                axes[1].text(
                    xx.mean(), yy.mean(), str(component),
                    color="yellow", fontsize=11,
                )
            axes[1].set_title(
                f"Components={n}; Combined={row.Combined_flag}"
            )
            for ax in axes:
                ax.axis("off")
            fig.tight_layout()
            fig.savefig(
                out / "previews" / f"components_{row.image_id}.png",
                dpi=160,
            )
            plt.close(fig)

        except Exception as exc:
            inspection_errors.append({
                "image_id": row.image_id,
                "stage": "components",
                "error": str(exc),
            })

    pd.DataFrame(component_rows).to_csv(
        out / "component_geometry.csv", index=False
    )

    # 所有图像的元数据对应关系，包含尚未完成空间检查的HESN。
    data.to_csv(out / "manifest_with_metadata.csv", index=False)

    checked_conflicts = checked[
        checked.category != checked.metadata_category
    ]
    checked_conflicts.to_csv(
        out / "metadata_mask_label_conflicts.csv", index=False
    )

    # 按元数据类别报告多普勒、文字/测量标记、拼接图数量。
    flag_counts = data.groupby("metadata_category").agg(
        images=("image_id", "size"),
        doppler=("Doppler_flag", "sum"),
        marks=("Marks_flag", "sum"),
        combined=("Combined_flag", "sum"),
    )
    flag_counts.to_csv(out / "metadata_flags_by_category.csv")

    # 拼接图另外导出原图预览，不自动拆分。
    combined = data[data.Combined_flag].copy()

    for _, row in combined.iterrows():
        try:
            image, mask, _, _, _ = load_case(row)
            fig, axes = plt.subplots(1, 2, figsize=(11, 5))
            axes[0].imshow(image)
            axes[0].set_title(row.image_id + " combined image")
            axes[1].imshow(mask)
            axes[1].set_title("Original mask")
            for ax in axes:
                ax.axis("off")
            fig.tight_layout()
            fig.savefig(
                out / "previews" / f"combined_{row.image_id}.png",
                dpi=160,
            )
            plt.close(fig)
        except Exception as exc:
            inspection_errors.append({
                "image_id": row.image_id,
                "stage": "combined",
                "error": str(exc),
            })

    pd.DataFrame(
        inspection_errors, columns=["image_id", "stage", "error"]
    ).to_csv(out / "inspection_errors.csv", index=False)

    summary = {
        "total_manifest_images": len(data),
        "hesn_images_inspected": len(geometry_rows),
        "duplicate_pairs_inspected": len(duplicate_pairs),
        "multiple_component_images": len(multi),
        "combined_images": len(combined),
        "doppler_images": int(data.Doppler_flag.sum()),
        "checked_metadata_mask_label_conflicts": len(checked_conflicts),
        "inspection_errors": len(inspection_errors),
        "images_modified": False,
        "cases_excluded": False,
        "models_run": False,
    }

    protocol = {
        "purpose": "Targeted source-data inspection before frozen evaluation",
        "audit_manifest_sha256": sha_file(audit / "uclm_manifest.csv"),
        "metadata_sha256": sha_file(tables[0]),
        "mask_geometry": "Native coordinates; no resizing or alignment guessed",
        "flag_mapping": "Yes/Y -> True; No/N -> False",
        "components": "8-neighbour connected components; not assumed independent lesions",
        "duplicates": "Compare original decoded pixels and lesion masks",
        "patient_ids": "Preserved; no automatic merging or relabelling",
        "prediction": False,
    }

    for name, content in [
        ("summary.json", summary),
        ("protocol.json", protocol),
    ]:
        (out / name).write_text(
            json.dumps(content, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    try:
        cell = get_ipython().history_manager.input_hist_raw[-1]
        (out / "executed_cell.py").write_text(cell, encoding="utf-8")
    except Exception:
        pass

    archive = Path(shutil.make_archive(str(out), "zip", out))

    print("\n检查摘要：")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    print("\nHESN尺寸：")
    if geometry_rows:
        display(pd.DataFrame(geometry_rows)[
            [
                "image_id", "metadata_category", "mask_category",
                "image_width", "image_height",
                "mask_width", "mask_height",
            ]
        ])

    print("\n重复图像与掩膜差异：")
    display(pd.DataFrame(duplicate_pairs))

    print("\n元数据标记：")
    display(flag_counts)

    print("\n结果包：", archive)
    print("无需你人工判断预览图，下载整个ZIP发给我即可。")
    display(FileLink(str(archive)))


main_uclm_detail()

# ======================== 代码结束 ========================

In [ ]:
# ================= BUS-UCLM 冻结模型外部验证 =================
# CPU运行；不训练、不调参、不重新校准、不改变阈值。
# 使用既有人工掩膜，属于有标注ROI条件下的分类验证。

from pathlib import Path
from collections import defaultdict
from datetime import datetime
import io
import json
import hashlib
import zipfile
import shutil
import platform

import numpy as np
import pandas as pd
import scipy
import sklearn
import joblib
from PIL import Image
from scipy.special import logit
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits
from IPython.display import display, FileLink

# 一般保持空白即可自动寻找。
# 若自动寻找失败，填写Kaggle中的ZIP或已解压目录路径。
MODEL_PATH = (
    "/kaggle/working/"
    "bus_v6_external_training_20261013_121259_911015.zip"
)

AUDIT_PATH = (
    "/kaggle/working/"
    "bus_uclm_targeted_check_20261013_130605_803176.zip"
)

BOOTSTRAPS = 2000
BOOTSTRAP_SEED = 20261013

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUT = WORK / ("bus_uclm_external_" + RUN_ID)
OUT.mkdir(parents=True, exist_ok=False)

EXTRACTOR_HASH = (
    "fc020bfc28022f1b476d435d0dc8728aaee"
    "48e03d181007bce545aee22a1dda2"
)
MANIFEST_HASH = (
    "7cce7f175b90e0d50cbe3c2933356a8d82"
    "ae33219e78d4a575ffc84a3456afe5"
)
MODEL_HASHES = {
    "base_LR.joblib":
        "8a9d637defd779932b20c98c19dd1da58c831532dc8673fcad5b8d708467366d",
    "base_ensemble.joblib":
        "4bf456dc03fffbcd155a83a5300349e0b51c40d4fae98994d60b0827f61bec8b",
    "base_missing_LR.joblib":
        "5c60b2a14849d3911aad681a8f77108c2ed8dd41ed061e2065c9f6bc8183d8c1",
    "base_missing_ensemble.joblib":
        "739c50d5c6a50fd349743fc541d2e4c6b9c0439b706d0848e6b993e4c25ef48a",
    "base_posterior_LR.joblib":
        "666fbafd822433e2bde58c937080e7c419d67d1856f2187ee416dfecc9f4b209",
    "base_posterior_ensemble.joblib":
        "66f46dc5319fb09f73a1142e8588f9869ff7fea3f34b716c31f43ca4b22ca999",
}


def sha_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def save_json(name, obj):
    (OUT / name).write_text(
        json.dumps(obj, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def read_member(source, basename):
    """只读取文件，不解压、不执行包内其他代码。"""
    source = Path(source)
    if source.is_dir():
        matches = list(source.rglob(basename))
        matches = [p for p in matches if p.is_file()]
        if len(matches) != 1:
            raise ValueError(
                f"{source} 内 {basename} 数量不是1：{len(matches)}"
            )
        return matches[0].read_bytes()

    with zipfile.ZipFile(source) as z:
        names = [
            n for n in z.namelist()
            if not n.endswith("/")
            and Path(n).name == basename
            and "__MACOSX" not in n
        ]
        if len(names) != 1:
            raise ValueError(
                f"{source.name} 内 {basename} 数量不是1"
            )
        return z.read(names[0])


def discover(explicit, zip_prefix, anchor):
    if explicit.strip():
        p = Path(explicit.strip())
        if not p.exists():
            raise FileNotFoundError(str(p))
        return p

    candidates = []
    for root in (INPUT, WORK):
        if not root.exists():
            continue
        candidates += list(root.rglob(zip_prefix + "*.zip"))
        candidates += [
            p.parent for p in root.rglob(anchor)
            if p.is_file()
        ]

    candidates = sorted(set(candidates), key=str)
    if not candidates:
        raise FileNotFoundError(
            f"没有找到 {zip_prefix}。请添加对应结果包，"
            "或填写顶部路径。"
        )

    # ZIP与解压目录同时存在时，仅合并内容一致的副本。
    groups = defaultdict(list)
    for p in candidates:
        try:
            groups[sha_bytes(read_member(p, anchor))].append(p)
        except Exception:
            continue

    if len(groups) != 1:
        print("候选位置：")
        for p in candidates:
            print(p)
        raise RuntimeError(
            "未找到唯一内容版本，请在顶部填写明确路径。"
        )
    return next(iter(groups.values()))[0]


def parse_flag(series):
    mapping = {
        "true": True, "yes": True, "y": True, "1": True,
        "false": False, "no": False, "n": False, "0": False,
    }
    values = series.astype(str).str.strip().str.lower().map(mapping)
    if values.isna().any():
        raise ValueError("发现无法识别的元数据标记。")
    return values.astype(bool)


def main():
    model_source = discover(
        MODEL_PATH,
        "bus_v6_external_training_20261013_121259_911015",
        "model_hashes.json",
    )
    audit_source = discover(
        AUDIT_PATH,
        "bus_uclm_targeted_check_20261013_130605_803176",
        "manifest_with_metadata.csv",
    )
    print("模型来源：", model_source)
    print("审计来源：", audit_source)

    protocol = json.loads(
        read_member(model_source, "protocol.json")
    )
    expected_version = protocol["versions"]["scikit-learn"]
    if sklearn.__version__ != expected_version:
        raise RuntimeError(
            f"当前scikit-learn={sklearn.__version__}，"
            f"原模型版本={expected_version}。"
            "先停止，不跨版本加载模型；请把这行输出发回。"
        )

    extractor_bytes = read_member(model_source, "v6_extractor.py")
    if sha_bytes(extractor_bytes) != EXTRACTOR_HASH:
        raise ValueError("V6特征提取器哈希不一致。")

    manifest_bytes = read_member(
        audit_source, "manifest_with_metadata.csv"
    )
    if sha_bytes(manifest_bytes) != MANIFEST_HASH:
        raise ValueError("审计清单与已核对版本不一致。")

    # 仅执行已核验哈希的原V6提取器。
    namespace = {"__name__": "frozen_v6_extractor"}
    exec(
        compile(extractor_bytes, "v6_extractor.py", "exec"),
        namespace,
    )
    extract_features = namespace["extract_features"]
    side = int(namespace["SIDE"])
    (OUT / "v6_extractor.py").write_bytes(extractor_bytes)

    manifest = pd.read_csv(
        io.BytesIO(manifest_bytes),
        dtype={"image_id": str, "patient_id": str},
    )
    if len(manifest) != 683 or not manifest.image_id.is_unique:
        raise ValueError("审计清单数量或图像ID异常。")

    for col in ("Doppler_flag", "Marks_flag", "Combined_flag"):
        manifest[col] = parse_flag(manifest[col])

    # 排除原因按固定顺序记录；每张图像只计入首个原因。
    def exclusion_reason(row):
        if str(row.image_id).startswith("HESN_"):
            return "HESN_unresolved_geometry"
        if row.category == "normal":
            return "normal_outside_binary_task"
        if row.status != "checked":
            return "audit_not_checked"
        if row.category not in ("benign", "malignant"):
            return "unknown_category"
        if row.image_id in ("CHCO_000", "CHVI_005"):
            return "duplicate_with_discordant_masks"
        if row.Doppler_flag:
            return "doppler"
        if row.Combined_flag:
            return "combined"
        if float(row.lesion_components) != 1:
            return "multiple_or_empty_components"
        return "included"

    manifest["selection"] = manifest.apply(
        exclusion_reason, axis=1
    )
    manifest.to_csv(OUT / "selection_manifest.csv", index=False)
    flow = (
        manifest.groupby("selection", dropna=False)
        .size().rename("n").reset_index()
    )
    flow.to_csv(OUT / "selection_flow.csv", index=False)
    display(flow)

    cohort = (
        manifest.loc[manifest.selection == "included"]
        .sort_values("image_id").reset_index(drop=True).copy()
    )
    cohort["label"] = pd.to_numeric(
        cohort["label"], errors="raise"
    ).astype(int)

    if (
        len(cohort) != 207
        or int((cohort.label == 0).sum()) != 141
        or int((cohort.label == 1).sum()) != 66
    ):
        raise ValueError(
            "队列与预先固定的207张（141良性/66恶性）不一致。"
        )

    # 已发现CHCO/CHVI跨编号重复，保守地将这两个编号视为
    # 一个相关性聚类；不宣称已确认其为同一患者。
    cohort["cluster_id"] = cohort.patient_id.replace({
        "CHCO": "CHCO_CHVI_linked",
        "CHVI": "CHCO_CHVI_linked",
    })
    cohort.to_csv(OUT / "cohort_frozen.csv", index=False)

    save_json("evaluation_protocol.json", {
        "version": "BUS-UCLM-frozen-external-1",
        "n": 207,
        "benign": 141,
        "malignant": 66,
        "endpoint": "image-level benign/malignant classification",
        "roi": "provided reference masks; not automatic detection",
        "selection": "fixed before UCLM model predictions",
        "training": False,
        "recalibration": False,
        "threshold_tuning": False,
        "primary_model": "LR",
        "primary_comparison": "base_posterior minus base_missing",
        "bootstrap": "patient/linked-ID cluster percentile bootstrap",
        "bootstrap_replicates": BOOTSTRAPS,
        "bootstrap_seed": BOOTSTRAP_SEED,
        "threshold_caveat":
            "Source patient-mean thresholds applied to target images; "
            "95% target sensitivity is not guaranteed.",
        "ci_caveat":
            "Conditional on fixed models and selected cohort; "
            "secondary intervals are descriptive, unadjusted.",
        "extractor_sha256": EXTRACTOR_HASH,
        "manifest_sha256": MANIFEST_HASH,
        "versions": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scipy": scipy.__version__,
            "scikit-learn": sklearn.__version__,
        },
    })

    print(
        f"冻结队列：{len(cohort)}张；"
        f"{cohort.patient_id.nunique()}个患者编号；"
        f"{cohort.cluster_id.nunique()}个统计聚类。",
        flush=True,
    )

    # 按文件名建索引，再用原审计SHA256确定文件。
    # 不会把同名Images与Masks混淆，也不会随便选第一份。
    wanted_names = {
        Path(str(p)).name
        for col in ("image_path", "mask_path")
        for p in cohort[col]
    }
    file_index = defaultdict(set)
    print("定位原始图像和掩膜……", flush=True)
    for root in (INPUT, WORK):
        if root.exists():
            for p in root.rglob("*"):
                if p.name in wanted_names and p.is_file():
                    file_index[p.name].add(p)

    hash_cache = {}

    def locate(saved_path, expected_hash):
        direct = Path(str(saved_path))
        candidates = set(file_index.get(direct.name, set()))
        if direct.is_file():
            candidates.add(direct)
        for p in sorted(candidates, key=str):
            if p not in hash_cache:
                hash_cache[p] = sha_file(p)
            if hash_cache[p] == expected_hash:
                return p
        raise FileNotFoundError(
            f"{direct.name}：没有找到与审计SHA256一致的文件。"
            "请确认原始BUS-UCLM数据集已添加并下载完成。"
        )

    feature_rows, quality_rows, file_rows = [], [], []
    for i, row in enumerate(cohort.itertuples(index=False), 1):
        ip = locate(row.image_path, row.image_sha256)
        mp = locate(row.mask_path, row.mask_sha256)

        with Image.open(ip) as im:
            gray = np.asarray(im.convert("L"), dtype=np.uint8)
            valid = np.asarray(im.convert("RGBA"))[:, :, 3] > 0

        with Image.open(mp) as im:
            rgb = np.asarray(im.convert("RGB"))
            alpha = np.asarray(im.convert("RGBA"))[:, :, 3] > 0

        green = np.all(rgb == [0, 255, 0], axis=2)
        red = np.all(rgb == [255, 0, 0], axis=2)
        black = np.all(rgb == [0, 0, 0], axis=2)
        if not np.all((green | red | black) | ~alpha):
            raise ValueError(f"{row.image_id}：发现未知掩膜颜色。")

        # 合并红/绿为二值ROI；颜色类别绝不作为模型特征。
        mask = (green | red) & alpha
        if gray.shape != mask.shape or (mask & ~valid).any():
            raise ValueError(f"{row.image_id}：掩膜几何异常。")

        scale = side / max(gray.shape)
        size = (
            max(1, round(gray.shape[1] * scale)),
            max(1, round(gray.shape[0] * scale)),
        )
        x = np.asarray(
            Image.fromarray(gray.astype(np.float32) / 255.0)
            .resize(size, Image.Resampling.BILINEAR)
        )

        def resize_binary(a):
            return np.asarray(
                Image.fromarray(a.astype("uint8"))
                .resize(size, Image.Resampling.NEAREST)
            ) > 0

        m, v = resize_binary(mask), resize_binary(valid)
        features, quality = extract_features(x, m, v)
        feature_rows.append({"image_id": row.image_id, **features})
        quality_rows.append({"image_id": row.image_id, **quality})
        file_rows.append({
            "image_id": row.image_id,
            "image_path": str(ip),
            "mask_path": str(mp),
            "image_sha256": row.image_sha256,
            "mask_sha256": row.mask_sha256,
        })
        if i % 50 == 0 or i == len(cohort):
            print(f"已提取特征：{i}/{len(cohort)}", flush=True)

    features = pd.DataFrame(feature_rows).set_index("image_id")
    quality = pd.DataFrame(quality_rows).set_index("image_id")
    features.to_csv(OUT / "features.csv")
    quality.to_csv(OUT / "quality.csv")
    pd.DataFrame(file_rows).to_csv(
        OUT / "resolved_files.csv", index=False
    )
    features = features.loc[cohort.image_id]
    if np.isinf(features.to_numpy(dtype=float)).any():
        raise ValueError("特征出现无穷值；停止，不自动修改。")

    predictions, thresholds = {}, {}
    frozen_rows = []
    print("加载原模型并预测……", flush=True)

    with threadpool_limits(limits=2):
        for filename, expected_hash in MODEL_HASHES.items():
            data = read_member(model_source, filename)
            if sha_bytes(data) != expected_hash:
                raise ValueError(f"{filename}：模型哈希不一致。")

            # 仅加载用户此前生成且哈希一致的模型文件。
            bundle = joblib.load(io.BytesIO(data))
            if bundle["feature_version"] != "V6-unified-features-1":
                raise ValueError("模型特征版本不一致。")

            key = filename.removesuffix(".joblib")
            columns = bundle["feature_columns"]
            X = features[columns]
            raw_parts = []
            for estimator in bundle["models"].values():
                if list(estimator.classes_) != [0, 1]:
                    raise ValueError("模型类别顺序异常。")
                raw_parts.append(estimator.predict_proba(X)[:, 1])
            raw = np.mean(raw_parts, axis=0)

            # 原先保存的Platt校准器，只预测，不重新拟合。
            calibrated = bundle["calibrator"].predict_proba(
                logit(np.clip(raw, 1e-6, 1 - 1e-6)).reshape(-1, 1)
            )[:, 1]
            if not np.isfinite(calibrated).all():
                raise ValueError("预测出现非有限值。")

            threshold = float(bundle["threshold"])
            predictions[key] = calibrated
            thresholds[key] = threshold
            frozen_rows.append(pd.DataFrame({
                "image_id": cohort.image_id.to_numpy(),
                "configuration": key,
                "raw_probability": raw,
                "probability": calibrated,
                "threshold": threshold,
                "predicted_label": (calibrated >= threshold).astype(int),
            }))
            print("完成：", key, flush=True)

    # 先保存固定预测，再计算成绩。
    pred = pd.concat(frozen_rows, ignore_index=True)
    pred.to_csv(OUT / "predictions_frozen.csv", index=False)
    pred.merge(
        cohort[["image_id", "patient_id", "cluster_id", "label"]],
        on="image_id", validate="many_to_one",
    ).to_csv(OUT / "predictions_with_labels.csv", index=False)
    save_json("model_hashes.json", MODEL_HASHES)

    y = cohort.label.to_numpy()
    metric_names = ["auc", "brier", "sensitivity", "specificity"]

    def metric_values(yv, pv, threshold):
        decision = pv >= threshold
        positive, negative = yv == 1, yv == 0
        return np.array([
            roc_auc_score(yv, pv),
            np.mean((pv - yv) ** 2),
            decision[positive].mean(),
            (~decision[negative]).mean(),
        ], dtype=float)

    # 同一轮抽样用于所有模型，保证差值为配对比较。
    cluster_values = cohort.cluster_id.to_numpy()
    clusters = np.unique(cluster_values)
    blocks = [
        np.flatnonzero(cluster_values == name) for name in clusters
    ]
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    draws = []
    for _ in range(BOOTSTRAPS):
        chosen = rng.integers(0, len(blocks), size=len(blocks))
        indices = np.concatenate([blocks[j] for j in chosen])
        if np.unique(y[indices]).size == 2:
            draws.append(indices)

    if len(draws) < 0.9 * BOOTSTRAPS:
        raise RuntimeError("有效聚类bootstrap次数不足。")

    bootstrap_results = {}
    metric_rows = []
    print("计算患者聚类置信区间……", flush=True)
    for key, p in predictions.items():
        threshold = thresholds[key]
        point = metric_values(y, p, threshold)
        boot = np.array([
            metric_values(y[ix], p[ix], threshold) for ix in draws
        ])
        bootstrap_results[key] = boot
        low, high = np.quantile(boot, [0.025, 0.975], axis=0)
        record = {
            "configuration": key,
            "n_images": len(y),
            "n_patient_ids": cohort.patient_id.nunique(),
            "n_clusters": len(clusters),
            "threshold": threshold,
            "missed_malignant": int(((y == 1) & (p < threshold)).sum()),
            "false_positive": int(((y == 0) & (p >= threshold)).sum()),
            "valid_bootstraps": len(draws),
        }
        for j, name in enumerate(metric_names):
            record[name] = point[j]
            record[name + "_low"] = low[j]
            record[name + "_high"] = high[j]
        metric_rows.append(record)

    metrics = pd.DataFrame(metric_rows)
    metrics.to_csv(OUT / "external_metrics.csv", index=False)

    difference_rows = []
    for model in ("LR", "ensemble"):
        for before, after, comparison in [
            ("base", "base_missing", "missing_indicator_addition"),
            ("base_missing", "base_posterior", "measurement_addition"),
            ("base", "base_posterior", "total_addition"),
        ]:
            a, b = before + "_" + model, after + "_" + model
            point_a = metric_values(y, predictions[a], thresholds[a])
            point_b = metric_values(y, predictions[b], thresholds[b])
            delta_boot = bootstrap_results[b] - bootstrap_results[a]
            low, high = np.quantile(
                delta_boot, [0.025, 0.975], axis=0
            )
            for j, name in enumerate(metric_names):
                difference_rows.append({
                    "model": model,
                    "comparison": comparison,
                    "metric": name,
                    "after_minus_before": point_b[j] - point_a[j],
                    "low": low[j],
                    "high": high[j],
                    "note": (
                        "Negative Brier difference favors after; "
                        "other positive differences favor after. "
                        "Sensitivity/specificity use each frozen threshold."
                    ),
                })
    differences = pd.DataFrame(difference_rows)
    differences.to_csv(
        OUT / "paired_metric_differences.csv", index=False
    )

    # 预先指定的描述性分层，不用于调整模型。
    usable = (
        quality.loc[cohort.image_id, "posterior_usable"]
        .to_numpy(dtype=bool)
    )
    marked = cohort.Marks_flag.to_numpy(dtype=bool)
    subgroup_rows = []
    for group_name, keep in {
        "posterior_measurable": usable,
        "posterior_unmeasurable": ~usable,
        "marks_present": marked,
        "marks_absent": ~marked,
    }.items():
        for key, p in predictions.items():
            result = {
                "subgroup": group_name,
                "configuration": key,
                "n": int(keep.sum()),
                "malignant_n": int(y[keep].sum()),
                "n_clusters": len(np.unique(cluster_values[keep])),
            }
            if np.unique(y[keep]).size == 2:
                result.update(dict(zip(
                    metric_names,
                    metric_values(y[keep], p[keep], thresholds[key]),
                )))
            subgroup_rows.append(result)
    pd.DataFrame(subgroup_rows).to_csv(
        OUT / "descriptive_subgroups.csv", index=False
    )

    save_json("summary.json", {
        "completed": True,
        "n_images": len(cohort),
        "benign": int((y == 0).sum()),
        "malignant": int((y == 1).sum()),
        "n_patient_ids": int(cohort.patient_id.nunique()),
        "n_clusters": len(clusters),
        "posterior_measurable": int(usable.sum()),
        "models_evaluated": len(predictions),
        "models_retrained": False,
        "thresholds_changed": False,
        "valid_bootstraps": len(draws),
    })

    # 保存当前单元代码，便于复现。
    try:
        source = get_ipython().history_manager.input_hist_raw[-1]
        (OUT / "executed_cell.py").write_text(
            source, encoding="utf-8"
        )
    except Exception:
        pass

    print("\n第二外部验证完成：")
    display(metrics[[
        "configuration", "n_images", "auc", "auc_low", "auc_high",
        "brier", "sensitivity", "specificity",
        "missed_malignant", "false_positive",
    ]])
    print("\n配对AUC差值：")
    display(differences[differences.metric == "auc"])

    archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
    print("\n结果包：", archive)
    print("请下载结果ZIP发回。此次没有训练模型或调整阈值。")
    display(FileLink(archive))


main()
# ======================= 代码结束 =======================

In [ ]:
# ============== 两个外部验证结果整理：不训练、不改阈值 ==============
from pathlib import Path
from datetime import datetime
import io
import json
import hashlib
import zipfile
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from IPython.display import display, FileLink

# 通常不需要填写。自动查找失败时填写Kaggle中的准确路径。
BREAST_PATH = ""
UCLM_PATH = ""

BREAST_NAME = "bus_v6_external_training_20261013_121259_911015"
UCLM_NAME = "bus_uclm_external_20261013_132503_790776"

OUT = Path("/kaggle/working") / (
    "bus_external_report_" +
    datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
OUT.mkdir(parents=True, exist_ok=False)

SETS = ["base", "base_missing", "base_posterior"]
MODELS = ["LR", "ensemble"]
COLORS = {
    "base": "#0072B2",
    "base_missing": "#E69F00",
    "base_posterior": "#009E73",
}
LABELS = {
    "base": "Base",
    "base_missing": "Base + missing indicator",
    "base_posterior": "Base + indicator + posterior",
}
DATASETS = ["BrEaST", "BUS-UCLM"]
UNITS = {
    "BrEaST": "patient/case",
    "BUS-UCLM": "image; patient-clustered intervals",
}


def read_bytes(source, name):
    source = Path(source)
    if source.is_dir():
        p = source / name
        if not p.is_file():
            raise FileNotFoundError(str(p))
        return p.read_bytes()
    with zipfile.ZipFile(source) as z:
        matches = [
            n for n in z.namelist()
            if not n.endswith("/")
            and Path(n).name == name
            and "__MACOSX" not in n
        ]
        if len(matches) != 1:
            raise ValueError(f"{source.name} 内 {name} 不是唯一文件")
        return z.read(matches[0])


def read_csv(source, name):
    return pd.read_csv(io.BytesIO(read_bytes(source, name)))


def find_package(explicit, stem, anchor):
    if explicit.strip():
        p = Path(explicit.strip())
        if not p.exists():
            raise FileNotFoundError(str(p))
        return p

    candidates = []
    for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
        if root.exists():
            # 限定准确名称，不再把其他实验目录加入候选。
            candidates.extend(root.rglob(stem + ".zip"))
            candidates.extend(
                p for p in root.rglob(stem) if p.is_dir()
            )

    candidates = sorted(set(candidates), key=str)
    if not candidates:
        raise FileNotFoundError(
            f"没有找到 {stem}.zip；请添加到Input，"
            "或在代码顶部填写路径。"
        )

    versions = {}
    for p in candidates:
        content = read_bytes(p, anchor)
        digest = hashlib.sha256(content).hexdigest()
        versions.setdefault(digest, []).append(p)

    if len(versions) != 1:
        print(*candidates, sep="\n")
        raise RuntimeError("存在不同内容的结果包，请明确填写路径。")
    return next(iter(versions.values()))[0]


def savefig(fig, name):
    fig.savefig(OUT / (name + ".png"), dpi=300, bbox_inches="tight")
    fig.savefig(OUT / (name + ".pdf"), bbox_inches="tight")
    plt.show()
    plt.close(fig)


def main():
    breast = find_package(
        BREAST_PATH, BREAST_NAME, "external_predictions.csv"
    )
    uclm = find_package(
        UCLM_PATH, UCLM_NAME, "predictions_with_labels.csv"
    )
    print("BrEaST：", breast)
    print("BUS-UCLM：", uclm)

    # 统一预测字段，保留各数据集原有评价单位。
    a = read_csv(breast, "external_predictions.csv")
    a["configuration"] = a.feature_set + "_" + a.model
    a = a.rename(columns={
        "patient_key": "unit_id",
        "p": "probability",
        "prediction": "predicted_label",
    })
    a["cluster_id"] = a.unit_id
    a["dataset"] = "BrEaST"

    b = read_csv(uclm, "predictions_with_labels.csv")
    b = b.rename(columns={"image_id": "unit_id"})
    b["dataset"] = "BUS-UCLM"

    columns = [
        "dataset", "unit_id", "cluster_id", "configuration",
        "label", "probability", "threshold", "predicted_label",
    ]
    pred = pd.concat([a[columns], b[columns]], ignore_index=True)
    for c in ["unit_id", "cluster_id"]:
        pred[c] = pred[c].astype(str)

    configs = [s + "_" + m for s in SETS for m in MODELS]
    expected = {"BrEaST": (252, 98), "BUS-UCLM": (207, 66)}

    for dataset, (n, malignant) in expected.items():
        d = pred[pred.dataset == dataset]
        if set(d.configuration) != set(configs):
            raise ValueError(f"{dataset} 模型配置不完整")
        if d.duplicated(["unit_id", "configuration"]).any():
            raise ValueError(f"{dataset} 出现重复预测")
        if d.groupby("unit_id").label.nunique().max() != 1:
            raise ValueError(f"{dataset} 同一单位的标签不一致")

        reference_ids = None
        for key, g in d.groupby("configuration"):
            ids = set(g.unit_id)
            if reference_ids is None:
                reference_ids = ids
            if ids != reference_ids:
                raise ValueError(f"{dataset} 配置间样本不一致")
            if len(g) != n or int(g.label.sum()) != malignant:
                raise ValueError(f"{dataset} 样本数或标签数量异常")
            if set(g.label) != {0, 1}:
                raise ValueError("标签不是0/1")
            p = g.probability.to_numpy(float)
            if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
                raise ValueError("概率异常")
            if g.threshold.nunique() != 1:
                raise ValueError("同一配置存在多个阈值")
            calculated = (g.probability >= g.threshold).astype(int)
            if not np.array_equal(calculated, g.predicted_label):
                raise ValueError("保存分类与冻结阈值不一致")

    # 两个外部集必须使用相同的冻结阈值。
    thresholds = pred.groupby(
        ["configuration", "dataset"]
    ).threshold.first().unstack()
    if not np.allclose(
        thresholds["BrEaST"], thresholds["BUS-UCLM"],
        rtol=0, atol=1e-14,
    ):
        raise ValueError("两个外部集的阈值不一致")

    # 读取已有置信区间，不改变原有bootstrap设计。
    ma = read_csv(breast, "external_metrics.csv")
    ma["configuration"] = ma.feature_set + "_" + ma.model
    ma["dataset"] = "BrEaST"
    ma = ma.rename(columns={
        "auc_ci_low": "auc_low",
        "auc_ci_high": "auc_high",
        "sensitivity_ci_low": "sensitivity_low",
        "sensitivity_ci_high": "sensitivity_high",
        "specificity_ci_low": "specificity_low",
        "specificity_ci_high": "specificity_high",
    })

    mb = read_csv(uclm, "external_metrics.csv")
    mb["dataset"] = "BUS-UCLM"
    mb = mb.rename(columns={"n_images": "n"})

    metric_columns = [
        "dataset", "configuration", "n", "threshold",
        "auc", "auc_low", "auc_high", "brier",
        "sensitivity", "sensitivity_low", "sensitivity_high",
        "specificity", "specificity_low", "specificity_high",
        "missed_malignant", "false_positive",
    ]
    metrics = pd.concat(
        [ma[metric_columns], mb[metric_columns]], ignore_index=True
    )
    metrics["evaluation_unit"] = metrics.dataset.map(UNITS)

    # 逐项重算点估计，对照保存结果。
    for row in metrics.itertuples(index=False):
        g = pred[
            (pred.dataset == row.dataset) &
            (pred.configuration == row.configuration)
        ]
        y = g.label.to_numpy(int)
        p = g.probability.to_numpy(float)
        decision = p >= row.threshold
        checks = {
            "auc": roc_auc_score(y, p),
            "brier": np.mean((p - y) ** 2),
            "sensitivity": decision[y == 1].mean(),
            "specificity": (~decision[y == 0]).mean(),
            "missed_malignant": ((y == 1) & ~decision).sum(),
            "false_positive": ((y == 0) & decision).sum(),
        }
        for name, value in checks.items():
            if not np.isclose(value, getattr(row, name), atol=1e-10):
                raise ValueError(
                    f"{row.dataset}/{row.configuration}/{name}不一致"
                )

    metrics.to_csv(OUT / "main_results_numeric.csv", index=False)

    table = metrics[[
        "dataset", "evaluation_unit", "configuration", "n"
    ]].copy()
    for metric in ["auc", "sensitivity", "specificity"]:
        table[metric + "_95CI"] = metrics.apply(
            lambda r: (
                f"{r[metric]:.3f} "
                f"({r[metric + '_low']:.3f}–"
                f"{r[metric + '_high']:.3f})"
            ), axis=1,
        )
    table["brier"] = metrics.brier.round(4)
    table["missed_malignant"] = metrics.missed_malignant
    table["false_positive"] = metrics.false_positive
    table.to_csv(
        OUT / "main_results_formatted.csv",
        index=False, encoding="utf-8-sig",
    )

    # 统一配对AUC差值：始终为“增加后减增加前”。
    da = read_csv(breast, "paired_auc_differences.csv")
    da = da.rename(columns={
        "auc_difference_a_minus_b": "difference",
        "ci_low": "low", "ci_high": "high",
    })
    da["dataset"] = "BrEaST"

    db = read_csv(uclm, "paired_metric_differences.csv")
    db = db[db.metric == "auc"].copy()
    db = db.rename(columns={"after_minus_before": "difference"})
    db["comparison"] = db.comparison.replace({
        "measurement_addition": "measurement_increment",
        "missing_indicator_addition": "missingness_increment",
        "total_addition": "total_increment",
    })
    db["dataset"] = "BUS-UCLM"
    diff_cols = [
        "dataset", "model", "comparison", "difference", "low", "high"
    ]
    differences = pd.concat(
        [da[diff_cols], db[diff_cols]], ignore_index=True
    )
    differences.to_csv(OUT / "paired_auc_results.csv", index=False)

    plt.rcParams.update({
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
    })

    # 图1：两个数据集分开，LR与集成分开。
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for i, dataset in enumerate(DATASETS):
        for j, model in enumerate(MODELS):
            ax = axes[i, j]
            for s in SETS:
                key = s + "_" + model
                g = pred[
                    (pred.dataset == dataset) &
                    (pred.configuration == key)
                ]
                fpr, tpr, _ = roc_curve(g.label, g.probability)
                auc = roc_auc_score(g.label, g.probability)
                ax.plot(
                    fpr, tpr, color=COLORS[s], lw=1.7,
                    label=f"{LABELS[s]}: {auc:.3f}",
                )
            ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
            ax.set(
                title=f"{dataset} | {model}",
                xlabel="False positive rate",
                ylabel="True positive rate",
                xlim=(0, 1), ylim=(0, 1.02),
            )
            ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    savefig(fig, "figure1_external_ROC")

    # 图2：固定5个等宽概率区间。
    # 仅描述性分箱校准图，不将图上的点当独立患者置信区间。
    bin_rows = []
    for (dataset, key), g in pred.groupby(
        ["dataset", "configuration"]
    ):
        g = g.copy()
        g["bin"] = np.minimum(
            (g.probability.to_numpy() * 5).astype(int), 4
        )
        for k, h in g.groupby("bin"):
            bin_rows.append({
                "dataset": dataset,
                "configuration": key,
                "bin": int(k),
                "lower": k / 5,
                "upper": (k + 1) / 5,
                "n": len(h),
                "n_clusters": h.cluster_id.nunique(),
                "mean_probability": h.probability.mean(),
                "observed_fraction": h.label.mean(),
            })
    bins = pd.DataFrame(bin_rows)
    bins.to_csv(OUT / "calibration_bins.csv", index=False)

    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for i, dataset in enumerate(DATASETS):
        for j, model in enumerate(MODELS):
            ax = axes[i, j]
            for s in SETS:
                g = bins[
                    (bins.dataset == dataset) &
                    (bins.configuration == s + "_" + model)
                ].sort_values("bin")
                ax.plot(
                    g.mean_probability, g.observed_fraction,
                    "o-", color=COLORS[s], label=LABELS[s],
                )
            ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
            ax.set(
                title=f"{dataset} | {model}",
                xlabel="Mean predicted probability",
                ylabel="Observed malignant fraction",
                xlim=(0, 1), ylim=(0, 1),
            )
            ax.legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "figure2_descriptive_calibration")

    # 图3：后方测量的净增益，控制缺失指示。
    d = differences[
        differences.comparison == "measurement_increment"
    ].copy().reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 4))
    for i, row in d.iterrows():
        ax.plot(
            [row.low, row.high], [i, i],
            color="#0072B2", lw=2,
        )
        ax.scatter(row.difference, i, color="#0072B2", s=45)
    ax.axvline(0, color="gray", linestyle="--")
    ax.set_yticks(
        range(len(d)),
        d.dataset + " | " + d.model,
    )
    ax.invert_yaxis()
    ax.set_xlabel(
        "AUC difference: posterior model minus missing-indicator model"
    )
    ax.set_title("Paired AUC increments with existing 95% intervals")
    fig.tight_layout()
    savefig(fig, "figure3_posterior_increment")

    # 图4：冻结阈值的敏感度及原始置信区间。
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, dataset in zip(axes, DATASETS):
        g = (
            metrics[metrics.dataset == dataset]
            .set_index("configuration").loc[configs].reset_index()
        )
        for i, row in g.iterrows():
            ax.plot(
                [row.sensitivity_low, row.sensitivity_high],
                [i, i], color="#0072B2", lw=2,
            )
            ax.scatter(row.sensitivity, i, color="#0072B2")
        ax.axvline(
            0.95, color="#D55E00", linestyle="--",
            label="Source target: 95%",
        )
        ax.set_yticks(range(len(g)), g.configuration)
        ax.invert_yaxis()
        ax.set_xlim(0, 1.02)
        ax.set_xlabel("Sensitivity at frozen source threshold")
        ax.set_title(dataset)
        ax.legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "figure4_frozen_threshold_sensitivity")

    # 输出错误明细与聚类分布，不更改任何预测。
    pred["error_type"] = np.select(
        [
            (pred.label == 1) & (pred.predicted_label == 0),
            (pred.label == 0) & (pred.predicted_label == 1),
        ],
        ["false_negative", "false_positive"],
        default="correct",
    )
    pred.to_csv(OUT / "all_predictions_checked.csv", index=False)
    pred[pred.error_type != "correct"].to_csv(
        OUT / "error_cases.csv", index=False
    )
    (
        pred.groupby(
            ["dataset", "configuration", "cluster_id", "error_type"]
        ).size().rename("n").reset_index()
        .to_csv(OUT / "errors_by_cluster.csv", index=False)
    )

    # 错误改变：配对比较测量加入前后，不计算未经校正的p值。
    transitions = []
    for dataset in DATASETS:
        for model in MODELS:
            before = pred[
                (pred.dataset == dataset) &
                (pred.configuration == "base_missing_" + model)
            ].set_index("unit_id")
            after = pred[
                (pred.dataset == dataset) &
                (pred.configuration == "base_posterior_" + model)
            ].set_index("unit_id").loc[before.index]
            for label in [0, 1]:
                use = before.label == label
                e0 = before.predicted_label != before.label
                e1 = after.predicted_label != after.label
                transitions.append({
                    "dataset": dataset,
                    "model": model,
                    "label": label,
                    "n": int(use.sum()),
                    "corrected_errors": int((use & e0 & ~e1).sum()),
                    "new_errors": int((use & ~e0 & e1).sum()),
                    "persistent_errors": int((use & e0 & e1).sum()),
                })
    pd.DataFrame(transitions).to_csv(
        OUT / "paired_error_changes.csv", index=False
    )

    # 保存筛选流程与来源记录。
    (OUT / "uclm_selection_flow.csv").write_bytes(
        read_bytes(uclm, "selection_flow.csv")
    )
    provenance = {
        "breast_source": str(breast),
        "uclm_source": str(uclm),
        "breast_predictions_sha256": hashlib.sha256(
            read_bytes(breast, "external_predictions.csv")
        ).hexdigest(),
        "uclm_predictions_sha256": hashlib.sha256(
            read_bytes(uclm, "predictions_with_labels.csv")
        ).hexdigest(),
        "models_trained": False,
        "thresholds_modified": False,
        "calibration_refitted": False,
        "confidence_intervals": "Retained from original result packages",
        "datasets_pooled": False,
    }
    (OUT / "provenance.json").write_text(
        json.dumps(provenance, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    (OUT / "figure_notes.txt").write_text(
        "All predictions and thresholds are frozen.\n"
        "BrEaST: 252 patient/case-level observations.\n"
        "BUS-UCLM: 207 selected images, 33 patient-ID clusters.\n"
        "Do not describe 207 images as 207 independent patients.\n"
        "No pooled cross-dataset AUC was calculated.\n"
        "ROC curves use provided reference lesion masks.\n"
        "Calibration curves use five fixed equal-width bins; "
        "they are descriptive, without binwise confidence intervals.\n"
        "AUC and sensitivity intervals retain the original designs; "
        "BUS-UCLM intervals use patient-cluster bootstrap.\n"
        "Secondary comparisons are descriptive and unadjusted.\n"
        "BrEaST was previously used for posterior-sign analysis; "
        "disclose this development history.\n",
        encoding="utf-8",
    )

    try:
        source = get_ipython().history_manager.input_hist_raw[-1]
        (OUT / "executed_cell.py").write_text(source, encoding="utf-8")
    except Exception:
        pass

    print("\n已核对全部预测和点估计。结果总表：")
    display(table)
    print("\n后方测量净增益：")
    display(differences[
        differences.comparison == "measurement_increment"
    ])

    archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
    print("\n完成：未训练模型、未改变预测或阈值。")
    print("结果包：", archive)
    display(FileLink(archive))


main()
# ========================= 代码结束 =========================

In [ ]:
# =========== BUS-UCLM患者等权敏感性分析 ===========
# 不训练模型、不调整阈值、不合并患者的良恶性标签。
# 每个患者聚类总权重相同；患者内部图像等权。
# 置信区间：患者聚类bootstrap，所有模型共用相同抽样。

from pathlib import Path
from datetime import datetime
import io
import json
import hashlib
import zipfile
import shutil

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from IPython.display import display, FileLink

# 自动查找失败时，填写Kaggle中的ZIP或解压目录路径。
RESULT_PATH = ""

PACKAGE_NAME = "bus_uclm_external_20261013_132503_790776"
EXPECTED_PREDICTION_HASH = (
    "8633521a5610e4e91a90543d1458cf893"
    "497684499d77d2163af51a5e91532ee"
)

BOOTSTRAPS = 2000
SEED = 20261014

OUT = Path("/kaggle/working") / (
    "bus_uclm_patient_equal_" +
    datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
OUT.mkdir(parents=True, exist_ok=False)

CONFIGS = [
    f"{feature_set}_{model}"
    for feature_set in ["base", "base_missing", "base_posterior"]
    for model in ["LR", "ensemble"]
]
METRICS = ["auc", "brier", "sensitivity", "specificity"]


def read_bytes(source, filename):
    source = Path(source)
    if source.is_dir():
        path = source / filename
        if not path.is_file():
            raise FileNotFoundError(str(path))
        return path.read_bytes()

    with zipfile.ZipFile(source) as z:
        matches = [
            name for name in z.namelist()
            if not name.endswith("/")
            and Path(name).name == filename
            and "__MACOSX" not in name
        ]
        if len(matches) != 1:
            raise ValueError(f"{filename}在结果包内不是唯一文件")
        return z.read(matches[0])


def find_source():
    if RESULT_PATH.strip():
        path = Path(RESULT_PATH.strip())
        if not path.exists():
            raise FileNotFoundError(str(path))
        return path

    candidates = []
    for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
        if root.exists():
            candidates.extend(root.rglob(PACKAGE_NAME + ".zip"))
            candidates.extend(
                p for p in root.rglob(PACKAGE_NAME) if p.is_dir()
            )

    candidates = sorted(set(candidates), key=str)
    if not candidates:
        raise FileNotFoundError(
            f"没有找到{PACKAGE_NAME}.zip。"
            "请添加到Input，或填写RESULT_PATH。"
        )

    # ZIP和目录同时存在时，仅接受预测内容一致的副本。
    versions = {}
    for path in candidates:
        content = read_bytes(path, "predictions_with_labels.csv")
        digest = hashlib.sha256(content).hexdigest()
        versions.setdefault(digest, []).append(path)

    if len(versions) != 1:
        print(*candidates, sep="\n")
        raise RuntimeError("发现不同结果版本，请填写明确路径。")
    return next(iter(versions.values()))[0]


def calculate(y, p, threshold, weights):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    weights = np.asarray(weights, dtype=float)

    active = weights > 0
    y, p, weights = y[active], p[active], weights[active]

    if len(np.unique(y)) != 2:
        return np.full(len(METRICS), np.nan)

    positive = y == 1
    negative = y == 0
    predicted = p >= threshold

    return np.array([
        roc_auc_score(y, p, sample_weight=weights),
        np.average((p - y) ** 2, weights=weights),
        np.average(predicted[positive], weights=weights[positive]),
        np.average(~predicted[negative], weights=weights[negative]),
    ], dtype=float)


def main():
    source = find_source()
    content = read_bytes(source, "predictions_with_labels.csv")
    digest = hashlib.sha256(content).hexdigest()

    if digest != EXPECTED_PREDICTION_HASH:
        raise ValueError("预测文件与已核对的冻结版本不一致。")

    print("结果来源：", source)
    pred = pd.read_csv(
        io.BytesIO(content),
        dtype={
            "image_id": str,
            "patient_id": str,
            "cluster_id": str,
            "configuration": str,
        },
    )

    if set(pred.configuration) != set(CONFIGS):
        raise ValueError("六套模型配置不完整")
    if pred.duplicated(["configuration", "image_id"]).any():
        raise ValueError("存在重复预测")
    if pred[["image_id", "patient_id", "cluster_id"]].isna().any().any():
        raise ValueError("图像或患者标识存在缺失")

    frames = {}
    reference = None

    for key in CONFIGS:
        frame = (
            pred[pred.configuration == key]
            .sort_values("image_id").reset_index(drop=True)
        )
        if (
            len(frame) != 207
            or set(frame.label) != {0, 1}
            or int(frame.label.sum()) != 66
        ):
            raise ValueError(f"{key}：队列数量或标签异常")

        identity = frame[
            ["image_id", "patient_id", "cluster_id", "label"]
        ]
        if reference is None:
            reference = identity.copy()
        elif not identity.equals(reference):
            raise ValueError("不同配置的图像、标签或患者对应关系不一致")

        p = frame.probability.to_numpy(float)
        if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
            raise ValueError("预测概率异常")
        if frame.threshold.nunique() != 1:
            raise ValueError("同一配置出现多个阈值")
        threshold = float(frame.threshold.iloc[0])
        if not np.array_equal(
            (p >= threshold).astype(int),
            frame.predicted_label.to_numpy(int),
        ):
            raise ValueError("原分类与冻结阈值不一致")
        frames[key] = frame

    cluster_names, cluster_index = np.unique(
        reference.cluster_id.to_numpy(), return_inverse=True
    )
    n_clusters = len(cluster_names)
    cluster_sizes = np.bincount(cluster_index)
    if n_clusters != 33:
        raise ValueError(f"患者聚类数异常：{n_clusters}，预期33")

    y = reference.label.to_numpy(int)

    # 图像级主分析：每张图权重1。
    image_weights = np.ones(len(y), dtype=float)

    # 患者等权补充分析：每个聚类内部图像权重1/n_i。
    equal_weights = 1.0 / cluster_sizes[cluster_index]
    if not np.allclose(
        np.bincount(cluster_index, weights=equal_weights), 1.0
    ):
        raise ValueError("患者等权构造失败")

    weights_table = reference.copy()
    weights_table["images_in_cluster"] = cluster_sizes[cluster_index]
    weights_table["patient_equal_weight"] = equal_weights
    weights_table.to_csv(OUT / "image_weights.csv", index=False)

    cluster_table = (
        reference.groupby("cluster_id")
        .agg(
            n_images=("image_id", "size"),
            malignant_images=("label", "sum"),
        )
        .reset_index()
    )
    cluster_table["benign_images"] = (
        cluster_table.n_images - cluster_table.malignant_images
    )
    cluster_table.to_csv(OUT / "cluster_composition.csv", index=False)

    print(f"队列：207张图像，{n_clusters}个患者聚类")
    print(
        "每个聚类图像数：",
        int(cluster_sizes.min()), "至", int(cluster_sizes.max()),
    )

    # 保存分析定义，避免把加权图像指标写成患者诊断指标。
    protocol = {
        "analysis": "patient-equal sensitivity analysis",
        "input_prediction_sha256": digest,
        "evaluation_unit": "image",
        "weight": "1 / number of included images in patient cluster",
        "interpretation":
            "Each cluster has equal total weight; labels remain image-level.",
        "not_patient_diagnosis":
            "No averaging of malignant and benign labels or predictions.",
        "auc_definition":
            "Weighted positive-negative image-pair AUC; "
            "not an average of within-patient AUCs.",
        "primary_analysis":
            "Original unweighted image-level results remain primary.",
        "bootstrap":
            "Resample 33 clusters with replacement; each sampled copy "
            "retains all its images and total weight 1.",
        "bootstrap_replicates": BOOTSTRAPS,
        "seed": SEED,
        "training": False,
        "recalibration": False,
        "threshold_change": False,
        "intervals":
            "Percentile cluster-bootstrap, conditional on frozen models; "
            "exploratory and unadjusted for multiple comparisons.",
    }
    (OUT / "protocol.json").write_text(
        json.dumps(protocol, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    # 每轮抽样用聚类出现次数表示，保留重复抽中的患者。
    rng = np.random.default_rng(SEED)
    multiplicities = []
    for _ in range(BOOTSTRAPS):
        sampled = rng.integers(0, n_clusters, size=n_clusters)
        counts = np.bincount(sampled, minlength=n_clusters)
        weights = counts[cluster_index] * equal_weights
        if np.unique(y[weights > 0]).size == 2:
            multiplicities.append(counts)

    multiplicities = np.asarray(multiplicities)
    valid_bootstraps = len(multiplicities)
    if valid_bootstraps < 0.9 * BOOTSTRAPS:
        raise RuntimeError("有效聚类bootstrap次数不足")
    np.save(OUT / "bootstrap_cluster_multiplicities.npy", multiplicities)
    pd.Series(cluster_names, name="cluster_id").to_csv(
        OUT / "bootstrap_cluster_order.csv", index=False
    )

    points = {}
    bootstrap = {}
    comparison_rows = []
    weighting_difference_rows = []

    print("计算两种权重下的指标和置信区间……", flush=True)

    for key in CONFIGS:
        frame = frames[key]
        p = frame.probability.to_numpy(float)
        threshold = float(frame.threshold.iloc[0])

        points[key] = {
            "image_equal": calculate(y, p, threshold, image_weights),
            "patient_equal": calculate(y, p, threshold, equal_weights),
        }

        boot_image, boot_equal = [], []
        for counts in multiplicities:
            repeats = counts[cluster_index]
            boot_image.append(calculate(
                y, p, threshold, repeats.astype(float)
            ))
            boot_equal.append(calculate(
                y, p, threshold, repeats * equal_weights
            ))

        bootstrap[key] = {
            "image_equal": np.asarray(boot_image),
            "patient_equal": np.asarray(boot_equal),
        }

        for weighting in ["image_equal", "patient_equal"]:
            point = points[key][weighting]
            samples = bootstrap[key][weighting]
            low, high = np.quantile(samples, [0.025, 0.975], axis=0)
            row = {
                "configuration": key,
                "weighting": weighting,
                "n_images": len(y),
                "n_clusters": n_clusters,
                "threshold": threshold,
                "valid_bootstraps": valid_bootstraps,
            }
            for j, metric in enumerate(METRICS):
                row[metric] = point[j]
                row[metric + "_low"] = low[j]
                row[metric + "_high"] = high[j]
            comparison_rows.append(row)

        # 同轮聚类抽样下，患者等权减图像等权。
        delta = (
            points[key]["patient_equal"] -
            points[key]["image_equal"]
        )
        samples = (
            bootstrap[key]["patient_equal"] -
            bootstrap[key]["image_equal"]
        )
        low, high = np.quantile(samples, [0.025, 0.975], axis=0)
        for j, metric in enumerate(METRICS):
            weighting_difference_rows.append({
                "configuration": key,
                "metric": metric,
                "patient_equal_minus_image_equal": delta[j],
                "low": low[j],
                "high": high[j],
            })
        print("完成：", key, flush=True)

    metrics = pd.DataFrame(comparison_rows)
    metrics.to_csv(OUT / "weighted_metrics.csv", index=False)
    pd.DataFrame(weighting_difference_rows).to_csv(
        OUT / "weighting_differences.csv", index=False
    )

    # 对照原主分析的点估计，必须一致。
    original = pd.read_csv(
        io.BytesIO(read_bytes(source, "external_metrics.csv"))
    ).set_index("configuration")

    for key in CONFIGS:
        for j, metric in enumerate(METRICS):
            if not np.isclose(
                points[key]["image_equal"][j],
                original.loc[key, metric],
                atol=1e-10, rtol=0,
            ):
                raise ValueError(f"{key}/{metric}与原结果不一致")

    # 患者等权后的配对特征增益。
    increment_rows = []
    for model in ["LR", "ensemble"]:
        for before, after, name in [
            ("base", "base_missing", "missing_indicator_addition"),
            ("base_missing", "base_posterior", "measurement_addition"),
            ("base", "base_posterior", "total_addition"),
        ]:
            key_before = before + "_" + model
            key_after = after + "_" + model

            delta = (
                points[key_after]["patient_equal"] -
                points[key_before]["patient_equal"]
            )
            samples = (
                bootstrap[key_after]["patient_equal"] -
                bootstrap[key_before]["patient_equal"]
            )
            low, high = np.quantile(samples, [0.025, 0.975], axis=0)

            for j, metric in enumerate(METRICS):
                increment_rows.append({
                    "model": model,
                    "comparison": name,
                    "metric": metric,
                    "after_minus_before": delta[j],
                    "low": low[j],
                    "high": high[j],
                    "note":
                        "Brier: negative favors after. "
                        "Other metrics: positive favors after. "
                        "Each configuration retains its own source threshold.",
                })

    increments = pd.DataFrame(increment_rows)
    increments.to_csv(
        OUT / "patient_equal_paired_increments.csv", index=False
    )

    # 各聚类错误计数供核查，不给混合标签患者强行定义单一诊断。
    error_rows = []
    for key in CONFIGS:
        frame = frames[key].copy()
        frame["false_negative"] = (
            (frame.label == 1) & (frame.predicted_label == 0)
        ).astype(int)
        frame["false_positive"] = (
            (frame.label == 0) & (frame.predicted_label == 1)
        ).astype(int)

        grouped = frame.groupby("cluster_id").agg(
            n_images=("image_id", "size"),
            malignant_images=("label", "sum"),
            false_negative_images=("false_negative", "sum"),
            false_positive_images=("false_positive", "sum"),
        ).reset_index()
        grouped["configuration"] = key
        error_rows.append(grouped)

    pd.concat(error_rows, ignore_index=True).to_csv(
        OUT / "cluster_error_counts.csv", index=False
    )

    summary = {
        "completed": True,
        "n_images": len(y),
        "n_clusters": n_clusters,
        "minimum_images_per_cluster": int(cluster_sizes.min()),
        "maximum_images_per_cluster": int(cluster_sizes.max()),
        "image_equal_malignant_fraction": float(y.mean()),
        "patient_equal_malignant_image_fraction": float(
            np.average(y, weights=equal_weights)
        ),
        "valid_bootstraps": valid_bootstraps,
        "original_point_estimates_verified": True,
        "models_retrained": False,
        "thresholds_changed": False,
    }
    (OUT / "summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    # 保存执行代码。原有结果文件不修改。
    try:
        code = get_ipython().history_manager.input_hist_raw[-1]
        (OUT / "executed_cell.py").write_text(code, encoding="utf-8")
    except Exception:
        pass

    print("\n图像等权与患者等权对照：")
    display(metrics[[
        "configuration", "weighting",
        "auc", "auc_low", "auc_high",
        "brier", "sensitivity", "specificity",
    ]])

    print("\n患者等权：加入后方测量的配对增益：")
    display(increments[
        increments.comparison == "measurement_addition"
    ][["model", "metric", "after_minus_before", "low", "high"]])

    archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
    print("\n完成。没有训练模型或改变阈值。")
    print("本次重算的主分析置信区间因抽样种子不同，可能略有差异；")
    print("原论文主结果仍使用原结果包中的置信区间。")
    print("结果包：", archive)
    display(FileLink(archive))


main()
# ======================= 代码结束 =======================

In [ ]:
# ============== BUS-UCLM轻微轮廓扰动实验 ==============
# 固定V6预处理后，4邻域腐蚀/膨胀1像素。
# 不训练、不校准、不改变阈值，不修改原始图像/掩膜。
# 原始条件必须复现已有预测，否则停止。

from pathlib import Path
from collections import defaultdict
from datetime import datetime
import io
import json
import hashlib
import zipfile
import shutil

import numpy as np
import pandas as pd
import scipy
import sklearn
import joblib
from PIL import Image
from scipy import ndimage as ndi
from scipy.special import logit
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits
from IPython.display import display, FileLink

# 自动查找失败时填写准确的Kaggle路径。
MODEL_PATH = ""
RESULT_PATH = ""

MODEL_NAME = "bus_v6_external_training_20261013_121259_911015"
RESULT_NAME = "bus_uclm_external_20261013_132503_790776"

EXTRACTOR_HASH = (
    "fc020bfc28022f1b476d435d0dc8728aaee"
    "48e03d181007bce545aee22a1dda2"
)
PREDICTION_HASH = (
    "8633521a5610e4e91a90543d1458cf893"
    "497684499d77d2163af51a5e91532ee"
)

OUT = Path("/kaggle/working") / (
    "bus_uclm_contour_robustness_" +
    datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
OUT.mkdir(parents=True, exist_ok=False)

MODES = ["original", "erode1", "dilate1"]


def sha(data):
    return hashlib.sha256(data).hexdigest()


def file_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def read_bytes(source, name):
    source = Path(source)
    if source.is_dir():
        p = source / name
        if not p.is_file():
            raise FileNotFoundError(str(p))
        return p.read_bytes()
    with zipfile.ZipFile(source) as z:
        names = [
            n for n in z.namelist()
            if not n.endswith("/")
            and Path(n).name == name
            and "__MACOSX" not in n
        ]
        if len(names) != 1:
            raise ValueError(f"{name}在ZIP内不是唯一文件")
        return z.read(names[0])


def read_csv(source, name):
    return pd.read_csv(io.BytesIO(read_bytes(source, name)))


def discover(explicit, stem, anchor):
    if explicit.strip():
        p = Path(explicit.strip())
        if not p.exists():
            raise FileNotFoundError(str(p))
        return p

    candidates = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            candidates.extend(root.rglob(stem + ".zip"))
            candidates.extend(
                p for p in root.rglob(stem) if p.is_dir()
            )
    groups = defaultdict(list)
    for p in sorted(set(candidates), key=str):
        groups[sha(read_bytes(p, anchor))].append(p)
    if len(groups) != 1:
        print("候选：", *candidates, sep="\n")
        raise RuntimeError(
            f"无法唯一确定{stem}，请填写顶部路径。"
        )
    return next(iter(groups.values()))[0]


def save_json(name, obj):
    (OUT / name).write_text(
        json.dumps(obj, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def performance(y, p, threshold):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    decision = p >= threshold
    positive, negative = y == 1, y == 0
    return {
        "n": len(y),
        "malignant_n": int(positive.sum()),
        "auc": (
            roc_auc_score(y, p)
            if positive.any() and negative.any() else np.nan
        ),
        "brier": float(np.mean((p - y) ** 2)),
        "sensitivity": (
            float(decision[positive].mean())
            if positive.any() else np.nan
        ),
        "specificity": (
            float((~decision[negative]).mean())
            if negative.any() else np.nan
        ),
        "false_negative_images": int((positive & ~decision).sum()),
        "false_positive_images": int((negative & decision).sum()),
    }


def main():
    model_source = discover(
        MODEL_PATH, MODEL_NAME, "model_hashes.json"
    )
    result_source = discover(
        RESULT_PATH, RESULT_NAME, "predictions_with_labels.csv"
    )
    print("模型：", model_source)
    print("外部结果：", result_source)

    prediction_bytes = read_bytes(
        result_source, "predictions_with_labels.csv"
    )
    if sha(prediction_bytes) != PREDICTION_HASH:
        raise ValueError("原预测文件与已核对版本不一致")

    original_predictions = pd.read_csv(io.BytesIO(prediction_bytes))
    cohort = read_csv(result_source, "cohort_frozen.csv")
    original_features = read_csv(
        result_source, "features.csv"
    ).set_index("image_id")

    if (
        len(cohort) != 207
        or not cohort.image_id.is_unique
        or int(cohort.label.sum()) != 66
    ):
        raise ValueError("队列与既定207张图像不一致")

    cohort = cohort.sort_values("image_id").reset_index(drop=True)
    cohort.to_csv(OUT / "cohort_frozen.csv", index=False)

    protocol = json.loads(read_bytes(model_source, "protocol.json"))
    if sklearn.__version__ != protocol["versions"]["scikit-learn"]:
        raise RuntimeError(
            "scikit-learn版本与原模型不一致："
            f"当前{sklearn.__version__}，"
            f"需要{protocol['versions']['scikit-learn']}。"
            "请把输出发回，不要强行加载。"
        )

    extractor = read_bytes(model_source, "v6_extractor.py")
    if sha(extractor) != EXTRACTOR_HASH:
        raise ValueError("V6提取器哈希不一致")

    scope = {"__name__": "frozen_v6"}
    exec(compile(extractor, "v6_extractor.py", "exec"), scope)
    extract_features = scope["extract_features"]
    side = int(scope["SIDE"])
    (OUT / "v6_extractor.py").write_bytes(extractor)

    # 对照外部验证记录，确保加载同一组模型。
    model_hashes = json.loads(
        read_bytes(model_source, "model_hashes.json")
    )
    recorded_hashes = json.loads(
        read_bytes(result_source, "model_hashes.json")
    )
    if model_hashes != recorded_hashes or len(model_hashes) != 6:
        raise ValueError("模型记录与原外部验证不一致")

    bundles = {}
    for filename, expected_hash in model_hashes.items():
        data = read_bytes(model_source, filename)
        if sha(data) != expected_hash:
            raise ValueError(f"{filename}哈希不一致")
        bundle = joblib.load(io.BytesIO(data))
        if bundle["feature_version"] != "V6-unified-features-1":
            raise ValueError("模型特征版本不一致")
        bundles[filename.removesuffix(".joblib")] = bundle

    save_json("protocol.json", {
        "dataset": "BUS-UCLM fixed 207-image cohort",
        "original_preprocessing": "Frozen V6, longest side 192",
        "perturbations": [
            "original",
            "one iteration of 4-neighbour erosion",
            "one iteration of 4-neighbour dilation",
        ],
        "perturbation_stage": "after original V6 resizing",
        "mask_border_value": 0,
        "physical_units": "pixels, not millimetres",
        "image_pixels_changed": False,
        "retraining": False,
        "recalibration": False,
        "threshold_changes": False,
        "invalid_conditions":
            "Retain failure records; never replace failed masks "
            "with original masks or remove them silently.",
        "comparison":
            "Paired comparison on images evaluable under all three modes",
        "inference":
            "Descriptive stress test; not inter-reader agreement "
            "and not a new independent external validation.",
        "prediction_sha256": PREDICTION_HASH,
        "extractor_sha256": EXTRACTOR_HASH,
        "versions": {
            "numpy": np.__version__,
            "scipy": scipy.__version__,
            "scikit-learn": sklearn.__version__,
        },
    })
    save_json("model_hashes.json", model_hashes)

    # 依据原文件SHA定位，兼容Kaggle路径变化和同名图像/掩膜。
    wanted = {
        Path(str(p)).name
        for col in ["image_path", "mask_path"]
        for p in cohort[col]
    }
    index = defaultdict(set)
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            for p in root.rglob("*"):
                if p.name in wanted and p.is_file():
                    index[p.name].add(p)

    hash_cache = {}

    def locate(saved_path, expected_hash):
        direct = Path(str(saved_path))
        candidates = set(index.get(direct.name, set()))
        if direct.is_file():
            candidates.add(direct)
        for p in sorted(candidates, key=str):
            if p not in hash_cache:
                hash_cache[p] = file_sha(p)
            if hash_cache[p] == expected_hash:
                return p
        raise FileNotFoundError(
            f"{direct.name}：没有找到与原审计哈希一致的文件。"
            "请确认原始BUS-UCLM已添加。"
        )

    structure = ndi.generate_binary_structure(2, 1)
    feature_rows, quality_rows, status_rows = [], [], []

    for i, row in enumerate(cohort.itertuples(index=False), 1):
        ip = locate(row.image_path, row.image_sha256)
        mp = locate(row.mask_path, row.mask_sha256)

        with Image.open(ip) as im:
            gray = np.asarray(im.convert("L"), dtype=np.uint8)
            valid = np.asarray(im.convert("RGBA"))[:, :, 3] > 0
        with Image.open(mp) as im:
            rgb = np.asarray(im.convert("RGB"))
            alpha = np.asarray(im.convert("RGBA"))[:, :, 3] > 0

        green = np.all(rgb == [0, 255, 0], axis=2)
        red = np.all(rgb == [255, 0, 0], axis=2)
        black = np.all(rgb == [0, 0, 0], axis=2)
        if not np.all((green | red | black) | ~alpha):
            raise ValueError(f"{row.image_id}：未知掩膜颜色")

        mask = (green | red) & alpha
        if gray.shape != mask.shape or (mask & ~valid).any():
            raise ValueError(f"{row.image_id}：原始几何异常")

        scale = side / max(gray.shape)
        size = (
            max(1, round(gray.shape[1] * scale)),
            max(1, round(gray.shape[0] * scale)),
        )
        x = np.asarray(
            Image.fromarray(gray.astype(np.float32) / 255.0)
            .resize(size, Image.Resampling.BILINEAR)
        )

        def resize_mask(a):
            return np.asarray(
                Image.fromarray(a.astype("uint8"))
                .resize(size, Image.Resampling.NEAREST)
            ) > 0

        m, v = resize_mask(mask), resize_mask(valid)
        variants = {
            "original": m,
            "erode1": ndi.binary_erosion(
                m, structure=structure, iterations=1, border_value=0
            ),
            "dilate1": ndi.binary_dilation(
                m, structure=structure, iterations=1, border_value=0
            ),
        }

        for mode, changed in variants.items():
            record = {
                "image_id": row.image_id,
                "mode": mode,
                "original_area": int(m.sum()),
                "perturbed_area": int(changed.sum()),
                "components": int(ndi.label(changed, structure)[1]),
                "dice_with_original": (
                    2 * (m & changed).sum() /
                    max(1, m.sum() + changed.sum())
                ),
                "status": "ok",
                "error": "",
            }
            try:
                f, q = extract_features(x, changed, v)
                if np.isinf(np.array(list(f.values()), float)).any():
                    raise ValueError("Infinite features")
                feature_rows.append({
                    "image_id": row.image_id, "mode": mode, **f
                })
                quality_rows.append({
                    "image_id": row.image_id, "mode": mode, **q
                })
            except Exception as e:
                record["status"] = "failed"
                record["error"] = str(e)
            status_rows.append(record)

        if i % 50 == 0 or i == len(cohort):
            print(f"轮廓扰动及特征提取：{i}/{len(cohort)}", flush=True)

    status = pd.DataFrame(status_rows)
    features = pd.DataFrame(feature_rows)
    quality = pd.DataFrame(quality_rows)
    status.to_csv(OUT / "extraction_status.csv", index=False)
    features.to_csv(OUT / "perturbed_features.csv", index=False)
    quality.to_csv(OUT / "perturbed_quality.csv", index=False)

    # 先核验原条件特征，未复现则停止。
    baseline = features[features["mode"] == "original"].set_index("image_id")
    ids = cohort.image_id.tolist()
    if set(baseline.index) != set(ids):
        raise RuntimeError("原始条件存在提取失败，请发回extraction_status.csv")

    feature_cols = original_features.columns.tolist()
    old = original_features.loc[ids, feature_cols].to_numpy(float)
    new = baseline.loc[ids, feature_cols].to_numpy(float)
    if not np.allclose(old, new, rtol=1e-6, atol=1e-7, equal_nan=True):
        raise RuntimeError("原条件特征未复现；停止，不能解释扰动结果")

    prediction_rows = []
    with threadpool_limits(limits=2):
        for mode in MODES:
            frame = features[features["mode"] == mode].set_index("image_id")
            if frame.empty:
                continue
            for key, bundle in bundles.items():
                X = frame[bundle["feature_columns"]]
                raw = np.mean([
                    estimator.predict_proba(X)[:, 1]
                    for estimator in bundle["models"].values()
                ], axis=0)
                p = bundle["calibrator"].predict_proba(
                    logit(np.clip(raw, 1e-6, 1 - 1e-6)).reshape(-1, 1)
                )[:, 1]
                if not np.isfinite(p).all():
                    raise ValueError("出现非有限预测概率")
                t = float(bundle["threshold"])
                prediction_rows.append(pd.DataFrame({
                    "image_id": frame.index,
                    "mode": mode,
                    "configuration": key,
                    "probability": p,
                    "threshold": t,
                    "predicted_label": (p >= t).astype(int),
                }))

    pred = pd.concat(prediction_rows, ignore_index=True)
    pred = pred.merge(
        cohort[["image_id", "patient_id", "cluster_id", "label"]],
        on="image_id", validate="many_to_one",
    )
    pred.to_csv(OUT / "all_predictions.csv", index=False)

    baseline_checks = []
    for key in bundles:
        old = original_predictions[
            original_predictions.configuration == key
        ].set_index("image_id").loc[ids]
        new = pred[
            (pred.configuration == key) & (pred["mode"] == "original")
        ].set_index("image_id").loc[ids]

        difference = np.max(np.abs(
            old.probability.to_numpy() - new.probability.to_numpy()
        ))
        same_decisions = np.array_equal(
            old.predicted_label.to_numpy(),
            new.predicted_label.to_numpy(),
        )
        same_threshold = np.allclose(
            old.threshold.to_numpy(), new.threshold.to_numpy(),
            rtol=0, atol=1e-14,
        )
        baseline_checks.append({
            "configuration": key,
            "maximum_probability_difference": difference,
            "same_decisions": same_decisions,
            "same_threshold": same_threshold,
        })
        if difference > 1e-6 or not same_decisions or not same_threshold:
            raise RuntimeError(f"{key}：原条件预测未复现，停止解释结果")

    pd.DataFrame(baseline_checks).to_csv(
        OUT / "baseline_reproduction.csv", index=False
    )

    # 三种条件均可计算的共同图像，避免不同样本间直接比较。
    ok = status[status.status == "ok"]
    counts = ok.groupby("image_id")["mode"].nunique()
    common_ids = sorted(counts[counts == len(MODES)].index)
    if not common_ids:
        raise RuntimeError("没有共同可评估图像")

    pd.Series(common_ids, name="image_id").to_csv(
        OUT / "common_evaluable_images.csv", index=False
    )

    metric_rows, drift_rows, case_changes = [], [], []
    for key in bundles:
        base = pred[
            (pred.configuration == key) & (pred["mode"] == "original")
        ].set_index("image_id").loc[common_ids]
        y = base.label.to_numpy(int)
        p0 = base.probability.to_numpy()
        t = float(base.threshold.iloc[0])

        for mode in MODES:
            changed = pred[
                (pred.configuration == key) & (pred["mode"] == mode)
            ].set_index("image_id").loc[common_ids]
            p1 = changed.probability.to_numpy()
            metric_rows.append({
                "configuration": key, "mode": mode,
                "n_clusters": changed.cluster_id.nunique(),
                **performance(y, p1, t),
            })

            if mode == "original":
                continue
            delta = p1 - p0
            d0, d1 = p0 >= t, p1 >= t
            e0, e1 = d0 != y, d1 != y

            drift_rows.append({
                "configuration": key,
                "mode": mode,
                "n": len(y),
                "median_absolute_probability_change": np.median(np.abs(delta)),
                "p95_absolute_probability_change": np.quantile(np.abs(delta), .95),
                "maximum_absolute_probability_change": np.max(np.abs(delta)),
                "threshold_crossings": int((d0 != d1).sum()),
                "threshold_crossing_fraction": float((d0 != d1).mean()),
                "new_false_negatives": int(((y == 1) & ~e0 & e1).sum()),
                "corrected_false_negatives": int(((y == 1) & e0 & ~e1).sum()),
                "new_false_positives": int(((y == 0) & ~e0 & e1).sum()),
                "corrected_false_positives": int(((y == 0) & e0 & ~e1).sum()),
            })
            case_changes.append(pd.DataFrame({
                "image_id": common_ids,
                "cluster_id": base.cluster_id.to_numpy(),
                "label": y,
                "configuration": key,
                "mode": mode,
                "original_probability": p0,
                "perturbed_probability": p1,
                "probability_change": delta,
                "threshold_crossed": d0 != d1,
            }))

    metrics = pd.DataFrame(metric_rows)
    drift = pd.DataFrame(drift_rows)
    metrics.to_csv(OUT / "common_cohort_metrics.csv", index=False)
    drift.to_csv(OUT / "prediction_stability.csv", index=False)
    pd.concat(case_changes, ignore_index=True).to_csv(
        OUT / "paired_prediction_changes.csv", index=False
    )

    # 后方测量可用性与数值稳定性。
    posterior_rows = []
    q0 = quality[quality["mode"] == "original"].set_index("image_id")
    f0 = baseline

    for mode in MODES:
        fm = features[features["mode"] == mode].set_index("image_id")
        qm = quality[quality["mode"] == mode].set_index("image_id")
        use = sorted(set(fm.index) & set(f0.index))
        usable0 = q0.loc[use, "posterior_usable"].to_numpy(bool)
        usable1 = qm.loc[use, "posterior_usable"].to_numpy(bool)
        both = usable0 & usable1
        values0 = f0.loc[use, "postdepth__delta"].to_numpy(float)
        values1 = fm.loc[use, "postdepth__delta"].to_numpy(float)
        differences = np.abs(values1[both] - values0[both])

        posterior_rows.append({
            "mode": mode,
            "total_planned_images": len(cohort),
            "extraction_success": len(use),
            "extraction_failed": len(cohort) - len(use),
            "posterior_usable": int(usable1.sum()),
            "usable_fraction_among_extracted": (
                float(usable1.mean()) if len(use) else np.nan
            ),
            "became_unusable_vs_original": int((usable0 & ~usable1).sum()),
            "became_usable_vs_original": int((~usable0 & usable1).sum()),
            "both_usable_n": int(both.sum()),
            "median_absolute_measurement_change": (
                float(np.median(differences)) if len(differences) else np.nan
            ),
            "p95_absolute_measurement_change": (
                float(np.quantile(differences, .95))
                if len(differences) else np.nan
            ),
        })

    posterior = pd.DataFrame(posterior_rows)
    posterior.to_csv(OUT / "posterior_stability.csv", index=False)

    # 仅描述增益方向是否随扰动变化，不进行新的模型选择。
    increment_rows = []
    for mode in MODES:
        for model in ["LR", "ensemble"]:
            table = metrics[metrics["mode"] == mode].set_index("configuration")
            before = table.loc["base_missing_" + model]
            after = table.loc["base_posterior_" + model]
            for metric in ["auc", "brier", "sensitivity", "specificity"]:
                increment_rows.append({
                    "mode": mode,
                    "model": model,
                    "metric": metric,
                    "posterior_minus_missing": after[metric] - before[metric],
                    "note": "Descriptive paired common-cohort difference; no CI",
                })
    pd.DataFrame(increment_rows).to_csv(
        OUT / "posterior_increment_by_perturbation.csv", index=False
    )

    save_json("summary.json", {
        "completed": True,
        "planned_images": len(cohort),
        "common_evaluable_images": len(common_ids),
        "failed_image_conditions": int((status.status != "ok").sum()),
        "baseline_reproduced": True,
        "models_retrained": False,
        "thresholds_changed": False,
    })

    try:
        code = get_ipython().history_manager.input_hist_raw[-1]
        (OUT / "executed_cell.py").write_text(code, encoding="utf-8")
    except Exception:
        pass

    print("\n原条件预测已成功复现。")
    print("三种条件共同可评估图像：", len(common_ids))
    print("\n后方测量稳定性：")
    display(posterior)
    print("\n预测稳定性：")
    display(drift)
    print("\n共同队列性能：")
    display(metrics)

    archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
    print("\n完成。请下载结果包：", archive)
    display(FileLink(archive))


main()
# ======================= 代码结束 =======================

In [ ]:
# 修复模型位于 models/ 子目录时的读取问题
from pathlib import Path
import zipfile

def read_bytes(source, name):
    source = Path(source)
    basename = Path(name).name

    if source.is_dir():
        exact = source / name
        if exact.is_file():
            return exact.read_bytes()

        matches = [
            p for p in source.rglob(basename)
            if p.is_file() and "__MACOSX" not in p.parts
        ]
        if len(matches) != 1:
            raise FileNotFoundError(
                f"{source} 内找到 {len(matches)} 个 {basename}：\n"
                + "\n".join(str(p) for p in matches)
            )
        return matches[0].read_bytes()

    with zipfile.ZipFile(source) as z:
        matches = [
            n for n in z.namelist()
            if not n.endswith("/")
            and Path(n).name == basename
            and "__MACOSX" not in n
        ]
        if len(matches) != 1:
            raise FileNotFoundError(
                f"{source.name} 内找到 {len(matches)} 个 {basename}"
            )
        return z.read(matches[0])

# 使用修复后的函数继续运行，原模型哈希检查仍保留。
main()